# Topological Structural Optimization Framework for Solids of Revolution

**Applied to an aeronautical component** | Undergraduate thesis, Engineering Physics, EAFIT University, 2026
Author: Fernando Londoño Palacio | Advisor: Ph.D. Ronald Mauricio Martinod Restrepo

This notebook is the single implementation of the framework described in Chapter 3 of the thesis
([`docs/TO_Final_Inform_FLP.pdf`](../docs/TO_Final_Inform_FLP.pdf)). Its ten blocks share their state in memory and must
run in order, as in Figure 3.1.1 of the thesis.

| Block | Purpose | Thesis |
|---|---|---|
| 1 | Material constants, meridian section, Q4 voxelization | Sec. 3.2, 3.3.1, 4.4, 4.5 |
| 2 | Constitutive matrix, supports and baseline finite element analysis | Sec. 3.3.2, 4.1.1 |
| 3 | SIMP engine (Helmholtz filter, Heaviside projection, OC) and benchmark | Sec. 3.5, 5.3 |
| 4 | Contours, closed STL revolution and benchmark figures | Sec. 3.7, 5.3 |
| 5 | Multi-load problem and search on the volume fraction | Sec. 3.6, 4.1.2, 4.1.3, 5.5, 5.6 |
| 6 | Single-case problems and cross-utilization | Sec. 3.6, 5.5, 5.8 |
| 7 | Export of the selected designs as rings and 7.5° sectors | Sec. 3.7, 4.7.1 |
| 8 | Virtual compression test between rigid platens | Sec. 3.8, 5.11, 5.12 |
| 9 | Independent verification (tables T01-T09, figures F01-F02) | Sec. 3.9, 5.1, 5.2, 5.9, 5.10 |
| 10 | Thesis figures and tables (F10_xx, T10_xx) and check against the thesis | Sec. 3.9, Chapter 5 |

**Units:** mm, N, MPa, g. **Voigt order:** $[\varepsilon_r, \varepsilon_z, \varepsilon_\theta, \gamma_{rz}]$, Eq. (2.1).

## Run configuration

The searches of Blocks 5 and 6 and the virtual tests of Block 8 take about three hours. Their results are stored in
`results/`, so by default the notebook **loads them and runs everything else in about 15 minutes**. Set the flags to `True`
to regenerate them from scratch.

In [ ]:
import csv
import gc
import json
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Patch
from matplotlib.path import Path as MplPath
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy import ndimage
from scipy.sparse.linalg import LinearOperator, cg, splu, spsolve
from scipy.spatial import cKDTree
from skimage import measure
from stl import mesh as stl_mesh

import mapbox_earcut as earcut

warnings.filterwarnings("ignore")

# ---- What to regenerate --------------------------------------------------------------------------------------------
RUN_SEARCHES = False       # True: run the 103 SIMP runs of Blocks 5 and 6 (about 3 h). False: load results/designs/
RUN_VIRTUAL_TEST = False   # True: run Block 8 (about 25 min). False: load results/virtual_test/
EXPORT_RINGS = True        # True: also export the 360-degree rings in Block 7 (249 MB, ignored by git)

# ---- Paths (works from the repository root or from notebooks/) ------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STL_FILE = ROOT / "data" / "geometry" / "Solido_half.stl"
RESULTS = ROOT / "results"
DIR_BENCH = RESULTS / "benchmark"           # Blocks 3 and 4
DIR_DESIGNS = RESULTS / "designs"           # Blocks 5 and 6: binary designs and search history
DIR_SEARCH = RESULTS / "searches"           # Blocks 5 and 6: figures
DIR_STL = RESULTS / "stl"                   # Blocks 4 and 7: sectors/, rings/, benchmark/
DIR_VTEST = RESULTS / "virtual_test"        # Block 8
DIR_VERIF = RESULTS / "verification"        # Block 9: T01-T09, F01, F02
DIR_THESIS = RESULTS / "thesis_figures"     # Block 10: F10_xx, T10_xx
for _d in [DIR_BENCH, DIR_DESIGNS, DIR_SEARCH, DIR_STL / "sectors", DIR_STL / "rings", DIR_STL / "benchmark",
           DIR_VTEST, DIR_VERIF, DIR_THESIS]:
    _d.mkdir(parents=True, exist_ok=True)

# ---- Sparse direct solver: Intel PARDISO when available, otherwise SuperLU ------------------------------------------
try:
    import pypardiso
    def solve_sparse(K, B):
        """Solves K X = B with PARDISO (B may hold several right-hand sides)."""
        return pypardiso.spsolve(K.tocsr(), B)
    SOLVER_NAME = "PARDISO"
except ImportError:
    def solve_sparse(K, B):
        """Solves K X = B with one SuperLU factorization (B may hold several right-hand sides)."""
        return splu(K.tocsc(), permc_spec="MMD_AT_PLUS_A").solve(B)
    SOLVER_NAME = "SuperLU"

SEARCH_LOG = DIR_DESIGNS / "search_log.txt"
def log(msg):
    """Prints a line and, during a full run, appends it to results/designs/search_log.txt."""
    print(msg)
    if RUN_SEARCHES:
        with open(SEARCH_LOG, "a", encoding="utf-8") as fh:
            fh.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}\n")

print(f"Repository root: {ROOT}")
print(f"Solver: {SOLVER_NAME} | RUN_SEARCHES = {RUN_SEARCHES} | RUN_VIRTUAL_TEST = {RUN_VIRTUAL_TEST}")

---
# Block 1. Constants, meridian section and Q4 voxelization

## 1.1 Transversely isotropic material (Sec. 3.2 and 4.4)

The part is printed with its axis along the build direction $z$, so the printed material is a homogenized, transversely
isotropic continuum with five independent constants: $E_r$, $E_z$, $\nu_{r\theta}$, $\nu_{rz}$ and $G_{rz}$.

Only the two moduli are measured, in `material_characterization.ipynb`. The others are estimated:

$$\nu_{rz} = \nu_{r\theta}\sqrt{E_r/E_z},\qquad \nu_{zr} = \nu_{r\theta}\sqrt{E_z/E_r}\qquad\text{(Huber's closure, Eq. 3.3)},\qquad
G_{rz} = \alpha_G E_z,\quad \alpha_G = 1/3 .$$

Before assembly the constants must satisfy the Lempriere conditions of Eq. (2.16).

In [ ]:
ACTIVE_MATERIAL = "PA6"    # Material of the baseline analysis and the benchmark (Blocks 1 to 4)
ALPHA_G = 1.0 / 3.0        # G_rz = ALPHA_G * E_z, varied in the sensitivity study of Block 9
NELR, NELZ = 216, 280      # Q4 elements along r and z (element size close to 0.29 mm)

# Table 4.4.1. E_r: ASTM D638, parallel to the layers. E_z and Sy: ASTM D695, across the layers.
# Density: gravimetric, 35 % gyroid. nu_rtheta: bulk PA6.
MATERIAL_CATALOG = {
    "PA6": dict(name="Polyamide 6 (PA6)", E_r=839.44, E_z=294.32, nu_rtheta=0.39, Sy=53.29, density_g_cm3=0.7933),
    "PA6_CF": dict(name="Carbon-fiber polyamide 6 (PA6-CF)", E_r=2056.59, E_z=684.46, nu_rtheta=0.39, Sy=25.84,
                   density_g_cm3=0.7445),
}


def derive_constants(props, alpha_G=ALPHA_G):
    """Completes the five constants with Huber's closure (Eq. 3.3) and checks the Lempriere conditions (Eq. 2.16)."""
    E_r, E_z, nu_p = props["E_r"], props["E_z"], props["nu_rtheta"]
    nu_rz = nu_p * np.sqrt(E_r / E_z)
    nu_zr = nu_p * np.sqrt(E_z / E_r)
    G_rz = alpha_G * E_z
    G_huber = np.sqrt(E_r * E_z) / (2.0 * (1.0 + np.sqrt(nu_rz * nu_zr)))     # Huber's estimate, Eq. (2.17)
    G_rtheta = E_r / (2.0 * (1.0 + nu_p))                                      # Eq. (2.12)
    Delta = (1.0 + nu_p) * (1.0 - nu_p - 2.0 * nu_rz * nu_zr)                  # Eq. (2.14)
    checks = {
        "E_r, E_z, G_rz > 0": (E_r > 0) and (E_z > 0) and (G_rz > 0),
        "|nu_rtheta| < 1": abs(nu_p) < 1.0,
        "|nu_rz| < sqrt(E_r/E_z)": abs(nu_rz) < np.sqrt(E_r / E_z),
        "Delta > 0": Delta > 0.0,
    }
    d = dict(props)
    d.update(nu_rz=nu_rz, nu_zr=nu_zr, G_rz=G_rz, G_rtheta=G_rtheta, Delta=Delta, alpha_G=alpha_G, G_huber=G_huber,
             alpha_huber=G_huber / E_z, admissible=all(checks.values()), checks=checks,
             rho_g_mm3=props["density_g_cm3"] * 1.0e-3)
    return d


MATERIALS = {k: derive_constants(v) for k, v in MATERIAL_CATALOG.items()}
for k, m in MATERIALS.items():
    if not m["admissible"]:
        raise ValueError(f"[{k}] constants not admissible: {[c for c, ok in m['checks'].items() if not ok]}")

pd.DataFrame({k: {"E_r [MPa]": m["E_r"], "E_z [MPa]": m["E_z"], "E_r/E_z [-]": m["E_r"] / m["E_z"],
                  "G_rz [MPa]": m["G_rz"], "G_rz Huber [MPa]": m["G_huber"], "alpha_G Huber [-]": m["alpha_huber"],
                  "nu_rtheta [-]": m["nu_rtheta"], "nu_rz [-]": m["nu_rz"], "nu_zr [-]": m["nu_zr"], "Delta [-]": m["Delta"],
                  "Sy [MPa]": m["Sy"], "density [g/cm3]": m["density_g_cm3"], "Sy/E_z [%]": 100 * m["Sy"] / m["E_z"]}
              for k, m in MATERIALS.items()}).round(4)

In [ ]:
# Constants of the active material, used by Blocks 1 to 4
mat = MATERIALS[ACTIVE_MATERIAL]
E_r, E_z, G_rz = mat["E_r"], mat["E_z"], mat["G_rz"]
nu_rz, nu_zr, nu_rt = mat["nu_rz"], mat["nu_zr"], mat["nu_rtheta"]
Sy_mat, rho_g_mm3 = mat["Sy"], mat["rho_g_mm3"]
print(f"Active material: {ACTIVE_MATERIAL} | {mat['name']}")

## 1.2 Meridian contour from the STL file (Sec. 3.3.1)

The input is a closed STL surface of one 180° half. If its axis lies along $x$, a rigid rotation aligns it with $z$, and
the base is moved to $z = 0$. The mesh is cut by the half-plane $x = 10^{-4}$ mm, $y < 0$ (the small offset avoids
vertices lying exactly on the plane), and each crossing is mapped to $(r, z)$. The result is the closed polygon $\partial\Omega$.

In [ ]:
stl_part = stl_mesh.Mesh.from_file(str(STL_FILE))
v_raw = stl_part.vectors.astype(np.float64).copy()

# Canonical orientation: axis of revolution along z, base at z = 0
x_span, z_span = np.ptp(v_raw[:, :, 0]), np.ptp(v_raw[:, :, 2])
v_can = np.zeros_like(v_raw)
if x_span < 100.0 and z_span > 200.0:
    v_can[:, :, 0] = v_raw[:, :, 2]
    v_can[:, :, 1] = v_raw[:, :, 1]
    v_can[:, :, 2] = -v_raw[:, :, 0]
else:
    v_can = v_raw.copy()
v_can[:, :, 2] -= np.min(v_can[:, :, 2])

R_nodes_3d = np.hypot(v_can[:, :, 0], v_can[:, :, 1])
R_in, R_out = float(np.min(R_nodes_3d)), float(np.max(R_nodes_3d))
H_part = float(np.max(v_can[:, :, 2]))

# Volume enclosed by the STL (divergence theorem), extrapolated from its angular span to the complete ring
_m = stl_mesh.Mesh(np.zeros(len(v_can), dtype=stl_mesh.Mesh.dtype))
_m.vectors[:] = v_can
V_stl_part = abs(float(_m.get_mass_properties()[0]))
ang = np.sort(np.mod(np.arctan2(v_can[:, :, 1], v_can[:, :, 0]).ravel(), 2 * np.pi))
gap_max = np.max(np.diff(np.concatenate([ang, [ang[0] + 2 * np.pi]])))
span_ang = 2 * np.pi - gap_max if gap_max > 1e-3 else 2 * np.pi
V_stl_360 = V_stl_part * (2 * np.pi / span_ang)

# Intersection with the meridian half-plane: two edge crossings per intersected triangle
x_cut = 1e-4
tri_sel = v_can[np.mean(v_can[:, :, 1], axis=1) < 0.0]
dxv = tri_sel[:, :, 0] - x_cut
tri_sel = tri_sel[(dxv.min(axis=1) < 0.0) & (dxv.max(axis=1) > 0.0)]
contour_segments = []
for tri in tri_sel:
    xv = tri[:, 0] - x_cut
    pts_seg = []
    for a_i in range(3):
        b_i = (a_i + 1) % 3
        if xv[a_i] * xv[b_i] < 0.0:
            t_cut = abs(xv[a_i]) / (abs(xv[a_i]) + abs(xv[b_i]))
            p_cut = tri[a_i] + t_cut * (tri[b_i] - tri[a_i])
            pts_seg.append([np.hypot(p_cut[0], p_cut[1]), p_cut[2]])
    if len(pts_seg) == 2:
        contour_segments.append(pts_seg)
contour_segments = np.array(contour_segments)
if len(contour_segments) == 0:
    raise RuntimeError("The meridian plane does not intersect the STL: check the orientation of the file.")
print(f"R_in = {R_in:.3f} mm | R_out = {R_out:.3f} mm | H = {H_part:.3f} mm | {len(contour_segments)} contour segments")

## 1.3 Structured Q4 mesh, voxelization and element volumes (Sec. 3.3.1)

The rectangle $[R_{in}, R_{out}] \times [0, H]$ is divided into a structured grid. This makes the Jacobian equal in every
element and turns the density field into an image for connectivity labeling and marching squares.

Each element is classified with the **even-odd rule**: a ray in the $+r$ direction is cast from its centroid and its
crossings with the contour are counted. Element volumes follow from the theorem of Pappus, Eq. (3.4):

$$V_e = \pi (r_2^2 - r_1^2)\,\Delta z = 2\pi r_c\,\Delta r\,\Delta z ,\qquad m(\rho) = \rho_{mat}\sum_e \rho_e V_e .$$

In [ ]:
nelr, nelz = NELR, NELZ
n_elem = nelr * nelz
n_nodes = (nelr + 1) * (nelz + 1)
dr, dz = (R_out - R_in) / nelr, H_part / nelz

r_grid = np.linspace(R_in, R_out, nelr + 1)
z_grid = np.linspace(0.0, H_part, nelz + 1)
R_mesh, Z_mesh = np.meshgrid(r_grid, z_grid)           # shape (nelz+1, nelr+1)
r_nodes, z_nodes = R_mesh.ravel(), Z_mesh.ravel()      # node index = j*(nelr+1) + i

# Counterclockwise connectivity [n1, n2, n3, n4]: lower-left, lower-right, upper-right, upper-left
j_idx, i_idx = np.meshgrid(np.arange(nelz), np.arange(nelr), indexing="ij")
n1_mat = j_idx * (nelr + 1) + i_idx
connectivity = np.column_stack([n1_mat.ravel(), (n1_mat + 1).ravel(), (n1_mat + nelr + 2).ravel(), (n1_mat + nelr + 1).ravel()])

centroid_r = np.mean(r_nodes[connectivity], axis=1)
centroid_z = np.mean(z_nodes[connectivity], axis=1)

# Even-odd voxelization, ray along +r
r1, z1 = contour_segments[:, 0, 0], contour_segments[:, 0, 1]
r2, z2 = contour_segments[:, 1, 0], contour_segments[:, 1, 1]
crossings = np.zeros(n_elem, dtype=np.int32)
for s_i in range(len(contour_segments)):
    cz = (z1[s_i] > centroid_z) != (z2[s_i] > centroid_z)
    if np.any(cz):
        r_int = r1[s_i] + (centroid_z[cz] - z1[s_i]) * (r2[s_i] - r1[s_i]) / (z2[s_i] - z1[s_i])
        crossings[cz] += (r_int > centroid_r[cz])

rho_0 = (crossings % 2 == 1).astype(np.float64)        # Design domain: original solid
rho_image = rho_0.reshape((nelz, nelr))

# Pappus volumes and mass of the solid
V_elem = 2.0 * np.pi * centroid_r * dr * dz            # [mm^3]
mass_elem_g = rho_g_mm3 * V_elem                       # [g]
V_box = float(np.sum(V_elem))                          # Box of revolution
V0 = float(np.sum(V_elem * rho_0))                     # Voxelized original solid
mass_0_g = float(np.sum(mass_elem_g * rho_0))
area_fraction_0 = float(np.mean(rho_0))
volume_fraction_0 = V0 / V_box
err_vox = 100.0 * (V0 - V_stl_360) / V_stl_360

print(f"Grid {nelr} x {nelz} = {n_elem:,} elements | dr x dz = {dr:.4f} x {dz:.4f} mm | {n_nodes:,} nodes, {2 * n_nodes:,} DOFs")
print(f"Solid elements: {int(rho_0.sum()):,} | planar area fraction {100 * area_fraction_0:.2f} % | volume fraction {100 * volume_fraction_0:.2f} %")
print(f"STL volume ({np.degrees(span_ang):.1f} deg): {V_stl_part:,.1f} mm3 | extrapolated to 360 deg: {V_stl_360:,.1f} mm3")
print(f"Voxelized volume V0: {V0:,.1f} mm3 ({err_vox:+.3f} % against the STL)")
for k, m in MATERIALS.items():
    print(f"Mass of the solid in {k:<7}: {m['rho_g_mm3'] * V0:,.2f} g")

In [ ]:
# Audit of the discretization: voxelized section with a zoom, and volume weight of each element
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), dpi=110)
rho_vis = np.ma.masked_where(rho_image < 0.5, rho_image)
cmap_solid = ListedColormap(["#4a86e8"])

ax0 = axes[0]
ax0.pcolormesh(R_mesh, Z_mesh, rho_vis, cmap=cmap_solid, alpha=0.65, shading="flat")
ax0.add_collection(LineCollection(contour_segments, colors="crimson", linewidths=1.2, label="Exact STL contour"))
ax0.set_title(f"Voxelized Q4 mesh ({nelr} x {nelz} = {n_elem:,} elements)", fontweight="bold")
ax0.legend(loc="lower left", fontsize=9)
r_zoom = np.percentile(centroid_r[rho_0 == 1], 97)
z_zoom = np.percentile(centroid_z[rho_0 == 1], 97)
ax_ins = inset_axes(ax0, width="38%", height="32%", loc="center left", borderpad=1.8)
ax_ins.pcolormesh(R_mesh, Z_mesh, rho_vis, cmap=cmap_solid, edgecolors="black", linewidth=0.3, alpha=0.8)
ax_ins.add_collection(LineCollection(contour_segments, colors="crimson", linewidths=1.5))
ax_ins.set_xlim(r_zoom - 12 * dr, r_zoom + 12 * dr)
ax_ins.set_ylim(z_zoom - 12 * dz, z_zoom + 12 * dz)
ax_ins.set_aspect("equal"); ax_ins.tick_params(labelsize=7)
ax_ins.set_title("Zoom: Q4 elements", fontsize=8, fontweight="bold")

ax1 = axes[1]
V_plot = np.where(rho_image == 1.0, V_elem.reshape(nelz, nelr), np.nan)
pc = ax1.pcolormesh(R_mesh, Z_mesh, V_plot, cmap="viridis", shading="flat")
ax1.add_collection(LineCollection(contour_segments, colors="black", linewidths=0.8))
fig.colorbar(pc, ax=ax1, fraction=0.046, pad=0.04).set_label(r"$V_e = 2\pi r_c\,\Delta r\,\Delta z$  [mm$^3$]")
ax1.set_title(f"Volume of revolution per element | mass {ACTIVE_MATERIAL} = {mass_0_g:,.1f} g", fontweight="bold")
for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(R_in - 4, R_out + 4); ax.set_ylim(-4, H_part + 4)
    ax.set_xlabel("Radius r [mm]"); ax.set_ylabel("Axial coordinate z [mm]")
    ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

---
# Block 2. Constitutive matrix, supports and baseline analysis

## 2.1 Compliance and stiffness matrices (Eq. 2.13, 2.15)

The axisymmetric compliance matrix, in the order of Eq. (2.1), is inverted to give $C_0 = S^{-1}$. The block checks symmetry,
inversion, positive definiteness and the reduction to the classical Lamé values for an isotropic input (Table 5.1.1).

In [ ]:
def compliance_matrix(E_r, E_z, nu_rt, nu_rz, nu_zr, G_rz):
    """4x4 compliance matrix of Eq. (2.13): transversely isotropic, plane of isotropy r-theta, axis z."""
    return np.array([
        [1.0 / E_r, -nu_zr / E_z, -nu_rt / E_r, 0.0],
        [-nu_rz / E_r, 1.0 / E_z, -nu_rz / E_r, 0.0],
        [-nu_rt / E_r, -nu_zr / E_z, 1.0 / E_r, 0.0],
        [0.0, 0.0, 0.0, 1.0 / G_rz]])


S_mat = compliance_matrix(E_r, E_z, nu_rt, nu_rz, nu_zr, G_rz)
C_mat = np.linalg.inv(S_mat)

asym_C = np.max(np.abs(C_mat - C_mat.T)) / np.max(np.abs(C_mat))
eig_C = np.linalg.eigvalsh(0.5 * (C_mat + C_mat.T))
err_inv = np.max(np.abs(C_mat @ S_mat - np.eye(4)))
C_mat = 0.5 * (C_mat + C_mat.T)                        # Removes round-off asymmetry

# Isotropic limit against the Lame matrix
E_t, nu_t = 1000.0, 0.3
C_iso_num = np.linalg.inv(compliance_matrix(E_t, E_t, nu_t, nu_t, nu_t, E_t / (2 * (1 + nu_t))))
lam_t, mu_t = E_t * nu_t / ((1 + nu_t) * (1 - 2 * nu_t)), E_t / (2 * (1 + nu_t))
C_iso_an = np.array([[lam_t + 2 * mu_t, lam_t, lam_t, 0], [lam_t, lam_t + 2 * mu_t, lam_t, 0],
                     [lam_t, lam_t, lam_t + 2 * mu_t, 0], [0, 0, 0, mu_t]])
err_iso = np.max(np.abs(C_iso_num - C_iso_an)) / np.max(C_iso_an)

print("C_0 [MPa] =\n", np.array2string(C_mat, precision=2, suppress_small=True))
print(f"Relative asymmetry of C_0        : {asym_C:.2e}")
print(f"Inversion error max|C_0 S - I|   : {err_inv:.2e}")
print(f"Eigenvalues of C_0 [MPa]         : {np.array2string(eig_C, precision=2)}")
print(f"Isotropic input against Lame     : {err_iso:.2e}")
if eig_C.min() <= 0:
    raise ValueError("C_0 is not positive definite: check the constants of Block 1.")

## 2.2 Element stiffness (Eq. 3.6, 3.7)

Each element uses the Q4 interpolation with the discrete hoop strain in the third row of $B$:

$$k_e^0 \approx \frac{\Delta r\,\Delta z}{4}\sum_{k,l=1}^{2} B^T(\xi_k,\eta_l)\, C_0\, B(\xi_k,\eta_l)\; 2\pi r(\xi_k,\eta_l),
\qquad \xi_k,\eta_l = \pm 1/\sqrt{3}.$$

The grid is regular, so all elements are integrated in one vectorized operation.

In [ ]:
detJ = 0.25 * dr * dz
gp = np.array([-1.0, 1.0]) / np.sqrt(3.0)
r_elem_nodes = r_nodes[connectivity]                   # (n_elem, 4)


def shape_functions(xi, eta):
    """Q4 shape functions and their derivatives with respect to r and z (Eq. 2.22)."""
    N = 0.25 * np.array([(1 - xi) * (1 - eta), (1 + xi) * (1 - eta), (1 + xi) * (1 + eta), (1 - xi) * (1 + eta)])
    dN_dr = (0.5 / dr) * np.array([-(1 - eta), (1 - eta), (1 + eta), -(1 + eta)])
    dN_dz = (0.5 / dz) * np.array([-(1 - xi), -(1 + xi), (1 + xi), (1 - xi)])
    return N, dN_dr, dN_dz


ke_all = np.zeros((n_elem, 8, 8), dtype=np.float64)
for xi in gp:
    for eta in gp:
        N_gp, dN_dr, dN_dz = shape_functions(xi, eta)
        r_gp = r_elem_nodes @ N_gp                     # Physical radius at the Gauss point
        B = np.zeros((n_elem, 4, 8))
        B[:, 0, 0::2] = dN_dr                          # eps_r     = du_r/dr
        B[:, 1, 1::2] = dN_dz                          # eps_z     = du_z/dz
        B[:, 2, 0::2] = N_gp[None, :] / r_gp[:, None]  # eps_theta = u_r/r
        B[:, 3, 0::2] = dN_dz                          # gamma_rz  = du_r/dz + du_z/dr
        B[:, 3, 1::2] = dN_dr
        dV = 2.0 * np.pi * r_gp * detJ                 # Unit Gauss weights
        ke_all += np.einsum("eji,jk,ekl->eil", B, C_mat, B) * dV[:, None, None]

elem_dofs = np.empty((n_elem, 8), dtype=np.int64)
elem_dofs[:, 0::2] = 2 * connectivity
elem_dofs[:, 1::2] = 2 * connectivity + 1
I_idx = np.repeat(elem_dofs, 8, axis=1).ravel()
J_idx = np.tile(elem_dofs, (1, 8)).ravel()
K_val_base = ke_all.ravel()                            # Solid stiffness (rho = 1) of ALL the elements

## 2.3 Supports and reference load (Sec. 3.5.4, 4.1.1)

Two kinematic conditions hold in every design load case: the base $z = 0$ slides ($u_z = 0$, $u_r$ free), and the outer wall
$r = R_{out}$ is clamped ($u_r = u_z = 0$). Nodes outside $\Omega$ are removed from the system, not regularized.

The baseline uses the **axial reference pressure** of the mesh study and the benchmark (5 MPa on the top face). A uniform
pressure gives the consistent nodal forces of Eq. (2.24):

$$F_a = \frac{2\pi L P}{6}(2r_a + r_b)(-n),\qquad F_b = \frac{2\pi L P}{6}(r_a + 2r_b)(-n).$$

In [ ]:
P_BENCH = 5.0     # [MPa] Axial reference pressure on the top face, direction -z

elem_active = np.where(rho_0 == 1.0)[0]
nodes_active = np.unique(connectivity[elem_active])
nodes_inactive = np.setdiff1d(np.arange(n_nodes), nodes_active)

nodes_sliding = nodes_active[z_nodes[nodes_active] <= 1e-6]                  # Base z = 0
nodes_clamped = nodes_active[r_nodes[nodes_active] >= R_out - 0.5 * dr]      # Outer wall r = R_out

dofs_fixed = np.unique(np.concatenate([
    2 * nodes_sliding + 1,
    2 * nodes_clamped, 2 * nodes_clamped + 1,
    2 * nodes_inactive, 2 * nodes_inactive + 1]).astype(np.int64))
dofs_free = np.setdiff1d(np.arange(2 * n_nodes), dofs_fixed)

# Loaded edges: upper edge of the solid elements with void above, within 1.5 dz of the top
jj, ii = np.divmod(np.arange(n_elem), nelr)
void_above = np.ones(n_elem, dtype=bool)
void_above[jj < nelz - 1] = rho_0[(jj[jj < nelz - 1] + 1) * nelr + ii[jj < nelz - 1]] == 0.0
e_top = np.where((rho_0 == 1.0) & void_above & (centroid_z >= H_part - 1.5 * dz))[0]
n_a, n_b = connectivity[e_top, 3], connectivity[e_top, 2]      # Upper edge n4 -> n3
r_a, r_b = r_nodes[n_a], r_nodes[n_b]
L_e = np.abs(r_b - r_a)

F_vector = np.zeros(2 * n_nodes)
np.add.at(F_vector, 2 * n_a + 1, -2.0 * np.pi * L_e * P_BENCH * (2 * r_a + r_b) / 6.0)
np.add.at(F_vector, 2 * n_b + 1, -2.0 * np.pi * L_e * P_BENCH * (r_a + 2 * r_b) / 6.0)

A_top = float(np.sum(np.pi * (r_b**2 - r_a**2)))               # Exact area of the loaded annulus
F_total = -float(np.sum(F_vector[1::2]))                       # [N]
err_F = abs(F_total - P_BENCH * A_top) / (P_BENCH * A_top)

print(f"Sliding base: {len(nodes_sliding)} nodes | clamped outer wall: {len(nodes_clamped)} nodes | free DOFs: {len(dofs_free):,}")
print(f"Loaded edges: {len(e_top)} | area {A_top:,.1f} mm2 | force {F_total:,.1f} N | error against P*A: {err_F:.1e}")

## 2.4 Solution of the original solid and verification (Sec. 5.1, 5.2)

The block solves $K U = F$ and checks residual, axial equilibrium and the energy identity $F^T U = U^T K U$ of Eq. (2.7).

Stresses are evaluated at the element centroid, and the von Mises stress follows from Eq. (2.19). The largest element stress
is singular at corners, so the yield check uses the **99.5th percentile** of Eq. (2.20).

In [ ]:
K_val_0 = K_val_base * np.repeat(rho_0, 64)
K_global = sp.coo_matrix((K_val_0, (I_idx, J_idx)), shape=(2 * n_nodes, 2 * n_nodes)).tocsc()
K_ff = K_global[dofs_free, :][:, dofs_free]

t_sol = time.time()
U_0 = np.zeros(2 * n_nodes)
U_0[dofs_free] = spsolve(K_ff, F_vector[dofs_free])
t_sol = time.time() - t_sol

# Equilibrium and energy checks
Reac = K_global @ U_0 - F_vector
res_free = np.linalg.norm(Reac[dofs_free]) / np.linalg.norm(F_vector)
R_z_total = float(np.sum(Reac[1::2]))
R_z_base = float(np.sum(Reac[2 * nodes_sliding + 1]))
R_z_wall = float(np.sum(Reac[2 * np.setdiff1d(nodes_clamped, nodes_sliding) + 1]))
J_0 = float(F_vector @ U_0)                                    # Compliance [N mm]
W_int = float(U_0 @ (K_global @ U_0))

# Post-processing: displacements and centroid stresses
ur_0, uz_0 = U_0[0::2], U_0[1::2]
umag_0 = np.hypot(ur_0, uz_0)
u_max_0 = float(np.max(umag_0[nodes_active]))
node_umax = nodes_active[np.argmax(umag_0[nodes_active])]

N_c = np.full(4, 0.25)
dNr_c = (0.5 / dr) * np.array([-1.0, 1.0, 1.0, -1.0])
dNz_c = (0.5 / dz) * np.array([-1.0, -1.0, 1.0, 1.0])
B_centroid = np.zeros((n_elem, 4, 8))
B_centroid[:, 0, 0::2] = dNr_c
B_centroid[:, 1, 1::2] = dNz_c
B_centroid[:, 2, 0::2] = N_c[None, :] / centroid_r[:, None]
B_centroid[:, 3, 0::2] = dNz_c
B_centroid[:, 3, 1::2] = dNr_c
CB_centroid = np.einsum("ij,ejk->eik", C_mat, B_centroid)


def von_mises(U):
    """Centroid von Mises stress (Eq. 2.19) and stress components of all the elements for a displacement vector U."""
    s = np.einsum("eik,ek->ei", CB_centroid, U[elem_dofs])
    sr, sz, st, trz = s[:, 0], s[:, 1], s[:, 2], s[:, 3]
    return np.sqrt(0.5 * ((sr - sz)**2 + (sz - st)**2 + (st - sr)**2) + 3.0 * trz**2), s


sigma_VM_0, sigma_0 = von_mises(U_0)
sigma_VM_0[rho_0 == 0.0] = 0.0
vm_act = sigma_VM_0[elem_active]
vm_max_0 = float(vm_act.max())
vm_p995_0 = float(np.percentile(vm_act, 99.5))
e_vmmax = elem_active[np.argmax(vm_act)]
FS_0, FS_p995_0 = Sy_mat / vm_max_0, Sy_mat / vm_p995_0

print(f"Solved in {t_sol:.2f} s")
print(f"Residual ||K_ff U - F|| / ||F||  : {res_free:.2e}")
print(f"Equilibrium sum(R_z) - F_total   : {R_z_total - F_total:+.3e} N")
print(f"  base reaction                  : {R_z_base:,.1f} N ({100 * R_z_base / F_total:.1f} %)")
print(f"  outer-wall reaction            : {R_z_wall:,.1f} N ({100 * R_z_wall / F_total:.1f} %)")
print(f"Energy |F.U - U.K.U| / F.U       : {abs(J_0 - W_int) / J_0:.2e}")
print(f"Mass                             : {mass_0_g:,.2f} g")
print(f"u_max                            : {u_max_0:.5f} mm at (r, z) = ({r_nodes[node_umax]:.2f}, {z_nodes[node_umax]:.2f})")
print(f"Compliance J_0                   : {J_0:,.3f} N mm")
print(f"sigma_VM max / p99.5             : {vm_max_0:.3f} / {vm_p995_0:.3f} MPa, max at ({centroid_r[e_vmmax]:.2f}, {centroid_z[e_vmmax]:.2f})")
print(f"FS = Sy / sigma_VM (max / p99.5) : {FS_0:.2f} / {FS_p995_0:.2f}")

In [ ]:
# Figure 4.1.1: supports on the meridian section
fig, ax = plt.subplots(figsize=(5, 6.2), dpi=140)
ax.pcolormesh(R_mesh, Z_mesh, rho_image, cmap=ListedColormap(["#999999", "#eeeeee"]), shading="flat")
ax.add_collection(LineCollection(contour_segments, colors="black", linewidths=1.0))
ax.plot(r_nodes[nodes_clamped], z_nodes[nodes_clamped], "rX", ms=5, label="Fixed Boundary")
ax.plot(r_nodes[nodes_sliding], z_nodes[nodes_sliding], "b^", ms=5, label="Sliding Boundary (Base Z=0)")
ax.set_aspect("equal"); ax.set_xlim(R_in - 4, R_out + 5); ax.set_ylim(-5, H_part + 5)
ax.set_xlabel("Radius r [mm]"); ax.set_ylabel("Axial axis z [mm]"); ax.grid(True, ls="--", alpha=0.5)
ax.set_title("Kinematic Boundary Conditions", fontweight="bold"); ax.legend(loc="lower left")
plt.tight_layout()
(DIR_THESIS / "01_load_cases").mkdir(parents=True, exist_ok=True)
fig.savefig(DIR_THESIS / "01_load_cases" / "supports.png", dpi=300, bbox_inches="tight")
plt.show()

# Baseline response under the axial reference load
fig, axes = plt.subplots(1, 2, figsize=(13, 7), dpi=110)
ax = axes[0]
scale_def = 0.08 * (R_out - R_in) / u_max_0
conn_act = connectivity[elem_active]
verts = np.stack([r_nodes[conn_act] + scale_def * ur_0[conn_act], z_nodes[conn_act] + scale_def * uz_0[conn_act]], axis=-1)
pc = PolyCollection(verts, array=umag_0[conn_act].mean(axis=1), cmap="viridis", edgecolors="face", linewidths=0.2)
ax.add_collection(pc)
ax.add_collection(LineCollection(contour_segments, colors="grey", linewidths=0.6, linestyles="--"))
fig.colorbar(pc, ax=ax, fraction=0.046, pad=0.04).set_label("|u| [mm]")
ax.set_title(f"Deformed shape (x{scale_def:.0f}) | u_max = {u_max_0:.4f} mm", fontweight="bold")
ax = axes[1]
pv = ax.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(rho_image < 0.5, sigma_VM_0.reshape(nelz, nelr)), cmap="jet",
                   vmin=0, vmax=vm_p995_0, shading="flat")
ax.add_collection(LineCollection(contour_segments, colors="black", linewidths=0.6))
ax.plot(centroid_r[e_vmmax], centroid_z[e_vmmax], "w*", ms=12, mec="k")
fig.colorbar(pv, ax=ax, fraction=0.046, pad=0.04, extend="max").set_label(r"$\sigma_{VM}$ [MPa] (capped at p99.5)")
ax.set_title(f"Von Mises stress | max {vm_max_0:.2f} MPa (star)", fontweight="bold")
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlim(R_in - 4, R_out + 4); ax.set_ylim(-4, H_part + 10)
    ax.set_xlabel("Radius r [mm]"); ax.set_ylabel("Axial coordinate z [mm]"); ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

---
# Block 3. SIMP engine and benchmark

## 3.1 Parameters and passive shell (Table 3.5.1, Sec. 3.5)

Elements whose centroids lie within $t_s = 2$ mm of the contour form the **passive shell**, fixed at $\bar\rho_e = 1$. It keeps the
supported, critical and loaded surfaces, so the loads never depend on the design. The target $V_f$ must exceed its volume fraction.

In [ ]:
VOLFRAC = 0.60          # Benchmark volume fraction
PENAL = 3.0             # SIMP exponent p
E_MIN = 1e-9            # Relative stiffness of the void
R_MIN = 2.6             # [mm] Equivalent filter radius (about 9 elements)
SHELL_THICKNESS = 2.0   # [mm] Passive shell t_s
MOVE = 0.15             # Base move limit of the OC update
BETAS = [1, 2, 4, 8, 16]   # Continuation of the Heaviside projection
ETA_H = 0.5             # Projection threshold
ITER_PER_BETA = 40      # Iteration cap per beta stage in the benchmark
TOL_CHANGE = 0.01       # A stage ends when max|d rho| < TOL_CHANGE ...
TOL_J = 2e-4            # ... or when J changes less than this over 5 iterations
MIN_CAVITY = 20         # [elements] Smaller enclosed voids are filled at binarization

BENCH_U_RATIO = 1.25    # Benchmark threshold: u_max <= 1.25 * u_max of the solid
BENCH_FS_MIN = 2.0      # Benchmark threshold: FS_p99.5 >= 2.0

act = elem_active                                      # Global indices of the elements of Omega
n_act = len(act)
V_act = V_elem[act]
V_ref = V0

pts = np.column_stack([centroid_r[act], centroid_z[act]])
d_min = np.full(n_act, np.inf)
for p1, p2 in contour_segments:
    v = p2 - p1; l2 = v @ v
    t_seg = np.clip(((pts - p1) @ v) / l2, 0.0, 1.0) if l2 > 1e-14 else np.zeros(n_act)
    d_min = np.minimum(d_min, np.linalg.norm(pts - (p1 + t_seg[:, None] * v), axis=1))
passive = d_min <= SHELL_THICKNESS
free_vars = ~passive

shell_fraction = V_act[passive].sum() / V_ref
if VOLFRAC <= shell_fraction:
    raise ValueError(f"VOLFRAC = {VOLFRAC:.2f} is below the volume fraction of the shell ({shell_fraction:.3f}).")
print(f"Design / passive elements: {free_vars.sum():,} / {passive.sum():,} | shell = {100 * shell_fraction:.1f} % of V0")

## 3.2 Axisymmetric Helmholtz filter and Heaviside projection (Eq. 3.9 to 3.11)

The filter solves $-r_H^2\nabla^2\tilde\rho + \tilde\rho = \rho$ with the measure $2\pi r\,dA$, so it smooths the density as a
three-dimensional filter on the solid of revolution, with $r_H = R_{min}/(2\sqrt{3})$ (Eq. 2.28). In discrete form:

$$K_F\phi = T\rho,\qquad \tilde\rho = D^{-1}T^T\phi,\qquad D = \mathrm{diag}(V_e).$$

$K_F$ does not depend on the design, so it is factorized once. Since $\sum_i T_{ie} = V_e$, a uniform density passes through unchanged.

The filtered density is projected with the smooth Heaviside function, and the volume constraint acts on the projected densities:

$$\bar\rho_e = \frac{\tanh(\beta\eta_H) + \tanh(\beta(\tilde\rho_e - \eta_H))}{\tanh(\beta\eta_H) + \tanh(\beta(1 - \eta_H))}.$$

In [ ]:
r_H = R_MIN / (2.0 * np.sqrt(3.0))
node_map = -np.ones(n_nodes, dtype=np.int64)
node_map[nodes_active] = np.arange(len(nodes_active))
conn_f = node_map[connectivity[act]]                   # Filter connectivity (n_act, 4)
n_fn = len(nodes_active)

kf = np.zeros((n_act, 4, 4)); tf = np.zeros((n_act, 4))
for xi in gp:
    for eta in gp:
        N_gp, dN_dr, dN_dz = shape_functions(xi, eta)
        w_gp = 2.0 * np.pi * (r_elem_nodes[act] @ N_gp) * detJ
        k_loc = r_H**2 * (np.outer(dN_dr, dN_dr) + np.outer(dN_dz, dN_dz)) + np.outer(N_gp, N_gp)
        kf += w_gp[:, None, None] * k_loc[None]
        tf += w_gp[:, None] * N_gp[None]

KF = sp.coo_matrix((kf.ravel(), (np.repeat(conn_f, 4, axis=1).ravel(), np.tile(conn_f, (1, 4)).ravel())),
                   shape=(n_fn, n_fn)).tocsc()
TF = sp.coo_matrix((tf.ravel(), (conn_f.ravel(), np.repeat(np.arange(n_act), 4))), shape=(n_fn, n_act)).tocsr()
KF_lu = splu(KF)
V_gauss = np.asarray(TF.sum(axis=0)).ravel()           # Equal to V_act (partition of unity)


def filter_density(x):
    """Design densities to filtered densities (Eq. 3.9)."""
    return (TF.T @ KF_lu.solve(TF @ x)) / V_gauss


def filter_transpose(g):
    """Transpose of the filter, used in the chain rule of the sensitivities (Eq. 3.14)."""
    return TF.T @ KF_lu.solve(TF @ (g / V_gauss))


def heaviside(x, beta):
    """Smooth Heaviside projection and its derivative (Eq. 3.11)."""
    den = np.tanh(beta * ETA_H) + np.tanh(beta * (1 - ETA_H))
    th = np.tanh(beta * (x - ETA_H))
    return (np.tanh(beta * ETA_H) + th) / den, beta * (1 - th**2) / den


def physical_density(x, beta):
    """Filter + projection, with the passive shell reset to 1 and zero derivative."""
    xb, dxb = heaviside(filter_density(x), beta)
    xb[passive], dxb[passive] = 1.0, 0.0
    return xb, dxb


err_filter = np.max(np.abs(filter_density(np.full(n_act, 0.37)) - 0.37))
print(f"r_H = {r_H:.4f} mm | {n_fn:,} filter nodes | error on a uniform field: {err_filter:.1e}")

# Precomputed sparsity pattern on the free DOFs, for a fast assembly
g2f = -np.ones(2 * n_nodes, dtype=np.int64)
g2f[dofs_free] = np.arange(len(dofs_free))
dofs_act = elem_dofs[act]
Ia = g2f[np.repeat(dofs_act, 8, axis=1).ravel()]
Ja = g2f[np.tile(dofs_act, (1, 8)).ravel()]
mask_ff = (Ia >= 0) & (Ja >= 0)
Ia, Ja = Ia[mask_ff], Ja[mask_ff]
ke_act = ke_all[act]
ke_flat = ke_act.reshape(n_act, 64)
F_f = F_vector[dofs_free]
n_f = len(dofs_free)


def solve_fea(E_rel):
    """Assembles K from the relative moduli of the elements of Omega and solves the benchmark load."""
    vals = (ke_flat * E_rel[:, None]).ravel()[mask_ff]
    K = sp.csc_matrix((vals, (Ia, Ja)), shape=(n_f, n_f))
    U = np.zeros(2 * n_nodes)
    U[dofs_free] = solve_sparse(K, F_f)
    return U

## 3.3 SIMP loop with the Optimality Criteria update (Eq. 3.13 to 3.15)

The stiffness follows $C(\bar\rho) = [E_{min} + \bar\rho^p(1 - E_{min})]C_0$ (Eq. 2.25), and the compliance is self-adjoint, so:

$$\frac{\partial J}{\partial \bar\rho_e} = -p\,\bar\rho_e^{\,p-1}(1 - E_{min})\,u_e^T k_e^0 u_e .$$

Gradients are carried back through the projection and the filter, and the design variables are updated with

$$\rho_e^{(k+1)} = \mathrm{clip}\left(\rho_e^{(k)}\sqrt{\frac{-\partial J/\partial\rho_e}{\lambda\,\partial V/\partial\rho_e}},\ \rho_e^{lo},\ \rho_e^{up}\right),\qquad
\Delta\rho_{lim} = \max\left(0.05,\ 0.15\min(1, 2/\beta)\right).$$

The multiplier $\lambda$ is found by bisection, evaluating the filter and the projection for each trial, so the constraint holds on
the projected densities. The volume gradient carries $V_e$, which makes material at large radius more expensive.

In [ ]:
x = np.where(passive, 1.0, (VOLFRAC * V_ref - V_act[passive].sum()) / V_act[free_vars].sum())
hist = {"J": [], "vol": [], "change": [], "beta": [], "gray": []}
it_total, t_loop = 0, time.time()

for beta in BETAS:
    for it_b in range(ITER_PER_BETA):
        xb, dxb = physical_density(x, beta)
        E_rel = E_MIN + xb**PENAL * (1 - E_MIN)
        U = solve_fea(E_rel)
        ue = U[dofs_act]
        ce = np.einsum("ei,eij,ej->e", ue, ke_act, ue)          # Element energies u_e^T k_e^0 u_e
        J = float(F_vector @ U)

        # Sensitivities with respect to the design variables (Eq. 3.13 and 3.14)
        dJ_dxb = -PENAL * (1 - E_MIN) * xb**(PENAL - 1) * ce
        dJ = filter_transpose(dJ_dxb * dxb)
        dV = filter_transpose(V_act * dxb)
        dJ[passive], dV[passive] = 0.0, 0.0
        dJ = np.minimum(dJ, -1e-30); dV = np.maximum(dV, 1e-30)

        # OC update with bisection on the projected, volume-weighted constraint (Eq. 3.15)
        l1, l2 = 1e-12, 1e12
        move_b = max(0.05, MOVE * min(1.0, 2.0 / beta))
        lo, hi = np.maximum(0.0, x - move_b), np.minimum(1.0, x + move_b)
        while (l2 - l1) / (l1 + l2) > 1e-4:
            lm = np.sqrt(l1 * l2)
            x_new = np.clip(x * np.sqrt(-dJ / (lm * dV)), lo, hi)
            x_new[passive] = 1.0
            if physical_density(x_new, beta)[0] @ V_act > VOLFRAC * V_ref:
                l1 = lm
            else:
                l2 = lm
        change = float(np.max(np.abs(x_new - x)))
        x = x_new
        it_total += 1

        gray = float(np.sum(4 * xb * (1 - xb) * V_act) / V_act.sum())   # Measure of non-discreteness, Eq. (3.12)
        hist["J"].append(J); hist["vol"].append(float(xb @ V_act / V_ref))
        hist["change"].append(change); hist["beta"].append(beta); hist["gray"].append(gray)
        if it_total == 1 or it_total % 10 == 0:
            print(f"  it {it_total:4d} | beta = {beta:>2} | J/J_0 = {J / J_0:6.3f} | V/V_0 = {hist['vol'][-1]:.4f} | "
                  f"gray = {100 * gray:5.2f} % | change = {change:.4f}")
        dJ_rel = abs(hist["J"][-1] - hist["J"][-6]) / hist["J"][-1] if it_b >= 5 else np.inf
        if it_b > 4 and (change < TOL_CHANGE or dJ_rel < TOL_J):
            print(f"  it {it_total:4d} | beta = {beta:>2} stage ended (change = {change:.4f}, dJ/J = {dJ_rel:.1e})")
            break

rho_bar_opt = physical_density(x, BETAS[-1])[0]
bench_time = time.time() - t_loop
print(f"SIMP loop: {it_total} iterations in {bench_time:.1f} s | final non-discreteness {100 * hist['gray'][-1]:.2f} %")

## 3.4 Binary design (Sec. 3.5.3)

The projected field is thresholded at $\bar\rho_e \geq 0.5$ and the shell is added. The image is labeled with **4-connectivity**, so
elements sharing only a corner are not connected. Only components connected to the shell or the supports are kept, and enclosed voids
smaller than 20 elements are filled. All reported quantities come from this binary design, with void nodes removed from the system.

In [ ]:
dom_2d = (rho_0 == 1.0).reshape(nelz, nelr)
shell_global = np.zeros(n_elem, dtype=bool); shell_global[act[passive]] = True
shell_2d = shell_global.reshape(nelz, nelr)
sol_global = np.zeros(n_elem); sol_global[act] = rho_bar_opt
sol_2d = ((sol_global.reshape(nelz, nelr) >= 0.5) | shell_2d) & dom_2d
st4 = ndimage.generate_binary_structure(2, 1)          # 4-adjacency

# (a) Keep only what is connected to the shell or to the supports
anchors = shell_2d.copy()
anchors.ravel()[np.unique(np.concatenate([np.where(np.isin(connectivity, nodes_clamped).any(1))[0],
                                          np.where(np.isin(connectivity, nodes_sliding).any(1))[0]]))] = True
labels, n_comp = ndimage.label(sol_2d, structure=st4)
ids_ok = np.unique(labels[anchors & sol_2d]); ids_ok = ids_ok[ids_ok > 0]
n_islands_removed = n_comp - len(ids_ok)
sol_2d = np.isin(labels, ids_ok)

# (b) Fill very small enclosed voids
labels_v, _ = ndimage.label(dom_2d & ~sol_2d, structure=st4)
size_v = np.bincount(labels_v.ravel())
small = np.where((np.arange(len(size_v)) > 0) & (size_v < MIN_CAVITY))[0]
sol_2d[np.isin(labels_v, small)] = True
_, n_comp_final = ndimage.label(sol_2d, structure=st4)
rho_opt_bin = sol_2d.ravel().astype(np.float64)

# Solution of the binary design: void means no material, not E_min
e_opt = np.where(rho_opt_bin == 1.0)[0]
n_opt = np.unique(connectivity[e_opt])
dofs_f_opt = np.setdiff1d(np.arange(2 * n_nodes), np.unique(np.concatenate([
    2 * np.intersect1d(nodes_sliding, n_opt) + 1,
    2 * np.intersect1d(nodes_clamped, n_opt), 2 * np.intersect1d(nodes_clamped, n_opt) + 1,
    2 * np.setdiff1d(np.arange(n_nodes), n_opt), 2 * np.setdiff1d(np.arange(n_nodes), n_opt) + 1])))
K_opt = sp.coo_matrix((K_val_base * np.repeat(rho_opt_bin, 64), (I_idx, J_idx)), shape=(2 * n_nodes, 2 * n_nodes)).tocsc()
F_opt = F_vector.copy()
F_opt[np.setdiff1d(np.arange(2 * n_nodes), dofs_f_opt)] = 0.0
U_opt = np.zeros(2 * n_nodes)
U_opt[dofs_f_opt] = spsolve(K_opt[dofs_f_opt, :][:, dofs_f_opt], F_opt[dofs_f_opt], permc_spec="MMD_AT_PLUS_A")
load_lost = 1.0 - (-F_opt[1::2].sum()) / F_total

sigma_VM_opt, sigma_opt = von_mises(U_opt)
sigma_VM_opt[rho_opt_bin == 0.0] = 0.0
umag_opt = np.hypot(U_opt[0::2], U_opt[1::2])
V_opt = float(V_elem @ rho_opt_bin)
mass_opt_g = rho_g_mm3 * V_opt
u_max_opt = float(umag_opt[n_opt].max())
J_opt = float(F_opt @ U_opt)
vm_o = sigma_VM_opt[e_opt]
vm_max_opt, vm_p995_opt = float(vm_o.max()), float(np.percentile(vm_o, 99.5))
e_vm_opt = e_opt[np.argmax(vm_o)]
FS_opt, FS_p995_opt = Sy_mat / vm_max_opt, Sy_mat / vm_p995_opt

## 3.5 Efficiency index of the benchmark (Eq. 3.16)

The benchmark uses its own thresholds, not the design criteria of Block 5. The index measures the stiffness gain per unit mass and
is used only here, because the component designs are selected by minimum mass subject to $U \leq 1$:

$$IE = \frac{J_{solid}\,m_0}{J_{opt}\,m_{opt}},\qquad IE_{val} = IE\cdot\min\left(1, \frac{1.25\,u_{solid}}{u_{opt}}\right)\cdot\min\left(1, \frac{FS_{p99.5}}{2.0}\right).$$

In [ ]:
r_m, r_J = mass_opt_g / mass_0_g, J_opt / J_0
r_u, r_s = u_max_opt / u_max_0, vm_p995_opt / vm_p995_0
IE = 1.0 / (r_J * r_m)
ok_u, ok_s = r_u <= BENCH_U_RATIO, FS_p995_opt >= BENCH_FS_MIN
IE_val = IE * min(1.0, BENCH_U_RATIO / r_u) * min(1.0, FS_p995_opt / BENCH_FS_MIN)
verdict = "PASSED" if (ok_u and ok_s) else "THRESHOLDS NOT MET"

metrics_base = dict(mass_g=mass_0_g, u_max=u_max_0, J=J_0, vm_max=vm_max_0, vm_p995=vm_p995_0, FS=FS_0, FS_p995=FS_p995_0)
metrics_opt = dict(mass_g=mass_opt_g, u_max=u_max_opt, J=J_opt, vm_max=vm_max_opt, vm_p995=vm_p995_opt, FS=FS_opt,
                   FS_p995=FS_p995_opt, IE=IE, IE_val=IE_val, verdict=verdict)

print(f"Islands removed: {n_islands_removed} | final components: {n_comp_final} | cavities filled: {len(small)} | load lost: {100 * load_lost:.3f} %")
print(pd.DataFrame({"Solid": [mass_0_g, V0, u_max_0, J_0, vm_max_0, vm_p995_0, FS_p995_0],
                    "Optimized": [mass_opt_g, V_opt, u_max_opt, J_opt, vm_max_opt, vm_p995_opt, FS_p995_opt]},
                   index=["Mass [g]", "Volume [mm3]", "u_max [mm]", "Compliance [N mm]", "VM max [MPa]", "VM p99.5 [MPa]", "FS p99.5 [-]"])
      .assign(ratio=lambda d: d["Optimized"] / d["Solid"]).round(4).to_string())
print(f"Mass saving: {100 * (1 - r_m):.2f} % | u ratio {r_u:.3f} (limit {BENCH_U_RATIO}) | FS {FS_p995_opt:.3f} (limit {BENCH_FS_MIN})")
print(f"IE = {IE:.4f} | IE_val = {IE_val:.4f} | {verdict}")

In [ ]:
# Convergence history and density field of the benchmark
fig, axes = plt.subplots(1, 2, figsize=(13, 6), dpi=110)
ax0 = axes[0]
its = np.arange(1, it_total + 1)
ax0.plot(its, np.array(hist["J"]) / J_0, "b-", lw=1.6, label="J / J_0")
ax0.set_xlabel("Iteration"); ax0.set_ylabel("J / J_0", color="b"); ax0.grid(True, ls="--", alpha=0.4)
ax0b = ax0.twinx()
ax0b.plot(its, hist["vol"], "r-", lw=1.2, label="V / V_0")
ax0b.plot(its, hist["gray"], "k:", lw=1.2, label="non-discreteness")
ax0b.axhline(VOLFRAC, color="r", ls="--", lw=0.8); ax0b.set_ylabel("V / V_0 | non-discreteness", color="r")
for i_change in np.where(np.diff(hist["beta"]) != 0)[0]:
    ax0.axvline(i_change + 1.5, color="grey", lw=0.7, ls="-.")
ax0.set_title("Convergence history (vertical lines: change of beta)", fontweight="bold")
ax0b.legend(loc="center right", fontsize=8); ax0.legend(loc="upper right", fontsize=8)
ax1 = axes[1]
pm = ax1.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(~dom_2d, sol_global.reshape(nelz, nelr)), cmap="gray_r", vmin=0, vmax=1, shading="flat")
ax1.add_collection(LineCollection(contour_segments, colors="crimson", linewidths=0.8))
fig.colorbar(pm, ax=ax1, fraction=0.046, pad=0.04).set_label(r"$\bar\rho$")
ax1.set_aspect("equal"); ax1.set_xlim(R_in - 4, R_out + 4); ax1.set_ylim(-4, H_part + 4)
ax1.set_xlabel("Radius r [mm]"); ax1.set_ylabel("Axial coordinate z [mm]"); ax1.grid(True, ls="--", alpha=0.4)
ax1.set_title(f"Projected density (V/V_0 = {hist['vol'][-1]:.3f})", fontweight="bold")
plt.tight_layout()
plt.show()

---
# Block 4. Benchmark figures and geometric reconstruction

## 4.1 Benchmark figures (Figure 5.3.1)

Solid and optimized design with common color scales, and each metric as a ratio to the solid.

In [ ]:
GEOM_LABEL = f"{ACTIVE_MATERIAL}_vf{int(round(100 * VOLFRAC))}"
umag_0_e = umag_0[connectivity].mean(axis=1)
umag_opt_e = umag_opt[connectivity].mean(axis=1)
u_lim = max(u_max_0, u_max_opt)
vm_lim = max(vm_p995_0, vm_p995_opt)
states = [("Solid", rho_0, umag_0_e, sigma_VM_0, mass_0_g), ("Optimized", rho_opt_bin, umag_opt_e, sigma_VM_opt, mass_opt_g)]

fig, axes = plt.subplots(2, 3, figsize=(16, 11), dpi=110)
for row, (name_s, rho_c, u_c, vm_c, m_c) in enumerate(states):
    sol = rho_c.reshape(nelz, nelr) >= 0.5
    ax = axes[row, 0]
    ax.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(~sol, np.ones((nelz, nelr))), cmap=ListedColormap(["black"]), shading="flat")
    ax.set_title(f"{name_s}: material layout, m = {m_c:,.1f} g")
    ax = axes[row, 1]
    pu = ax.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(~sol, u_c.reshape(nelz, nelr)), cmap="viridis", vmin=0, vmax=u_lim, shading="flat")
    fig.colorbar(pu, ax=ax, fraction=0.046, pad=0.04).set_label(r"$|\mathbf{u}|$ [mm]")
    ax.set_title(f"{name_s}: displacement magnitude")
    ax = axes[row, 2]
    pv = ax.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(~sol, vm_c.reshape(nelz, nelr)), cmap="jet", vmin=0, vmax=vm_lim, shading="flat")
    fig.colorbar(pv, ax=ax, fraction=0.046, pad=0.04, extend="max").set_label(r"$\sigma_{\mathrm{VM}}$ [MPa] (capped at p99.5)")
    ax.set_title(f"{name_s}: von Mises stress")
for ax in axes.ravel():
    ax.add_collection(LineCollection(contour_segments, colors="crimson", linewidths=0.7))
    ax.set_aspect("equal"); ax.set_xlim(R_in - 4, R_out + 4); ax.set_ylim(-4, H_part + 4)
    ax.set_xlabel("Radius $r$ [mm]"); ax.set_ylabel("Axial coordinate $z$ [mm]"); ax.grid(True, ls="--", alpha=0.35)
plt.tight_layout()
fig.savefig(DIR_BENCH / f"benchmark_comparison_{GEOM_LABEL}.png", dpi=300, bbox_inches="tight")
plt.show()

metric_names = ["Mass", "Compliance", "Max. displacement", r"$\sigma_{\mathrm{VM},p99.5}$", r"FS$_{p99.5}$", "Efficiency index"]
metric_ratios = [r_m, r_J, r_u, r_s, FS_p995_opt / FS_p995_0, IE]
better = [v <= 1 for v in metric_ratios[:4]] + [v >= 1 for v in metric_ratios[4:]]    # Green: better than the solid
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
bars = ax.bar(metric_names, metric_ratios, color=["#2ca02c" if b else "#d62728" for b in better], edgecolor="black", lw=0.5)
ax.bar_label(bars, fmt="%.3f", padding=4)
ax.axhline(1.0, color="black", ls="--", lw=1.2, label="Solid = 1")
ax.axhline(BENCH_U_RATIO, color="#ff7f0e", ls=":", lw=1.2, label=f"Displacement threshold = {BENCH_U_RATIO}")
ax.set_ylabel("Optimized / solid"); ax.set_ylim(0, 1.15 * max(metric_ratios))
ax.tick_params(axis="x", rotation=15); ax.grid(True, ls="--", alpha=0.4); ax.legend(loc="upper left")
ax.set_title(f"Benchmark, {mat['name']}, $V_f$ = {VOLFRAC:.2f}: IE = {IE:.2f}, IE$_{{\\mathrm{{val}}}}$ = {IE_val:.2f}")
plt.tight_layout()
fig.savefig(DIR_BENCH / f"benchmark_metrics_{GEOM_LABEL}.png", dpi=300, bbox_inches="tight")
plt.show()

## 4.2 Meridian contours (Sec. 3.7.1)

Each binary image becomes smooth closed polygons:

1. Gaussian smoothing with $\sigma = 0.7$ cells removes the staircase of inclined members.
2. A one-cell border of zeros closes the contours that touch the domain boundary.
3. Marching squares at level 0.5 extracts the iso-contours. The offset of 0.5 in the mapping compensates the padding.
4. Douglas-Peucker with a tolerance of 0.03 mm simplifies them, and duplicated and collinear vertices are removed.
5. Polygons are oriented by nesting depth, with the solid to the left of each edge: outer boundaries run counterclockwise and cavities
clockwise, as Eq. (2.37) requires.

In [ ]:
SMOOTH_SIGMA = 0.7     # [cells] Gaussian smoothing before marching squares
POLY_TOL = 0.03        # [mm] Douglas-Peucker tolerance


def clean_polygon(p, tol_rel=1e-9):
    """Removes duplicated and collinear vertices. Ear clipping discards them silently when triangulating the caps;
    if they stayed in the revolved contour, the seam between cap and lateral surface would remain open."""
    scale = max(np.ptp(p[:, 0]), np.ptp(p[:, 1]), 1e-12)
    changed = True
    while changed and len(p) > 3:
        a, b = np.roll(p, 1, axis=0), np.roll(p, -1, axis=0)
        cross = (p[:, 0] - a[:, 0]) * (b[:, 1] - a[:, 1]) - (p[:, 1] - a[:, 1]) * (b[:, 0] - a[:, 0])
        dup = np.all(np.abs(p - a) <= tol_rel * scale, axis=1)
        remove = dup | (np.abs(cross) <= tol_rel * scale**2)
        remove &= ~np.roll(remove, 1)      # At most one vertex of each consecutive pair
        changed = bool(remove.any())
        p = p[~remove]
    return p


def meridian_contours(rho_bin):
    """Closed polygons (r, z) oriented with the solid to the left, with their nesting depth and area."""
    field = rho_bin.reshape(nelz, nelr).astype(float)
    if SMOOTH_SIGMA > 0:
        field = ndimage.gaussian_filter(field, SMOOTH_SIGMA, mode="constant", cval=0.0)
    field = np.pad(field, 1, mode="constant", constant_values=0.0)
    polys = []
    for c_line in measure.find_contours(field, 0.5):
        pz = (c_line[:, 0] - 0.5) * dz
        pr = R_in + (c_line[:, 1] - 0.5) * dr
        p = measure.approximate_polygon(np.column_stack([pr, pz]), tolerance=POLY_TOL)
        if np.allclose(p[0], p[-1]):
            p = p[:-1]
        p = clean_polygon(p)
        if len(p) >= 3:
            polys.append(p)
    area = lambda p: 0.5 * np.sum(p[:, 0] * np.roll(p[:, 1], -1) - np.roll(p[:, 0], -1) * p[:, 1])
    depth = [sum(MplPath(q).contains_point(p[0]) for j, q in enumerate(polys) if j != i) for i, p in enumerate(polys)]
    out = []
    for p, d in zip(polys, depth):
        outer = (d % 2 == 0)
        if (area(p) > 0) != outer:
            p = p[::-1]
        out.append((p, d, abs(area(p))))
    return out


def pappus_volume(polys):
    """Exact volume of revolution of the oriented polygons (Eq. 2.37)."""
    V = 0.0
    for p, _, _ in polys:
        r1_, z1_ = p[:, 0], p[:, 1]; r2_, z2_ = np.roll(r1_, -1), np.roll(z1_, -1)
        V += np.pi * np.sum((r1_**2 + r1_ * r2_ + r2_**2) * (z2_ - z1_)) / 3.0
    return -V       # Solid to the left in (r, z) gives a negative sign in this integral

## 4.3 Revolution and closed meshes (Sec. 3.7.2)

The polygons are revolved by indexed copying: each vertex is copied to $N$ angular positions and each edge produces two triangles with
shared indices. For a full revolution the indices are taken modulo $N$, so the surface is closed by construction. A sector adds two planar
caps triangulated by ear clipping. The angular step follows from the chordal tolerance of Eq. (3.18):

$$R_{out}\left[1 - \cos(\Delta\theta/2)\right] \leq t_{ch}.$$

Every mesh is checked: each directed edge must appear exactly once, and its opposite edge must also be present.

In [ ]:
CHORD_TOL_RING = 0.005      # [mm] Largest facet-circle distance at R_out for 360 and 270 degrees
CHORD_TOL_SECTOR = 0.0005   # [mm] The same for the printed sectors


def n_angular_divisions(angle_deg, chord_tol, R=None):
    """Angular divisions so that the sagitta of each facet does not exceed chord_tol (Eq. 3.18)."""
    R = R_out if R is None else R
    dth = 2.0 * np.arccos(1.0 - chord_tol / R)
    return max(4, int(np.ceil(np.radians(angle_deg) / dth)))


def revolve(polys, angle_deg, n_div):
    """Indexed revolution of the oriented polygons, with two planar caps when angle_deg < 360."""
    full = angle_deg >= 360.0 - 1e-9
    n_rings = n_div if full else n_div + 1
    th = np.linspace(0.0, np.radians(angle_deg), n_div + 1)[:n_rings]
    V, F, base, offsets = [], [], 0, []
    for p, _, _ in polys:
        n = len(p)
        R = p[:, 0][None, :]; Z = p[:, 1][None, :]
        V.append(np.stack([R * np.cos(th)[:, None], R * np.sin(th)[:, None], np.repeat(Z, n_rings, 0)], -1).reshape(-1, 3))
        idx = lambda k_, i_: base + (k_ % n_rings) * n + (i_ % n)
        k_a, i_a = np.meshgrid(np.arange(n_div), np.arange(n), indexing="ij")
        k_a, i_a = k_a.ravel(), i_a.ravel()
        a0, b0, a1, b1 = idx(k_a, i_a), idx(k_a, i_a + 1), idx(k_a + 1, i_a), idx(k_a + 1, i_a + 1)
        F.append(np.column_stack([a0, b1, b0])); F.append(np.column_stack([a0, a1, b1]))
        offsets.append(base); base += n_rings * n
    if not full:
        outers = [i for i, (_, d, _) in enumerate(polys) if d % 2 == 0]
        for ie in outers:
            pe, de, _ = polys[ie]
            holes = [j for j, (ph, dh, _) in enumerate(polys) if dh == de + 1 and MplPath(pe).contains_point(ph[0])]
            loops_cap = [ie] + holes
            pts_cap = np.vstack([polys[j][0] for j in loops_cap])
            ends = np.cumsum([len(polys[j][0]) for j in loops_cap]).astype(np.uint32)
            tri = earcut.triangulate_float64(pts_cap, ends).reshape(-1, 3).astype(np.int64)
            a2 = ((pts_cap[tri[:, 1], 0] - pts_cap[tri[:, 0], 0]) * (pts_cap[tri[:, 2], 1] - pts_cap[tri[:, 0], 1]) -
                  (pts_cap[tri[:, 1], 1] - pts_cap[tri[:, 0], 1]) * (pts_cap[tri[:, 2], 0] - pts_cap[tri[:, 0], 0]))
            # The whole cap is oriented by its total area: nearly degenerate triangles have a sign that is only noise
            if a2.sum() < 0:
                tri = tri[:, ::-1]
            glob = np.concatenate([offsets[j] + np.arange(len(polys[j][0])) for j in loops_cap])
            n_per = np.concatenate([np.full(len(polys[j][0]), len(polys[j][0])) for j in loops_cap])
            t0 = glob[tri]
            tN = glob[tri] + (n_rings - 1) * n_per[tri]
            F.append(t0); F.append(tN[:, ::-1])
    return np.vstack(V), np.vstack(F)


def audit_mesh(V, F):
    """Closure check: every directed edge appears once and its opposite edge is present."""
    e = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]]).astype(np.int64)
    nV = len(V) + 1
    h, h_inv = e[:, 0] * nV + e[:, 1], e[:, 1] * nV + e[:, 0]
    unique = len(np.unique(h)) == len(h)
    closed = bool(np.all(np.isin(h, h_inv)))
    return closed, unique


def export_stl(V, F, path):
    """Writes the mesh, with the normals pointing outward according to its signed volume. Returns the volume."""
    m = stl_mesh.Mesh(np.zeros(len(F), dtype=stl_mesh.Mesh.dtype))
    m.vectors[:] = V[F]
    vol = float(m.get_mass_properties()[0])
    if vol < 0:
        F = F[:, ::-1]; m.vectors[:] = V[F]; vol = -vol
    m.save(str(path))
    return vol, F

## 4.4 Export of the benchmark (Table 3.7.1)

The solid and the optimized benchmark are exported at 360°, and the benchmark also as a 270° cut-away view of its internal structure.

In [ ]:
N_THETA_RING = n_angular_divisions(360.0, CHORD_TOL_RING)
CUT_ANGLE = 270.0
stl_records = []
for name_s, rho_c, V_ref_c in [("base", rho_0, V0), ("opt", rho_opt_bin, V_opt)]:
    polys = meridian_contours(rho_c)
    V_pap = pappus_volume(polys)
    sectors = [(360.0, N_THETA_RING)]
    if name_s == "opt":
        sectors.append((CUT_ANGLE, n_angular_divisions(CUT_ANGLE, CHORD_TOL_RING)))
    for angle, ndiv in sectors:
        Vx, Fx = revolve(polys, angle, ndiv)
        closed, unique = audit_mesh(Vx, Fx)
        suffix = "360" if angle >= 360 else f"cut{int(angle)}"
        file_name = f"{name_s}_{GEOM_LABEL}_{suffix}.stl"
        vol_stl, Fx = export_stl(Vx, Fx, DIR_STL / "benchmark" / file_name)
        stl_records.append(dict(file=file_name, angle=angle, n_div=ndiv, triangles=len(Fx), closed=bool(closed and unique), V_stl=vol_stl,
                                V_expected=V_ref_c * angle / 360.0, V_pappus=V_pap * angle / 360.0, contours=len(polys),
                                cavities=sum(d % 2 == 1 for _, d, _ in polys)))
        if name_s == "opt" and angle < 360:
            V_cut, F_cut = Vx, Fx

print(pd.DataFrame(stl_records).assign(err_pct=lambda d: 100 * (d.V_stl - d.V_expected) / d.V_expected).round(3).to_string(index=False))

# 3D view of the cut-away sector
fig = plt.figure(figsize=(9, 8), dpi=100)
ax3 = fig.add_subplot(111, projection="3d")
step_f = max(1, len(F_cut) // 60000)
tri3 = V_cut[F_cut[::step_f]]
nrm = np.cross(tri3[:, 1] - tri3[:, 0], tri3[:, 2] - tri3[:, 0])
nrm /= np.linalg.norm(nrm, axis=1, keepdims=True) + 1e-15
light = np.clip(nrm @ np.array([0.4, -0.6, 0.7]) / np.linalg.norm([0.4, -0.6, 0.7]), 0.15, 1.0)
ax3.add_collection3d(Poly3DCollection(tri3, facecolors=plt.cm.Blues(0.35 + 0.6 * light), edgecolors="none"))
ax3.set_xlim(-R_out, R_out); ax3.set_ylim(-R_out, R_out); ax3.set_zlim(0, H_part)
ax3.set_box_aspect((2 * R_out, 2 * R_out, H_part)); ax3.view_init(elev=22, azim=-60)
ax3.set_xlabel("x [mm]"); ax3.set_ylabel("y [mm]"); ax3.set_zlabel("z [mm]")
ax3.set_title(f"Optimized benchmark, {CUT_ANGLE:.0f} degree cut-away | {ACTIVE_MATERIAL}", fontweight="bold")
plt.tight_layout()
plt.show()

with open(DIR_BENCH / f"results_{GEOM_LABEL}.json", "w", encoding="utf-8") as fh:
    json.dump(dict(material=ACTIVE_MATERIAL, volfrac=VOLFRAC, penal=PENAL, R_min=R_MIN, shell_mm=SHELL_THICKNESS, alpha_G=ALPHA_G,
                   iterations=it_total, time_s=bench_time, base=metrics_base, optimized=metrics_opt, stl=stl_records),
              fh, indent=2, default=float)

---
# Block 5. Multi-load problem and search on the volume fraction

## 5.1 Design limits and control nodes (Sec. 4.1.3)

Each design must satisfy three limits in every load case, combined into the **utilization index** of Eq. (3.8):

$$U_c = \max\left(\frac{\max_{i\in\Gamma_{crit}}|u_{i,c}|}{u_{adm,crit}},\ \frac{\max_{i\in\Gamma_{rem}}|u_{i,c}|}{u_{adm,rem}},\ \frac{FS_{min}\,\sigma_{VM,p99.5,c}}{S_y}\right),\qquad U = \max_c U_c .$$

A design is feasible if $U \le 1$. The critical walls are the lower part of the bore and the upper part of the right cavity wall.

In [ ]:
MAT_KEYS = ["PA6", "PA6_CF"]
U_ADM_CRIT = 0.10       # [mm] Critical walls
U_ADM_REM = 0.50        # [mm] Remaining boundary
FS_VM = 1.5             # Sy / sigma_VM,p99.5 >= 1.5
ITER_PER_BETA_SEARCH = 30   # Iteration cap per beta stage in Blocks 5 and 6 (at most 150 per run)

CRIT_ZONE_1 = dict(r=(-np.inf, 58.6), z=(2.8, 10.6))    # Lower part of the inner bore
CRIT_ZONE_2 = dict(r=(90.2, 91.6), z=(68.0, 78.6))      # Upper part of the right cavity wall

# Boundary edges of the solid: element sides whose neighbor is void or lies outside the grid
pad = np.pad(rho_image, 1)
jj, ii = np.divmod(np.arange(n_elem), nelr)
solid = rho_0 == 1.0
void_at = lambda dj, di: pad[jj + 1 + dj, ii + 1 + di] == 0.0
v_below, v_above, v_left, v_right = solid & void_at(-1, 0), solid & void_at(1, 0), solid & void_at(0, -1), solid & void_at(0, 1)
nodes_boundary = np.unique(np.concatenate([connectivity[v_below][:, [0, 1]].ravel(), connectivity[v_above][:, [3, 2]].ravel(),
                                           connectivity[v_left][:, [0, 3]].ravel(), connectivity[v_right][:, [1, 2]].ravel()]))
r_bnd, z_bnd = r_nodes[nodes_boundary], z_nodes[nodes_boundary]
in_zone = lambda zone: (r_bnd >= zone["r"][0]) & (r_bnd <= zone["r"][1]) & (z_bnd >= zone["z"][0]) & (z_bnd <= zone["z"][1])
nodes_crit = nodes_boundary[in_zone(CRIT_ZONE_1) | in_zone(CRIT_ZONE_2)]
nodes_rem = np.setdiff1d(nodes_boundary, nodes_crit)
print(f"Critical nodes: {len(nodes_crit)} | remaining boundary nodes: {len(nodes_rem)}")

## 5.2 Load cases at unit pressure (Sec. 4.1.2, Table 4.1.1)

Loaded edges are found on the voxelized boundary and converted into consistent nodal forces (Eq. 2.24). Because they act on the passive shell,
the applied loads do not change during the optimization.

| Case | Loaded surfaces | Pressure [MPa] |
|---|---|---|
| C1, axial | Top face | $P = 0.390$ |
| C2, internal | U-cavity: bottom, left wall and right wall | $P_i = 0.344$ |
| C3, combined | Top face and U-cavity | $P = 0.225$, $P_i = 0.375$ |
| C4, radial | Inner bore and right cavity wall, towards $+r$ | $P_r = 0.368$ |

In [ ]:
rc_e, zc_e = centroid_r, centroid_z
top_face = v_above & (zc_e >= H_part - 1.5 * dz)
cavity_bottom = v_above & ~top_face & (rc_e >= 70.0) & (rc_e <= 96.0) & (zc_e >= 10.0) & (zc_e <= 40.0)
cavity_right = v_left & (rc_e >= 88.0) & (rc_e <= 97.0) & (zc_e >= 12.5)        # Pushes towards +r
bore_wall = v_left & ~cavity_right & (rc_e <= 63.0)                             # Pushes towards +r
cavity_left = solid & (ii < nelr - 1) & void_at(0, 1) & (rc_e >= 68.0) & (rc_e <= 76.0) & (zc_e >= 12.5)   # Pushes towards -r


def edge_force(mask, na, nb, nr, nz):
    """Consistent nodal forces of a unit pressure on straight edges (Eq. 2.24), along the direction (nr, nz)."""
    F = np.zeros(2 * n_nodes)
    a, b = connectivity[mask, na], connectivity[mask, nb]
    ra, rb = r_nodes[a], r_nodes[b]
    L = np.hypot(r_nodes[b] - ra, z_nodes[b] - z_nodes[a])
    fa, fb = 2 * np.pi * L * (2 * ra + rb) / 6, 2 * np.pi * L * (ra + 2 * rb) / 6
    for nod, f in ((a, fa), (b, fb)):
        np.add.at(F, 2 * nod, f * nr); np.add.at(F, 2 * nod + 1, f * nz)
    return F


F_top, F_bottom = edge_force(top_face, 3, 2, 0, -1), edge_force(cavity_bottom, 3, 2, 0, -1)
F_right, F_bore = edge_force(cavity_right, 0, 3, 1, 0), edge_force(bore_wall, 0, 3, 1, 0)
F_left = edge_force(cavity_left, 1, 2, -1, 0)
F_int = F_bottom + F_right + F_left

# The design pressures are nominal, since the drawing gives no service loads (Sec. 4.1.2). They were set from these starting
# values with common factors lam_c, chosen so that the solid of the most demanding material uses about half of its capacity.
# Linear elasticity makes this scaling exact (Eq. 2.21).
P_NOM = {"comp": 5.0, "int": 8.0, "comp_comb": 3.0, "int_comb": 5.0, "radial": 8.0}
CASE_NAMES = ["C1 Axial compression", "C2 Internal pressure", "C3 Combined", "C4 Radial pressure +r"]
KAPPA_LOAD = 0.50
F_nom = np.column_stack([P_NOM["comp"] * F_top,
                         P_NOM["int"] * F_int,
                         P_NOM["comp_comb"] * F_top + P_NOM["int_comb"] * F_int,
                         P_NOM["radial"] * (F_right + F_bore)])
n_cases = F_nom.shape[1]

## 5.3 Evaluation of a binary design

`evaluate_design` solves a binary design in all the load cases. The problem is linear, so all cases share one factorization. Void elements and
their nodes are removed from the system. `tensor_operators` prepares the element stiffness and the stress operator of each material.

In [ ]:
def tensor_operators(m):
    """Element stiffness matrices and centroid stress operator of a material, for the elements of Omega."""
    S_m = compliance_matrix(m["E_r"], m["E_z"], m["nu_rtheta"], m["nu_rz"], m["nu_zr"], m["G_rz"])
    C_m = np.linalg.inv(S_m); C_m = 0.5 * (C_m + C_m.T)
    ke = np.zeros((len(act), 8, 8))
    rn = r_elem_nodes[act]
    for xi in gp:
        for eta in gp:
            N_gp, dN_dr, dN_dz = shape_functions(xi, eta)
            r_gp = rn @ N_gp
            B = np.zeros((len(act), 4, 8))
            B[:, 0, 0::2] = dN_dr; B[:, 1, 1::2] = dN_dz
            B[:, 2, 0::2] = N_gp[None, :] / r_gp[:, None]
            B[:, 3, 0::2] = dN_dz; B[:, 3, 1::2] = dN_dr
            ke += np.einsum("eji,jk,ekl->eil", B, C_m, B) * (2 * np.pi * r_gp * detJ)[:, None, None]
    return dict(C=C_m, ke=ke, ke_flat=ke.reshape(len(act), 64), CB=np.einsum("ij,ejk->eik", C_m, B_centroid[act]),
                Sy=m["Sy"], rho=m["rho_g_mm3"])


MAT_OPS = {k: tensor_operators(MATERIALS[k]) for k in MAT_KEYS}
I_act = I_idx.reshape(n_elem, 64)[act].ravel()
J_act = J_idx.reshape(n_elem, 64)[act].ravel()


def evaluate_design(rho_bin, T, F_full):
    """Responses of a binary design in every load case: displacements, stress percentile, compliance and utilization (Eq. 3.8)."""
    sol_act = rho_bin[act] >= 0.5
    e_sol = act[sol_act]
    n_sol = np.unique(connectivity[e_sol])
    inactive = np.setdiff1d(np.arange(n_nodes), n_sol)
    fixed = np.unique(np.concatenate([2 * np.intersect1d(nodes_sliding, n_sol) + 1,
                                      2 * np.intersect1d(nodes_clamped, n_sol),
                                      2 * np.intersect1d(nodes_clamped, n_sol) + 1, 2 * inactive, 2 * inactive + 1]))
    free = np.setdiff1d(np.arange(2 * n_nodes), fixed)
    K = sp.coo_matrix(((T["ke_flat"] * sol_act[:, None]).ravel(), (I_act, J_act)),
                      shape=(2 * n_nodes, 2 * n_nodes)).tocsc()[free, :][:, free]
    U = np.zeros((2 * n_nodes, F_full.shape[1]))
    U[free] = solve_sparse(K, F_full[free])
    out = []
    for c_i in range(F_full.shape[1]):
        um = np.hypot(U[0::2, c_i], U[1::2, c_i])
        s = np.einsum("eik,ek->ei", T["CB"][sol_act], U[elem_dofs[e_sol], c_i])
        vm = np.sqrt(0.5 * ((s[:, 0] - s[:, 1])**2 + (s[:, 1] - s[:, 2])**2 + (s[:, 2] - s[:, 0])**2) + 3 * s[:, 3]**2)
        u_c = float(um[np.intersect1d(nodes_crit, n_sol)].max()) if len(nodes_crit) else 0.0
        u_r = float(um[np.intersect1d(nodes_rem, n_sol)].max())
        p995 = float(np.percentile(vm, 99.5))
        out.append(dict(u_crit=u_c, u_rem=u_r, vm_p995=p995, vm_max=float(vm.max()), J=float(F_full[:, c_i] @ U[:, c_i]),
                        FS=T["Sy"] / p995, util=max(u_c / U_ADM_CRIT, u_r / U_ADM_REM, FS_VM * p995 / T["Sy"])))
    return out, U


# Common load factors: lam_c = KAPPA * min over the materials of (1 / utilization of the solid), rounded down to two digits
eval_nom = {k: evaluate_design(rho_0, MAT_OPS[k], F_nom)[0] for k in MAT_KEYS}
lam_solid = np.array([[1.0 / eval_nom[k][c_i]["util"] for c_i in range(n_cases)] for k in MAT_KEYS])
lam_c = KAPPA_LOAD * lam_solid.min(axis=0)
lam_c = np.array([float(f"{l:.2g}") if float(f"{l:.2g}") <= l else float(f"{l * 0.95:.2g}") for l in lam_c])
F_design = F_nom * lam_c[None, :]                      # Design loads of Table 4.1.1
eval_solid = {k: evaluate_design(rho_0, MAT_OPS[k], F_design)[0] for k in MAT_KEYS}

pressures_text = [f"P = {P_NOM['comp'] * lam_c[0]:.3g} MPa", f"Pi = {P_NOM['int'] * lam_c[1]:.3g} MPa",
                  f"P = {P_NOM['comp_comb'] * lam_c[2]:.3g} MPa, Pi = {P_NOM['int_comb'] * lam_c[2]:.3g} MPa",
                  f"Pr = {P_NOM['radial'] * lam_c[3]:.3g} MPa"]
# The pressures must be those of Table 4.1.1
assert np.allclose(lam_c * np.array([5.0, 8.0, 3.0, 8.0]), [0.390, 0.344, 0.225, 0.368]), "Load factors differ from Table 4.1.1"
print(pd.DataFrame([dict(case=CASE_NAMES[c_i], pressures=pressures_text[c_i], axial_force_N=float(F_design[1::2, c_i].sum()),
                         U_PA6=eval_solid["PA6"][c_i]["util"], U_PA6_CF=eval_solid["PA6_CF"][c_i]["util"]) for c_i in range(n_cases)])
      .round(3).to_string(index=False))

## 5.4 Inner level: SIMP with weighted compliance (Sec. 3.1, Eq. 2.35)

The same engine of Block 3 minimizes $J_w = \sum_c w_c F_c^T U_c$. Since $J_w$ is linear in the compliances the problem remains self-adjoint, and all
the cases share one factorization. `binarize` applies the rules of Section 3.4 to the final field.

In [ ]:
F_design_f = F_design[dofs_free]


def binarize(rho_bar):
    """Binary design: threshold 0.5 plus shell, connected components kept, small enclosed voids filled (Sec. 3.5.3)."""
    sg = np.zeros(n_elem); sg[act] = rho_bar
    s2 = ((sg.reshape(nelz, nelr) >= 0.5) | shell_2d) & dom_2d
    st4_ = ndimage.generate_binary_structure(2, 1)
    anchor = shell_2d.copy()
    anchor.ravel()[np.where(np.isin(connectivity, np.concatenate([nodes_clamped, nodes_sliding])).any(1))[0]] = True
    lab, _ = ndimage.label(s2, structure=st4_)
    ok = np.unique(lab[anchor & s2]); s2 = np.isin(lab, ok[ok > 0])
    lab_v, _ = ndimage.label(dom_2d & ~s2, structure=st4_)
    size_ = np.bincount(lab_v.ravel())
    s2[np.isin(lab_v, np.where((np.arange(len(size_)) > 0) & (size_ < MIN_CAVITY))[0])] = True
    return s2.ravel().astype(float)


def simp_weighted(T, w, volfrac, x0=None):
    """SIMP run at a fixed volume fraction, minimizing the weighted compliance. Returns x, rho_bar, iterations and J_w."""
    if x0 is None:
        x_ = np.where(passive, 1.0, (volfrac * V_ref - V_act[passive].sum()) / V_act[free_vars].sum())
    else:
        x_ = x0.copy()
    x_ = np.clip(x_, 0, 1); n_it = 0; Jw = np.nan
    for beta_ in BETAS:
        hist_J = []
        for it_b_ in range(ITER_PER_BETA_SEARCH):
            xb_, dxb_ = physical_density(x_, beta_)
            Er = E_MIN + xb_**PENAL * (1 - E_MIN)
            K = sp.csc_matrix(((T["ke_flat"] * Er[:, None]).ravel()[mask_ff], (Ia, Ja)), shape=(n_f, n_f))
            U_ = np.zeros((2 * n_nodes, n_cases)); U_[dofs_free] = solve_sparse(K, F_design_f)
            ue_ = U_[dofs_act]                                      # (n_act, 8, n_cases)
            ce_ = np.einsum("eic,eij,ejc->ec", ue_, T["ke"], ue_)
            Jw = float(np.sum(w * np.einsum("ic,ic->c", F_design, U_)))
            dJ_ = filter_transpose(-PENAL * (1 - E_MIN) * xb_**(PENAL - 1) * (ce_ @ w) * dxb_)
            dV_ = filter_transpose(V_act * dxb_)
            dJ_[passive], dV_[passive] = 0, 0
            dJ_ = np.minimum(dJ_, -1e-30); dV_ = np.maximum(dV_, 1e-30)
            mv = max(0.05, MOVE * min(1.0, 2.0 / beta_))
            lo_, hi_ = np.maximum(0, x_ - mv), np.minimum(1, x_ + mv)
            l1_, l2_ = 1e-12, 1e12
            while (l2_ - l1_) / (l1_ + l2_) > 1e-4:
                lm_ = np.sqrt(l1_ * l2_)
                xn = np.clip(x_ * np.sqrt(-dJ_ / (lm_ * dV_)), lo_, hi_); xn[passive] = 1.0
                if physical_density(xn, beta_)[0] @ V_act > volfrac * V_ref:
                    l1_ = lm_
                else:
                    l2_ = lm_
            change_ = np.max(np.abs(xn - x_)); x_ = xn; n_it += 1; hist_J.append(Jw)
            if it_b_ > 4 and (change_ < TOL_CHANGE or abs(hist_J[-1] - hist_J[-6]) / hist_J[-1] < TOL_J):
                break
    return x_, physical_density(x_, BETAS[-1])[0], n_it, Jw

## 5.5 Storage of the searches

The binary designs are the main product of the searches and cannot be recovered from the figures. They are stored in `results/designs/`: the
history of all the runs in `search_history.json` and every binary design, packed, in `designs.npz`. With `RUN_SEARCHES = False` the notebook reads them.

In [ ]:
def _clean_run(q):
    """Serializable record of a run, without the design arrays."""
    return {kk: (v.tolist() if hasattr(v, "tolist") else v) for kk, v in q.items() if kk not in ("x", "rho_bin")}


def save_searches(pareto_multi, selected_multi, pareto_cases, selected_cases):
    """Stores the history of every run and all the binary designs of Blocks 5 and 6."""
    arrays, history = {}, {}
    def add(group, key_txt, pts, chosen):
        history.setdefault(group, {})[key_txt] = dict(selected_index=None if chosen is None else next(i for i, q in enumerate(pts) if q is chosen),
                                                      runs=[_clean_run(q) for q in pts])
        for i, q in enumerate(pts):
            arrays[f"{group}|{key_txt}|{i}"] = np.packbits(q["rho_bin"] >= 0.5)
    for (k, prob), pts in pareto_multi.items():
        add("multi_load", k, pts, selected_multi[(k, prob)])
    for (k, c_i), pts in pareto_cases.items():
        add("single_case", f"{k}|{c_i}", pts, selected_cases[(k, c_i)])
    np.savez_compressed(DIR_DESIGNS / "designs.npz", **arrays)
    with open(DIR_DESIGNS / "search_history.json", "w", encoding="utf-8") as fh:
        json.dump(dict(material_keys=MAT_KEYS, case_names=CASE_NAMES, lambda_cases=lam_c.tolist(), pressures=pressures_text, searches=history), fh, indent=1)


def load_searches():
    """Rebuilds the dictionaries of runs and of selected designs from results/designs/."""
    data = np.load(DIR_DESIGNS / "designs.npz")
    with open(DIR_DESIGNS / "search_history.json", encoding="utf-8") as fh:
        history = json.load(fh)["searches"]
    def rebuild(group, key_txt):
        rec = history[group][key_txt]
        pts = []
        for i, q in enumerate(rec["runs"]):
            q = dict(q); q["rho_bin"] = np.unpackbits(data[f"{group}|{key_txt}|{i}"])[:n_elem].astype(float)
            pts.append(q)
        return pts, (None if rec["selected_index"] is None else pts[rec["selected_index"]])
    p_m, s_m, p_c, s_c = {}, {}, {}, {}
    for k in history["multi_load"]:
        p_m[(k, "Multi-load")], s_m[(k, "Multi-load")] = rebuild("multi_load", k)
    for key_txt in history["single_case"]:
        k, c_i = key_txt.split("|")
        p_c[(k, int(c_i))], s_c[(k, int(c_i))] = rebuild("single_case", key_txt)
    return p_m, s_m, p_c, s_c


def print_runs(pts, label):
    """Prints the runs of a search in the order of decreasing volume fraction."""
    print(f"-> {label}")
    for q in sorted(pts, key=lambda q_: -q_["vf"]):
        print(f"   Vf = {q['vf']:.4f} | mass = {q['mass']:8.1f} g | U = {q['util']:7.3f} | {'feasible' if q['feasible'] else 'infeasible':<10} | {q['nit']} it, {q['t']:.0f} s")

## 5.6 Outer level: search on the volume fraction (Sec. 3.6)

1. **Descending sweep** over $V_f = 0.90, 0.80, 0.70, 0.60, 0.50, 0.45, 0.40, 0.35, 0.30$, each run starting from the converged design of the previous one. Fractions
with $V_fV_0 \le 1.02\,V_{shell}$ are skipped. The sweep stops after two consecutive infeasible designs.
2. **Three bisection steps** between the lowest feasible fraction and the highest infeasible fraction below it, each starting from the current feasible design.
3. **Selection** of the feasible binary design of minimum mass (Eq. 2.36).

Compliance does not directly control the critical walls, so the weights are adapted after each run, favoring the cases with the highest utilization (Eq. 3.17):

$$w_c \leftarrow \tfrac12 w_c + \tfrac12\,\frac{U_c^q/J_{c,solid}}{\sum_k U_k^q/J_{k,solid}},\qquad q = 2.$$

In [ ]:
VF_SWEEP = [0.90, 0.80, 0.70, 0.60, 0.50, 0.45, 0.40, 0.35, 0.30]
N_BISECTION = 3
Q_WEIGHTS = 2.0


def multi_load_search(k):
    """Search on the volume fraction of the multi-load problem for material k, with adaptive weights."""
    T = MAT_OPS[k]
    J_solid = np.array([eval_solid[k][c_i]["J"] for c_i in range(n_cases)])
    w = (1.0 / J_solid) / (1.0 / J_solid).sum()         # Initial weights: each case counts the same for the solid
    log(f"\n-> [{k}] Multi-load: search on the volume fraction ({SOLVER_NAME})")

    def run(vf, x0):
        nonlocal w
        t0 = time.time()
        x_, xb_, nit, Jw = simp_weighted(T, w, vf, x0)
        rb = binarize(xb_)
        ev, _ = evaluate_design(rb, T, F_design)
        util = max(e["util"] for e in ev)
        p = dict(vf=vf, x=x_, rho_bin=rb, mass=float(T["rho"] * V_elem @ rb), util=util, feasible=util <= 1.0, eval=ev, Jw=Jw, nit=nit,
                 t=time.time() - t0, w=w.copy())
        log(f"   Vf = {vf:.4f} | mass = {p['mass']:8.1f} g | U = {util:6.3f} | {'feasible' if p['feasible'] else 'infeasible'} | "
            f"{nit} it, {p['t']:.0f} s | w = [{', '.join(f'{v:.2f}' for v in w)}]")
        u_c = np.array([e["util"] for e in ev])         # Weights for the next run (Eq. 3.17)
        w_new = (u_c ** Q_WEIGHTS) / J_solid
        w = 0.5 * w + 0.5 * w_new / w_new.sum()
        return p

    pts, x_prev, infeasible_row = [], None, 0
    for vf in VF_SWEEP:
        if vf * V_ref <= V_act[passive].sum() * 1.02:
            continue
        p = run(vf, x_prev); pts.append(p); x_prev = p["x"]
        infeasible_row = 0 if p["feasible"] else infeasible_row + 1
        if infeasible_row >= 2:
            break
    feas = [q for q in pts if q["feasible"]]
    infeas = [q for q in pts if not q["feasible"] and feas and q["vf"] < min(f_["vf"] for f_ in feas)]
    if feas and infeas:
        p_ok, p_fail = min(feas, key=lambda q: q["vf"]), max(infeas, key=lambda q: q["vf"])
        for _ in range(N_BISECTION):
            p = run(0.5 * (p_ok["vf"] + p_fail["vf"]), p_ok["x"]); pts.append(p)
            if p["feasible"]:
                p_ok = p
            else:
                p_fail = p
    pts.sort(key=lambda q: q["mass"])
    feas = [q for q in pts if q["feasible"]]
    return pts, (min(feas, key=lambda q: q["mass"]) if feas else None)


if RUN_SEARCHES:
    SEARCH_LOG.unlink(missing_ok=True)
    t_b5 = time.time()
    pareto, selected = {}, {}
    for k in MAT_KEYS:
        pareto[(k, "Multi-load")], selected[(k, "Multi-load")] = multi_load_search(k)
    log(f"Block 5 searches completed in {(time.time() - t_b5) / 60:.1f} min")
else:
    pareto, selected, _p, _s = load_searches()
    for key_s, pts_s in pareto.items():
        print_runs(pts_s, f"[{key_s[0]}] Multi-load (loaded from results/designs/)")

In [ ]:
# Minimum-mass feasible designs of the multi-load problem, in the four cases
rows = []
for (k, prob), p in selected.items():
    m0 = MAT_OPS[k]["rho"] * V0
    for c_i in range(n_cases):
        e = p["eval"][c_i]
        rows.append(dict(material=k, Vf=p["vf"], mass_g=p["mass"], saving_pct=100 * (1 - p["mass"] / m0), case=f"C{c_i + 1}",
                         u_crit=e["u_crit"], u_rem=e["u_rem"], vm_p995=e["vm_p995"], FS=e["FS"], U=e["util"]))
print(pd.DataFrame(rows).round(4).to_string(index=False))


def non_dominated_front(pts):
    """Non-dominated pairs of mass and utilization (Sec. 2.8)."""
    pts = sorted(pts, key=lambda q: (q["mass"], q["util"]))
    fr, best = [], np.inf
    for q in pts:
        if q["util"] < best:
            fr.append(q); best = q["util"]
    return fr


# Mass-utilization fronts of the multi-load searches
col_mat = {"PA6": "#1f77b4", "PA6_CF": "#d62728"}
fig, ax = plt.subplots(figsize=(10, 6), dpi=110)
for (k, prob), pts in pareto.items():
    cm = col_mat[k]
    ms = np.array([q["mass"] for q in pts]); us = np.array([q["util"] for q in pts]); fa = np.array([q["feasible"] for q in pts])
    ax.scatter(ms[fa], us[fa], color=cm, marker="o", s=40, zorder=3)
    ax.scatter(ms[~fa], us[~fa], facecolors="none", edgecolors=cm, marker="o", s=40, zorder=3)
    fr = non_dominated_front(pts)
    ax.plot([q["mass"] for q in fr], [q["util"] for q in fr], "-", color=cm, lw=1.5, label=f"{k} | {prob}")
    q = selected[(k, prob)]
    ax.scatter(q["mass"], q["util"], marker="*", s=260, color=cm, edgecolors="k", zorder=4)
    ax.scatter(MAT_OPS[k]["rho"] * V0, max(eval_solid[k][c_i]["util"] for c_i in range(n_cases)), marker="s", s=70, color=cm, edgecolors="k", zorder=4)
ax.axhline(1.0, color="k", ls=":", lw=1.2, label="feasibility limit, U = 1")
ax.set_xlabel("Mass [g]"); ax.set_ylabel("Governing utilization U"); ax.set_ylim(0, 2)
ax.set_title("Mass-utilization fronts (square: solid, filled: feasible, open: infeasible, star: selected)", fontweight="bold")
ax.grid(True, ls="--", alpha=0.4); ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(DIR_SEARCH / "pareto_multi_load.png", dpi=300, bbox_inches="tight")
plt.show()

---
# Block 6. Single-case problems and cross-utilization

## 6.1 One search per load case (Sec. 3.6, Table 3.5.1)

The single-case problems consider one case at a time and show whether the multi-load formulation is necessary. Each sweep starts from
$V_{f,multi} + 0.10$ rounded up to a multiple of 0.05, at most 0.90, in steps of 0.05, with the same bisection and selection rules. If its start
is already infeasible, the fraction is raised until a design is feasible.

In [ ]:
VF_STEP = 0.05
VF_MAX = 0.90
VF_MARGIN = 0.10
v_shell = V_act[passive].sum()


def run_single_case(T, c_i, vf, x0):
    """SIMP run for one load case. The design is evaluated in all the cases, for the cross-utilization."""
    t0 = time.time()
    w = np.zeros(n_cases); w[c_i] = 1.0
    x_, xb_, nit, Jw = simp_weighted(T, w, vf, x0)
    rb = binarize(xb_)
    ev, _ = evaluate_design(rb, T, F_design)
    p = dict(vf=vf, x=x_, rho_bin=rb, mass=float(T["rho"] * V_elem @ rb), util=ev[c_i]["util"], feasible=ev[c_i]["util"] <= 1.0,
             eval=ev, Jw=Jw, nit=nit, t=time.time() - t0)
    log(f"   C{c_i + 1} | Vf = {vf:.4f} | mass = {p['mass']:8.1f} g | U = {p['util']:6.3f} | {'feasible' if p['feasible'] else 'infeasible'} | {nit} it, {p['t']:.0f} s")
    return p


def single_case_search(k, c_i, vf_start):
    """Search on the volume fraction for load case c_i and material k."""
    T = MAT_OPS[k]
    log(f"\n-> [{k}] {CASE_NAMES[c_i]} (sweep from Vf = {vf_start:.2f})")
    pts, x_prev, infeasible_row = [], None, 0
    for vf in np.round(np.arange(vf_start, 0.0, -VF_STEP), 4):
        if vf * V_ref <= v_shell * 1.02:
            break
        p = run_single_case(T, c_i, vf, x_prev); pts.append(p); x_prev = p["x"]
        infeasible_row = 0 if p["feasible"] else infeasible_row + 1
        if infeasible_row >= 2:
            break
    if not any(q["feasible"] for q in pts) and vf_start < VF_MAX:   # Raise Vf if the start was already infeasible
        for vf in np.round(np.arange(vf_start + VF_STEP, VF_MAX + 1e-9, VF_STEP), 4):
            p = run_single_case(T, c_i, vf, None); pts.append(p)
            if p["feasible"]:
                break
    feas = [q for q in pts if q["feasible"]]
    if feas:
        p_ok = min(feas, key=lambda q: q["vf"])
        infeas = [q for q in pts if not q["feasible"] and q["vf"] < p_ok["vf"]]
        if infeas:
            p_fail = max(infeas, key=lambda q: q["vf"])
            for _ in range(N_BISECTION):
                p = run_single_case(T, c_i, round(0.5 * (p_ok["vf"] + p_fail["vf"]), 5), p_ok["x"]); pts.append(p)
                if p["feasible"]:
                    p_ok = p
                else:
                    p_fail = p
    pts.sort(key=lambda q: q["mass"])
    feas = [q for q in pts if q["feasible"]]
    return pts, (min(feas, key=lambda q: q["mass"]) if feas else None)


if RUN_SEARCHES:
    t_b6 = time.time()
    pareto_single, selected_single = {}, {}
    for k in MAT_KEYS:
        vf_start = min(VF_MAX, np.ceil((selected[(k, "Multi-load")]["vf"] + VF_MARGIN) / VF_STEP) * VF_STEP)
        for c_i in range(n_cases):
            pareto_single[(k, c_i)], selected_single[(k, c_i)] = single_case_search(k, c_i, vf_start)
    log(f"Block 6 searches completed in {(time.time() - t_b6) / 60:.1f} min")
    save_searches(pareto, selected, pareto_single, selected_single)
else:
    _p, _s, pareto_single, selected_single = load_searches()
    for key_s, pts_s in pareto_single.items():
        print_runs(pts_s, f"[{key_s[0]}] {CASE_NAMES[key_s[1]]} (loaded from results/designs/)")

## 6.2 Selected designs (Table 5.5.1) and cross-utilization (Figure 5.8.1)

Every selected design is then evaluated under all four cases. Rows identify designs and columns identify evaluated cases, and values above 1 violate a limit.

In [ ]:
CONSTRAINT_NAMES = ["u_crit <= 0.1", "u_rem <= 0.5", "FS >= 1.5"]


def active_constraint(e, Sy):
    """Name of the term that attains the maximum in Eq. (3.8)."""
    r = [e["u_crit"] / U_ADM_CRIT, e["u_rem"] / U_ADM_REM, FS_VM * e["vm_p995"] / Sy]
    return CONSTRAINT_NAMES[int(np.argmax(r))]


rows = []
for k in MAT_KEYS:
    m0 = MAT_OPS[k]["rho"] * V0
    entries = [(selected[(k, "Multi-load")], None, "Multi-load")] + [(selected_single[(k, c_i)], c_i, f"C{c_i + 1}") for c_i in range(n_cases)]
    for p, c_i, name_d in entries:
        c_g = int(np.argmax([e["util"] for e in p["eval"]])) if c_i is None else c_i     # Governing case
        e = p["eval"][c_g]
        rows.append(dict(material=k, design=name_d, Vf=p["vf"], mass_g=p["mass"], saving_pct=100 * (1 - p["mass"] / m0), gov_case=f"C{c_g + 1}",
                         u_crit=e["u_crit"], u_rem=e["u_rem"], FS=e["FS"], U=e["util"], active=active_constraint(e, MAT_OPS[k]["Sy"])))
print("Minimum-mass feasible designs (Table 5.5.1)")
print(pd.DataFrame(rows).round(4).to_string(index=False))

U_cross = {}
for k in MAT_KEYS:
    M_rows = [[selected_single[(k, c_i)]["eval"][j]["util"] for j in range(n_cases)] for c_i in range(n_cases)]
    M_rows.append([e["util"] for e in selected[(k, "Multi-load")]["eval"]])
    labels_d = [f"Design C{c_i + 1}" for c_i in range(n_cases)] + ["Multi-load"]
    U_cross[k] = (np.array(M_rows), labels_d)
    print(f"\nCross-utilization [{k}] (row: design, column: evaluated case)")
    print(pd.DataFrame(U_cross[k][0], index=labels_d, columns=[f"C{j + 1}" for j in range(n_cases)]).round(3).to_string())

In [ ]:
def vm_field(T, U, c_i):
    """Von Mises stress of all the elements of the grid for load case c_i."""
    s = np.einsum("eik,ek->ei", T["CB"], U[elem_dofs[act], c_i])
    vm = np.zeros(n_elem)
    vm[act] = np.sqrt(0.5 * ((s[:, 0] - s[:, 1])**2 + (s[:, 1] - s[:, 2])**2 + (s[:, 2] - s[:, 0])**2) + 3 * s[:, 3]**2)
    return vm


# Fronts per load case
col_case = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a"]
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=100, sharey=True)
for ax, k in zip(axes, MAT_KEYS):
    for c_i in range(n_cases):
        pts = pareto_single[(k, c_i)]
        ms = np.array([q["mass"] for q in pts]); us = np.array([q["util"] for q in pts]); fa = np.array([q["feasible"] for q in pts])
        ax.scatter(ms[fa], us[fa], color=col_case[c_i], s=35, zorder=3)
        ax.scatter(ms[~fa], us[~fa], facecolors="none", edgecolors=col_case[c_i], s=35, zorder=3)
        fr = non_dominated_front(pts)
        ax.plot([q["mass"] for q in fr], [q["util"] for q in fr], "-", color=col_case[c_i], lw=1.5, label=CASE_NAMES[c_i])
        q = selected_single[(k, c_i)]
        ax.scatter(q["mass"], q["util"], marker="*", s=240, color=col_case[c_i], edgecolors="k", zorder=4)
    fr = non_dominated_front(pareto[(k, "Multi-load")])
    ax.plot([q["mass"] for q in fr], [q["util"] for q in fr], "k--", lw=1.8, label="Multi-load (Block 5)")
    q = selected[(k, "Multi-load")]
    ax.scatter(q["mass"], q["util"], marker="*", s=300, color="k", zorder=5)
    ax.axhline(1.0, color="k", ls=":", lw=1.1)
    ax.set_xlabel("Mass [g]"); ax.set_title(f"{k}: fronts per load case (filled: feasible, open: infeasible, star: selected)", fontweight="bold")
    ax.grid(True, ls="--", alpha=0.4); ax.legend(fontsize=8); ax.set_ylim(0, 3)
axes[0].set_ylabel("Utilization in its own case")
plt.tight_layout()
fig.savefig(DIR_SEARCH / "pareto_single_case.png", dpi=300, bbox_inches="tight")
plt.show()

# Cross-utilization matrices and masses
fig, axes = plt.subplots(1, 3, figsize=(20, 5.8), dpi=100, constrained_layout=True)
for ax, k in zip(axes[:-1], MAT_KEYS):
    M, labels_d = U_cross[k]
    im = ax.imshow(M, cmap="RdYlGn_r", vmin=0, vmax=2, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=10, fontweight="bold" if M[i, j] > 1 else "normal")
    ax.set_xticks(range(n_cases)); ax.set_xticklabels([f"C{j + 1}" for j in range(n_cases)])
    ax.set_yticks(range(len(labels_d))); ax.set_yticklabels(labels_d)
    ax.set_xlabel("Evaluated load case"); ax.set_title(f"{k}: cross-utilization (above 1 violates)", fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).set_label("utilization")
ax = axes[-1]
bar_w = 0.4
bar_labels = [f"C{c_i + 1}" for c_i in range(n_cases)] + ["Multi-load"]
for i_m, k in enumerate(MAT_KEYS):
    masses = [selected_single[(k, c_i)]["mass"] for c_i in range(n_cases)] + [selected[(k, "Multi-load")]["mass"]]
    xs = np.arange(len(bar_labels)) + (i_m - 0.5) * bar_w
    ax.bar(xs, masses, bar_w, label=k, color=["#1f77b4", "#d62728"][i_m], edgecolor="k")
    for xx, mm in zip(xs, masses):
        ax.text(xx, mm + 10, f"{mm:.0f}", ha="center", fontsize=8)
    ax.axhline(MAT_OPS[k]["rho"] * V0, color=["#1f77b4", "#d62728"][i_m], ls=":", lw=1)
ax.set_xticks(range(len(bar_labels))); ax.set_xticklabels(bar_labels)
ax.set_ylabel("Minimum feasible mass [g] (dotted: solid)")
ax.set_title("Selected masses", fontweight="bold"); ax.legend(loc="upper right", fontsize=9); ax.grid(True, axis="y", ls="--", alpha=0.4)
ax.set_ylim(0, max(MAT_OPS[k]["rho"] * V0 for k in MAT_KEYS) * 1.12)
fig.savefig(DIR_SEARCH / "cross_utilization_masses.png", dpi=300, bbox_inches="tight")
plt.show()

---
# Block 7. Export of the selected designs (Sec. 3.7, Table 3.7.1)

The ten optimized designs are exported as complete rings and as the 7.5° sectors that were printed and tested. The chord tolerance gives 345 facets of 1.043° in the
rings and 23 facets of 0.326° in the sectors. If a mesh is not closed, the export is repeated with a larger collinearity tolerance.

The signed volume of each mesh is compared with the finite element volume scaled by $\theta/360$. All 20 meshes must be closed (Table 5.1.1).

In [ ]:
SECTOR_ANGLE = 7.5                                      # [deg]
N_DIV_SECTOR = n_angular_divisions(SECTOR_ANGLE, CHORD_TOL_SECTOR)
N_DIV_RING = n_angular_divisions(360.0, CHORD_TOL_RING)
step_deg = SECTOR_ANGLE / N_DIV_SECTOR
sagitta = R_out * (1 - np.cos(np.radians(step_deg) / 2))
print(f"Sector: {N_DIV_SECTOR} facets of {step_deg:.3f} deg, largest sagitta {1000 * sagitta:.2f} um at r = {R_out:.0f} mm | Ring: {N_DIV_RING} facets of {360.0 / N_DIV_RING:.3f} deg")


def closed_solid(rb, angle, ndiv):
    """Contours and revolution of a binary design. Retries with a larger collinearity tolerance if the mesh is not closed."""
    base = meridian_contours(rb)
    for tol_rel in (1e-9, 1e-7, 1e-5):
        polys = [(clean_polygon(p, tol_rel), d, a) for p, d, a in base]
        polys = [q for q in polys if len(q[0]) >= 3]
        Vx, Fx = revolve(polys, angle, ndiv)
        closed, unique = audit_mesh(Vx, Fx)
        if closed and unique:
            break
    return polys, Vx, Fx, closed, unique


# Inventory: multi-load (Block 5) and single-case designs (Block 6), both materials
pieces = []
for k in MAT_KEYS:
    p = selected[(k, "Multi-load")]
    pieces.append((k, "Multi-load", p["vf"], p["rho_bin"]))
    for c_i in range(n_cases):
        p = selected_single[(k, c_i)]
        pieces.append((k, f"C{c_i + 1}", p["vf"], p["rho_bin"]))

export_jobs = [(SECTOR_ANGLE, N_DIV_SECTOR, "sec7p5", DIR_STL / "sectors")]
if EXPORT_RINGS:
    export_jobs.append((360.0, N_DIV_RING, "360", DIR_STL / "rings"))
export_records = []
for angle, ndiv, suffix, folder in export_jobs:
    for k, name_d, vf, rb in pieces:
        polys, Vx, Fx, closed, unique = closed_solid(rb, angle, ndiv)
        file_stl = folder / f"{k}_{name_d.replace('-', '')}_vf{int(round(1000 * vf))}_{suffix}.stl"
        vol, _ = export_stl(Vx, Fx, file_stl)
        V_fea = float(V_elem @ rb) * angle / 360.0
        export_records.append(dict(file=file_stl.name, material=k, design=name_d, Vf=vf, angle=angle, facets=ndiv, triangles=len(Fx),
                                   cavities=sum(d % 2 == 1 for _, d, _ in polys), closed=bool(closed and unique), V_stl_mm3=vol,
                                   mass_g=MAT_OPS[k]["rho"] * vol, volume_error_pct=100 * (vol - V_fea) / V_fea, size_MB=os.path.getsize(file_stl) / 1e6))
stl_table = pd.DataFrame(export_records)
stl_table.to_csv(DIR_STL / "stl_exports.csv", index=False)
print(stl_table.drop(columns=["material", "design", "Vf", "facets"]).round(3).to_string(index=False))
print(f"Closed surfaces: {int(stl_table.closed.sum())} of {len(stl_table)} | volume error from {stl_table.volume_error_pct.min():.2f} to {stl_table.volume_error_pct.max():.2f} %")

---
# Block 8. Virtual compression test (Sec. 3.8)

## 8.1 Model and solution

Each design is compressed between two rigid platens at $z = 0$ and $z = H$, under displacement control and without the supports of the design problem. The contact is bonded,
$u_r = 0$ on the platen nodes. By Eq. (2.9), the stiffness scaled by $\theta/360$ is a conditional upper bound for an otherwise equivalent sector with relaxed constraints.

Yielding uses the **elastic compensation method** (Sec. 2.9.2): after each linear analysis, the modulus of every element above the yield limit is reduced in proportion to the excess,

$$d_e \leftarrow \max\left(d_e\,\frac{S_y}{\sigma_{VM,e}},\ 10^{-9}\right).$$

For given factors the problem is linear, so it is solved once for a unit platen displacement and scaled by $\delta$. Only a few factors change between solutions, so the systems are solved by preconditioned
conjugate gradients with the LU factorization of an earlier state, recomputed only when the solver fails or needs more than 80 iterations.

## 8.2 Metrics

- $K_{ini}$: slope of the elastic solution, **the main quantity compared with the physical tests**.
- $(\delta_y, F_y)$: yield onset, where $\sigma_{VM,p99.5} = S_y$.
- $(\delta_{col}, F_{col})$: first step with tangent stiffness below 5 % of $K_{ini}$. Indicative only: the model has no contact and is geometrically linear, and every collapse with $\delta/H > 5\%$ is flagged.

In [ ]:
PLATEN_CONTACT = "bonded"     # u_r = 0 on both platens (high friction, no sliding)
STEPS_PER_YIELD = 8           # Displacement increments per yield-onset displacement
TOL_SECANT = 0.01             # Yield condition checked at the centroids within 1 %
MAX_IT_SECANT = 40            # Secant iterations per increment
K_COLLAPSE = 0.05             # Collapse when the tangent stiffness falls below 5 % of the initial one
MAX_STEPS = 400               # Largest number of displacement increments per design
MAX_NOMINAL_STRAIN = 0.30     # The test also ends when delta/H exceeds 30 %

I_act2, J_act2 = I_act.reshape(len(act), 64), J_act.reshape(len(act), 64)
top_row = (np.arange(n_elem) // nelr) == nelz - 1


def solve_pcg(K, b, x0, lu):
    """Conjugate gradients preconditioned with the LU factorization of an earlier state (low-rank changes)."""
    it = [0]
    M = LinearOperator(K.shape, matvec=lu.solve)
    def count(_): it[0] += 1
    x_, info = cg(K, b, x0=x0, M=M, rtol=1e-10, maxiter=150, callback=count)
    return x_, info, it[0]


def virtual_compression_test(k, name_d, rb):
    """Force-displacement curve of a design between rigid platens. Returns metrics, curve, collapse state and solid elements."""
    T = MAT_OPS[k]; Sy = T["Sy"]
    sol = rb[act] >= 0.5
    ids = np.where(sol)[0]                              # Indices within act
    e_sol = act[ids]
    n_sol = np.unique(connectivity[e_sol])
    bottom = n_sol[z_nodes[n_sol] <= 1e-6]              # Nodes on the lower platen
    top = n_sol[z_nodes[n_sol] >= H_part - 1e-6]        # Nodes on the upper platen
    inactive = np.setdiff1d(np.arange(n_nodes), n_sol)

    presc_z = 2 * top + 1                               # u_z = -delta on the upper platen
    fixed = [2 * bottom + 1, 2 * inactive, 2 * inactive + 1]
    if PLATEN_CONTACT == "bonded":
        fixed += [2 * bottom, 2 * top]
    fixed = np.unique(np.concatenate(fixed + [presc_z]))
    free = np.setdiff1d(np.arange(2 * n_nodes), fixed)
    g2f_t = -np.ones(2 * n_nodes, dtype=np.int64); g2f_t[free] = np.arange(len(free))
    is_presc = np.zeros(2 * n_nodes, bool); is_presc[presc_z] = True

    gI, gJ = I_act2[ids].ravel(), J_act2[ids].ravel()
    kv = T["ke_flat"][ids]                              # (n_sol, 64)
    pI, pJ = g2f_t[gI], g2f_t[gJ]
    m_ff = (pI >= 0) & (pJ >= 0)
    m_fp = (pI >= 0) & is_presc[gJ]                     # Free-prescribed coupling
    m_rx = is_presc[gI]                                 # Reaction rows of the upper platen
    nf_t = len(free)
    A_contact = float(np.sum(np.pi * (r_nodes[connectivity[e_sol, 2]]**2 - r_nodes[connectivity[e_sol, 3]]**2) * top_row[e_sol]))
    mass = float(T["rho"] * V_elem @ rb)

    CB_s = T["CB"][ids]; dofs_s = elem_dofs[e_sol]
    state = dict(lu=None, x=None, n_fact=0)

    def unit_field(d):
        """Solution for delta = 1 mm with relative moduli d: U, force per mm and von Mises stress per mm."""
        vals = (kv * d[:, None]).ravel()
        K = sp.csc_matrix((vals[m_ff], (pI[m_ff], pJ[m_ff])), shape=(nf_t, nf_t))
        b = np.bincount(pI[m_fp], weights=vals[m_fp], minlength=nf_t)    # -K_fp u_p with u_p = -1
        if state["lu"] is None:
            state["lu"] = splu(K, permc_spec="MMD_AT_PLUS_A"); state["n_fact"] += 1
            x_ = state["lu"].solve(b)
        else:
            x_, info, nit = solve_pcg(K, b, state["x"], state["lu"])
            if info != 0 or nit > 80:                   # Outdated preconditioner: factorize again
                state["lu"] = splu(K, permc_spec="MMD_AT_PLUS_A"); state["n_fact"] += 1
                x_ = state["lu"].solve(b)
        state["x"] = x_
        U = np.zeros(2 * n_nodes); U[free] = x_; U[presc_z] = -1.0
        F1 = -float(np.sum(vals[m_rx] * U[gJ[m_rx]]))
        s_ = d[:, None] * np.einsum("eik,ek->ei", CB_s, U[dofs_s])
        vm = np.sqrt(0.5 * ((s_[:, 0] - s_[:, 1])**2 + (s_[:, 1] - s_[:, 2])**2 + (s_[:, 2] - s_[:, 0])**2) + 3 * s_[:, 3]**2)
        return U, F1, vm

    d = np.ones(len(ids))
    U1, K_ini, vm1 = unit_field(d)
    delta_yield = Sy / np.percentile(vm1, 99.5)         # Yield onset: sigma_VM,p99.5 = Sy
    curve, peak_state = [], None

    step = delta_yield / STEPS_PER_YIELD
    delta, collapse, extra = 0.0, False, 0
    while delta / H_part < MAX_NOMINAL_STRAIN and len(curve) < MAX_STEPS:
        delta += step
        for it in range(MAX_IT_SECANT):                 # Secant iterations at fixed delta
            U1, F1, vm1 = unit_field(d)
            vm = vm1 * delta
            plastic = vm > Sy * (1 + TOL_SECANT)
            if not plastic.any():
                break
            d[plastic] = np.maximum(d[plastic] * Sy / vm[plastic], 1e-9)
        F = F1 * delta
        curve.append((delta, F, int((d < 0.999).sum())))
        if len(curve) > 1 and not collapse:
            k_t = (curve[-1][1] - curve[-2][1]) / step
            if k_t < K_COLLAPSE * K_ini:
                collapse = True
                peak_state = dict(F=F, delta=delta, U=U1 * delta, vm=np.minimum(vm, Sy), yielded=d < 0.999, step=len(curve) - 1)
        if collapse:
            extra += 1
            if extra > 4:
                break
    if peak_state is None:                              # No collapse before MAX_NOMINAL_STRAIN
        peak_state = dict(F=F, delta=delta, U=U1 * delta, vm=np.minimum(vm, Sy), yielded=d < 0.999, step=len(curve) - 1)

    h = np.array(curve)
    i_pk = peak_state["step"]
    res = dict(material=k, design=name_d, mass_g=mass, A_contact_mm2=A_contact, K_ini_N_mm=K_ini, F_yield_N=K_ini * delta_yield,
               delta_yield_mm=delta_yield, F_max_N=float(h[i_pk, 1]), delta_Fmax_mm=float(h[i_pk, 0]), strain_nom_Fmax=float(h[i_pk, 0] / H_part),
               stress_nom_Fmax_MPa=float(h[i_pk, 1] / A_contact), F_spec_N_g=float(h[i_pk, 1] / mass),
               energy_peak_J=float(np.trapezoid(np.r_[0, h[:i_pk + 1, 1]], np.r_[0, h[:i_pk + 1, 0]]) / 1000.0), steps=len(h),
               yielded_elems=int(h[i_pk, 2]), factorizations=state["n_fact"], collapse=bool(i_pk < len(h) - 1 or len(h) < MAX_STEPS), n_elem=len(ids))
    return res, h, peak_state, e_sol

In [ ]:
def test_figure(res, h, pk, e_sol, Sy):
    """Figure of one test: force-displacement curve, displacement field and von Mises field at collapse (true scale)."""
    k, name_d = res["material"], res["design"]
    fig, axes = plt.subplots(1, 3, figsize=(20, 6.6), dpi=100, gridspec_kw=dict(width_ratios=[1.25, 1, 1]))
    ax = axes[0]
    ax.plot(np.r_[0, h[:, 0]], np.r_[0, h[:, 1]] / 1e3, "-", color="#424242", lw=1.6, label="Elastic-perfectly plastic response")
    ax.plot([0, res["delta_yield_mm"]], [0, res["F_yield_N"] / 1e3], "--", color="#1f77b4", lw=1.2, label=f"Initial stiffness {res['K_ini_N_mm'] / 1e3:,.1f} kN/mm")
    ax.plot(res["delta_yield_mm"], res["F_yield_N"] / 1e3, "o", color="#ff7f0e", ms=9, label=f"Yield onset: {res['F_yield_N'] / 1e3:,.1f} kN")
    ax.plot(res["delta_Fmax_mm"], res["F_max_N"] / 1e3, "*", color="#d62728", ms=18, mec="k", label=f"Collapse: {res['F_max_N'] / 1e3:,.1f} kN at {res['delta_Fmax_mm']:.3f} mm")
    ax.set_xlabel("Platen displacement [mm]"); ax.set_ylabel("Force [kN]"); ax.grid(True, ls="--", alpha=0.4); ax.legend(fontsize=8.5, loc="lower right")
    ax.set_title(f"{k} | {name_d}: force-displacement curve", fontweight="bold")
    conn = connectivity[e_sol]
    U = pk["U"]; um = np.hypot(U[0::2], U[1::2])
    verts = np.stack([r_nodes[conn] + U[2 * conn], z_nodes[conn] + U[2 * conn + 1]], axis=-1)
    ax = axes[1]
    pc = PolyCollection(verts, array=um[conn].mean(1), cmap="viridis", edgecolors="face", linewidths=0.2)
    ax.add_collection(pc)
    if pk["yielded"].any():
        ax.add_collection(PolyCollection(verts[pk["yielded"]], facecolors=(0, 0, 0, 0.30), edgecolors="none"))
    fig.colorbar(pc, ax=ax, fraction=0.046, pad=0.04).set_label("|u| [mm]")
    ax.axhline(H_part + U[2 * conn + 1].min(), color="#555", lw=3); ax.axhline(0, color="#555", lw=3)
    ax.set_title(f"Displacement at collapse (true scale)\nshaded: {int(pk['yielded'].sum())} yielded elements", fontweight="bold")
    ax = axes[2]
    pv = PolyCollection(verts, array=pk["vm"], cmap="jet", edgecolors="face", linewidths=0.2); pv.set_clim(0, Sy)
    ax.add_collection(pv)
    fig.colorbar(pv, ax=ax, fraction=0.046, pad=0.04, extend="max").set_label(r"$\sigma_{VM}$ [MPa] (capped at Sy)")
    ax.set_title(f"Von Mises stress at collapse (Sy = {Sy:.2f} MPa)", fontweight="bold")
    for ax in axes[1:]:
        ax.set_aspect("equal"); ax.set_xlim(R_in - 4, R_out + 4); ax.set_ylim(-4, H_part + 6)
        ax.set_xlabel("r [mm]"); ax.set_ylabel("z [mm]"); ax.grid(True, ls="--", alpha=0.3)
    plt.tight_layout()
    fig.savefig(DIR_VTEST / f"test_{k}_{name_d.replace('-', '')}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)                                      # Saved to results/virtual_test/, not shown, to keep the notebook light


def tested_designs():
    """The twelve tested designs: the solid and the five selected designs of each material."""
    out = []
    for k in MAT_KEYS:
        out.append((k, "Base", rho_0))
        out.append((k, "Multi-load", selected[(k, "Multi-load")]["rho_bin"]))
        for c_i in range(n_cases):
            out.append((k, f"C{c_i + 1}", selected_single[(k, c_i)]["rho_bin"]))
    return out


if RUN_VIRTUAL_TEST:
    t_b8 = time.time()
    vtest_results, vtest_curves = [], {}
    for k, name_d, rb in tested_designs():
        t0 = time.time()
        res, h, pk, e_sol = virtual_compression_test(k, name_d, rb)
        res["t_s"] = time.time() - t0
        test_figure(res, h, pk, e_sol, MAT_OPS[k]["Sy"])
        vtest_results.append(res); vtest_curves[(k, name_d)] = h
        with open(DIR_VTEST / f"curve_{k}_{name_d.replace('-', '')}.csv", "w", newline="") as fc:
            wr = csv.writer(fc); wr.writerow(["delta_mm", "F_N", "yielded_elements"]); wr.writerows(h.tolist())
        log(f"   {k:<7}{name_d:<11} F_col = {res['F_max_N'] / 1e3:9.1f} kN | delta = {res['delta_Fmax_mm']:.3f} mm | {res['steps']} steps, {res['factorizations']} LU, {res['t_s']:.0f} s")
    with open(DIR_VTEST / "virtual_test_results.json", "w", encoding="utf-8") as fh:
        json.dump(dict(contact=PLATEN_CONTACT, k_collapse=K_COLLAPSE, results=vtest_results), fh, indent=1, default=float)
    log(f"Block 8 completed in {(time.time() - t_b8) / 60:.1f} min")
else:
    with open(DIR_VTEST / "virtual_test_results.json", encoding="utf-8") as fh:
        vtest_results = json.load(fh)["results"]
    vtest_curves = {(r["material"], r["design"]): np.loadtxt(DIR_VTEST / f"curve_{r['material']}_{r['design'].replace('-', '')}.csv", delimiter=",", skiprows=1)
                    for r in vtest_results}
    print("Virtual tests loaded from results/virtual_test/")

SECTOR_FACTOR = SECTOR_ANGLE / 360.0                    # 1/48, Eq. (4.3)
vt = pd.DataFrame(vtest_results)
vt_table = pd.DataFrame(dict(material=vt.material, design=vt.design, m_g=vt.mass_g, K_ini_kN_mm=vt.K_ini_N_mm / 1e3, K_over_m=vt.K_ini_N_mm / vt.mass_g,
                             F_y_kN=vt.F_yield_N / 1e3, F_col_kN=vt.F_max_N / 1e3, delta_col_mm=vt.delta_Fmax_mm, delta_over_H_pct=100 * vt.strain_nom_Fmax,
                             K_pred_kN_mm=vt.K_ini_N_mm * SECTOR_FACTOR / 1e3, F_pred_kN=vt.F_max_N * SECTOR_FACTOR / 1e3))
vt_table.to_csv(DIR_VTEST / "virtual_test_table.csv", index=False)
print("Virtual test (Table 5.11.1) and predictions for the 7.5 degree sectors (Table 5.12.1)")
print(vt_table.round(3).to_string(index=False))
print(f"Every collapse above 5 % nominal strain: {bool((vt.strain_nom_Fmax > 0.05).all())}. Collapse values are indicative.")

In [ ]:
# Comparison of all the curves, absolute and per unit mass
styles = {"Base": ("k", "-"), "Multi-load": ("#d62728", "-"), "C1": ("#1b9e77", "--"), "C2": ("#d95f02", "--"), "C3": ("#7570b3", "--"), "C4": ("#e7298a", "--")}
fig, axes = plt.subplots(2, 2, figsize=(14, 9.5), dpi=100, squeeze=False)
for j, k in enumerate(MAT_KEYS):
    for r in [q for q in vtest_results if q["material"] == k]:
        h = vtest_curves[(k, r["design"])]; col_, ls_ = styles[r["design"]]
        x_c, y_c = np.r_[0, h[:, 0]], np.r_[0, h[:, 1]]
        axes[0, j].plot(x_c, y_c / 1e3, ls_, color=col_, lw=1.4, label=r["design"])
        axes[0, j].plot(r["delta_Fmax_mm"], r["F_max_N"] / 1e3, "*", color=col_, ms=13, mec="k")
        axes[1, j].plot(x_c, y_c / r["mass_g"], ls_, color=col_, lw=1.4, label=r["design"])
        axes[1, j].plot(r["delta_Fmax_mm"], r["F_spec_N_g"], "*", color=col_, ms=13, mec="k")
    axes[0, j].set_title(f"{k}: force-displacement (star: collapse)", fontweight="bold")
    axes[1, j].set_title(f"{k}: force per unit mass (star: collapse)", fontweight="bold")
    axes[0, j].set_ylabel("F [kN]"); axes[1, j].set_ylabel("F/m [N/g]")
    for i in range(2):
        axes[i, j].set_xlabel("Platen displacement [mm]"); axes[i, j].grid(True, ls="--", alpha=0.4); axes[i, j].legend(fontsize=9)
plt.tight_layout()
fig.savefig(DIR_VTEST / "virtual_test_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

---
# Block 9. Independent verification (Sec. 3.9)

The block runs seven independent checks on the stored results, without generating new designs. All of them read the designs from memory (Blocks 5, 6 and 8).

## 9.0 Configuration and references

Block 9 generates no new designs. It verifies the stored ones. The outputs are tables T01 to T09 and figures F01 and F02, in `results/verification/`.

In [ ]:
_b9_t0 = time.time()
print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] BLOCK 9: independent verification of the existing results")
B9_CFG = dict(
    SAVE_OUTPUTS=True,
    OUT_ROOT=str(DIR_VERIF),
    TOL_REPRO=1e-6,                       # relative tolerance for "reproduced"
    RUN_SENSITIVITY=True,                 # 9.7 (about 5 factorizations per design)
    ALPHA_G_SWEEP=[0.32, 0.41, "huber"],  # 0.32-0.41: empirical FFF range quoted in Block 1; 'huber': upper bound
    RUN_MESH_STUDY=True,                  # 9.8
    MESH_FACTORS=[0.5, 1.0, 1.5],         # multiples of the current (nelr, nelz); 1.0 must reproduce Block 2
    RUN_B8_ELASTIC_CHECK=True,            # 9.9 reproduces K_ini of every platen test (1 factorization per design)
    B8_RERUN_PENETRATION=[("PA6", "Base"), ("PA6_CF", "Base")],
                                          # 9.9 re-runs Block 8 for these designs and measures how far the
                                          #     material crosses the rigid platens (no contact in Block 8).
                                          #     About 5 min each. [] to skip.
)

# Reference values computed in material_characterization.ipynb.
# Tension: xy plane (deposition plane). Compression: z (build direction). Source of truth = that notebook.
B9_CHAR_REF = {
    "PA6":    dict(E_t=839.44,  Sy_t=20.07, Su_t=50.84, E_c=294.32, Sy_c=53.30, Su_c=54.49, eps_y_c=0.0987),
    "PA6_CF": dict(E_t=2056.59, Sy_t=42.19, Su_t=75.90, E_c=684.46, Sy_c=25.80, Su_c=42.64, eps_y_c=0.0413),
}

## 9.1 Kernel-state guard

Checks that every variable needed by the block exists and has a consistent size.

In [ ]:
_b9_required = ["ACTIVE_MATERIAL", "ALPHA_G", "MATERIAL_CATALOG", "MATERIALS",
                "derive_constants", "nelr", "nelz", "n_elem", "n_nodes", "dr", "dz", "R_in", "R_out",
                "H_part", "r_nodes", "z_nodes", "connectivity", "centroid_r", "centroid_z", "rho_0", "V_elem",
                "V0", "contour_segments", "R_mesh", "Z_mesh", "P_BENCH", "U_0", "J_0", "F_vector",
                "nodes_sliding", "nodes_clamped", "elem_dofs", "vm_max_0", "vm_p995_0", "u_max_0",
                "act", "MAT_OPS", "F_design", "lam_c", "nodes_crit", "nodes_rem", "U_ADM_CRIT", "U_ADM_REM",
                "FS_VM", "CASE_NAMES", "n_cases", "MAT_KEYS", "eval_solid", "selected", "pareto",
                "selected_single", "pareto_single", "vtest_results", "PLATEN_CONTACT"]
_b9_missing = [n for n in _b9_required if n not in globals()]
print("\n" + "=" * 96)
print(" 9.1 KERNEL-STATE GUARD")
print("=" * 96)
if _b9_missing:
    raise RuntimeError(f"Missing variables (run the corresponding blocks first): {_b9_missing}")
_b9_checks = {
    "r_nodes.size == (nelr+1)(nelz+1)": r_nodes.size == (nelr + 1) * (nelz + 1) == n_nodes,
    "rho_0.size == nelr*nelz": rho_0.size == nelr * nelz == n_elem,
    "U_0.size == 2*n_nodes": U_0.size == 2 * n_nodes,
    "F_design.shape == (2*n_nodes, n_cases)": F_design.shape == (2 * n_nodes, n_cases),
    "uniform grid (dr, dz)": np.allclose(np.diff(np.unique(np.round(r_nodes, 9))), dr)
                             and np.allclose(np.diff(np.unique(np.round(z_nodes, 9))), dz),
    "J_0 == F_vector . U_0": np.isclose(J_0, float(F_vector @ U_0), rtol=1e-10),
    "every stored design has n_elem densities": all(
        p is None or p["rho_bin"].size == n_elem
        for p in list(selected.values()) + list(selected_single.values())),
    "MAT_OPS has both materials": all(k in MAT_OPS for k in MAT_KEYS),
}
for _n, _ok in _b9_checks.items():
    print(f"  [{'PASS' if _ok else 'FAIL'}] {_n}")
print(f"  mesh {nelr} x {nelz} | active material {ACTIVE_MATERIAL} | alpha_G {ALPHA_G:.4f}")
if not all(_b9_checks.values()):
    raise RuntimeError("Inconsistent kernel state: restart and run the notebook in order.")

b9_run_dir = None
if B9_CFG["SAVE_OUTPUTS"]:
    b9_run_dir = B9_CFG["OUT_ROOT"]
    os.makedirs(b9_run_dir, exist_ok=True)
    print(f"  output folder: {b9_run_dir}")


def _b9_save(df, name):
    if b9_run_dir is not None:
        df.to_csv(os.path.join(b9_run_dir, f"{name}.csv"), index=False)


def _b9_savefig(fig, name):
    if b9_run_dir is not None:
        fig.savefig(os.path.join(b9_run_dir, f"{name}.png"), dpi=250, bbox_inches="tight")


try:
    import pypardiso
    def _b9_solve(K, B):
        return pypardiso.spsolve(K.tocsr(), B)
    b9_solver = "PARDISO"
except ImportError:
    def _b9_solve(K, B):
        return splu(K.tocsc(), permc_spec="MMD_AT_PLUS_A").solve(B)
    b9_solver = "SuperLU"

## 9.2 Traceability of the material data

Compares the constants used in the model with those computed in `material_characterization.ipynb` (table T01).

In [ ]:
_rows = []
for k in MAT_KEYS:
    m, ref = MATERIALS[k], B9_CHAR_REF[k]
    _rows.append(dict(material=k, E_r_used=m["E_r"], E_t_char=ref["E_t"], E_z_used=m["E_z"], E_c_char=ref["E_c"],
                      Sy_used=m["Sy"], Sy_c_char=ref["Sy_c"], Su_c_char=ref["Su_c"], Sy_t_xy_char=ref["Sy_t"],
                      Sy_used_over_Su_c=m["Sy"] / ref["Su_c"],
                      eps_y_linear_model_pct=100 * m["Sy"] / m["E_z"], eps_y_measured_pct=100 * ref["eps_y_c"],
                      G_rz=m["G_rz"], nu_rz=m["nu_rz"], nu_zr=m["nu_zr"], alpha_G=m["alpha_G"],
                      alpha_huber=m["alpha_huber"]))
b9_material = pd.DataFrame(_rows)
print("\n" + "=" * 96)
print(" 9.2 MATERIAL-DATA TRACEABILITY (catalog used in Blocks 1-8 vs material_characterization.ipynb)")
print("=" * 96)
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 4):
    print(b9_material.T.to_string(header=False))
_b9_save(b9_material, "T01_material_traceability")

## 9.3 Independent re-implementation (Table 5.1.1)

The element matrices and the evaluation are written again, without reusing the functions of Blocks 2 and 5. The grid is structured, so $k_e$ depends only on the radial column. Every design is then re-evaluated in every load case and compared with the stored values (T02, T03).

In [ ]:
_b9_col = np.arange(n_elem) % nelr
_b9_gp = np.array([-1.0, 1.0]) / np.sqrt(3.0)


def _b9_compliance_matrix(m):
    Er, Ez, nrt, nrz, nzr, G = m["E_r"], m["E_z"], m["nu_rtheta"], m["nu_rz"], m["nu_zr"], m["G_rz"]
    return np.array([[1 / Er, -nzr / Ez, -nrt / Er, 0.0],
                     [-nrz / Er, 1 / Ez, -nrz / Er, 0.0],
                     [-nrt / Er, -nzr / Ez, 1 / Er, 0.0],
                     [0.0, 0.0, 0.0, 1 / G]])


def _b9_column_operators(C, nelr_, dr_, dz_, R_in_):
    """Per-radial-column stiffness (nelr, 8, 8) and centroid stress operator C·B_c (nelr, 4, 8)."""
    ke = np.zeros((nelr_, 8, 8)); CB = np.zeros((nelr_, 4, 8))
    for i in range(nelr_):
        r_e = R_in_ + dr_ * np.array([i, i + 1, i + 1, i], float)
        for xi, eta in [(a, b) for a in _b9_gp for b in _b9_gp] + [(0.0, 0.0)]:
            N = 0.25 * np.array([(1 - xi) * (1 - eta), (1 + xi) * (1 - eta), (1 + xi) * (1 + eta), (1 - xi) * (1 + eta)])
            dNr = (0.5 / dr_) * np.array([-(1 - eta), (1 - eta), (1 + eta), -(1 + eta)])
            dNz = (0.5 / dz_) * np.array([-(1 - xi), -(1 + xi), (1 + xi), (1 - xi)])
            rg = N @ r_e
            B = np.zeros((4, 8))
            B[0, 0::2] = dNr; B[1, 1::2] = dNz; B[2, 0::2] = N / rg; B[3, 0::2] = dNz; B[3, 1::2] = dNr
            if xi == 0.0 and eta == 0.0:
                CB[i] = C @ B
            else:
                ke[i] += B.T @ C @ B * (2.0 * np.pi * rg * 0.25 * dr_ * dz_)
    return ke, CB


def _b9_material_ops(m):
    S = _b9_compliance_matrix(m)
    C = np.linalg.inv(S); C = 0.5 * (C + C.T)
    ke_col, CB_col = _b9_column_operators(C, nelr, dr, dz, R_in)
    return dict(C=C, S=S, ke_col=ke_col, CB_col=CB_col, Sy=m["Sy"], rho=m["rho_g_mm3"], eigC=np.linalg.eigvalsh(C))


def _b9_wpercentile(v, w, q):
    o = np.argsort(v); cw = np.cumsum(w[o]); cw /= cw[-1]
    return float(v[o][min(np.searchsorted(cw, q / 100.0), len(v) - 1)])


_b9_dof_r, _b9_dof_z = 2 * np.arange(n_nodes), 2 * np.arange(n_nodes) + 1
_b9_Iall = np.repeat(elem_dofs, 8, axis=1)
_b9_Jall = np.tile(elem_dofs, (1, 8))


def b9_evaluate(rho_bin, ops, k, F_full=F_design):
    """Independent evaluation of a binary design in all load cases (same constraint definitions as Block 5),
    plus equilibrium, energy, load-accounting and directional-stress metrics."""
    sol_e = np.where((rho_bin >= 0.5) & (rho_0 == 1.0))[0]       # design is a subset of the original solid
    n_sol = np.unique(connectivity[sol_e])
    inact = np.setdiff1d(np.arange(n_nodes), n_sol)
    fixed = np.unique(np.concatenate([2 * np.intersect1d(nodes_sliding, n_sol) + 1,
                                      2 * np.intersect1d(nodes_clamped, n_sol),
                                      2 * np.intersect1d(nodes_clamped, n_sol) + 1,
                                      2 * inact, 2 * inact + 1]))
    free = np.setdiff1d(np.arange(2 * n_nodes), fixed)
    vals = ops["ke_col"][_b9_col[sol_e]].reshape(len(sol_e), 64).ravel()
    K = sp.coo_matrix((vals, (_b9_Iall[sol_e].ravel(), _b9_Jall[sol_e].ravel())),
                      shape=(2 * n_nodes, 2 * n_nodes)).tocsr()
    Kff = K[free, :][:, free]
    U = np.zeros((2 * n_nodes, F_full.shape[1]))
    U[free] = np.asarray(_b9_solve(Kff, F_full[free])).reshape(len(free), -1)
    KU = K @ U
    is_inact = np.zeros(2 * n_nodes, bool); is_inact[2 * inact] = True; is_inact[2 * inact + 1] = True
    is_supp = np.zeros(2 * n_nodes, bool); is_supp[fixed] = True; is_supp &= ~is_inact
    Vs = V_elem[sol_e]
    out = []
    for c in range(F_full.shape[1]):
        Fc, Uc = F_full[:, c], U[:, c]
        um = np.hypot(Uc[0::2], Uc[1::2])
        s = np.einsum("eik,ek->ei", ops["CB_col"][_b9_col[sol_e]], Uc[elem_dofs[sol_e]])
        sr, sz, st, trz = s.T
        vm = np.sqrt(0.5 * ((sr - sz) ** 2 + (sz - st) ** 2 + (st - sr) ** 2) + 3.0 * trz ** 2)
        nv, nr_ = np.intersect1d(nodes_crit, n_sol), np.intersect1d(nodes_rem, n_sol)
        u_v = float(um[nv].max()) if len(nv) else 0.0
        u_r = float(um[nr_].max())
        p995 = float(np.percentile(vm, 99.5))
        t_ip = np.maximum(np.maximum(sr, st), 0.0)              # in-plane (r-theta) tension
        c_z = np.maximum(-sz, 0.0)                               # axial compression (build direction)
        t_z = np.maximum(sz, 0.0)                                # interlaminar tension (not characterized)
        Fabs = np.abs(Fc).sum()
        Ff, Uf = Fc[free], Uc[free]
        res = np.linalg.norm(KU[free, c] - Ff) / max(np.linalg.norm(Ff), 1e-30)
        J = float(Ff @ Uf)
        W = float(Uc @ KU[:, c])
        sumFz = np.abs(Fc[1::2]).sum()
        out.append(dict(
            u_crit=u_v, u_rem=u_r, vm_p995=p995, vm_max=float(vm.max()),
            vm_p995_vol=_b9_wpercentile(vm, Vs, 99.5), J=J, FS=ops["Sy"] / p995,
            util=max(u_v / U_ADM_CRIT, u_r / U_ADM_REM, FS_VM * p995 / ops["Sy"]),
            governs=["u_crit", "u_rem", "FS_VM"][int(np.argmax([u_v / U_ADM_CRIT, u_r / U_ADM_REM,
                                                                      FS_VM * p995 / ops["Sy"]]))],
            t_ip_p995=float(np.percentile(t_ip, 99.5)), c_z_p995=float(np.percentile(c_z, 99.5)),
            t_z_p995=float(np.percentile(t_z, 99.5)), tau_p995=float(np.percentile(np.abs(trz), 99.5)),
            residual=res, energy_err=abs(J - W) / max(abs(J), 1e-30),
            axial_balance=abs(KU[1::2, c][n_sol].sum()) / max(sumFz, 1e-30) if sumFz > 0 else 0.0,
            load_on_removed_pct=100 * np.abs(Fc[is_inact]).sum() / max(Fabs, 1e-30),
            load_on_supports_pct=100 * np.abs(Fc[is_supp]).sum() / max(Fabs, 1e-30)))
    return out, U, sol_e


# Inventory of the existing designs (nothing is optimized here)
b9_designs = []
for k in MAT_KEYS:
    b9_designs.append((k, "Base", None, rho_0, eval_solid[k], 1.0))
    p = selected.get((k, "Multi-load"))
    if p is not None:
        b9_designs.append((k, "Multi-load", None, p["rho_bin"], p["eval"], p["vf"]))
    for cc in range(n_cases):
        p = selected_single.get((k, cc))
        if p is not None:
            b9_designs.append((k, f"C{cc + 1}", cc, p["rho_bin"], p["eval"], p["vf"]))

print("\n" + "=" * 96)
print(f" 9.3 INDEPENDENT RE-EVALUATION OF {len(b9_designs)} EXISTING DESIGNS x {n_cases} LOAD CASES ({b9_solver})")
print("=" * 96)
b9_ops = {k: _b9_material_ops(MATERIALS[k]) for k in MAT_KEYS}
_ke_check = {k: float(np.max(np.abs(b9_ops[k]["ke_col"][_b9_col[act]] - MAT_OPS[k]["ke"])) / np.max(np.abs(MAT_OPS[k]["ke"])))
             for k in MAT_KEYS}
_C_check = {k: float(np.max(np.abs(b9_ops[k]["C"] - MAT_OPS[k]["C"])) / np.max(np.abs(MAT_OPS[k]["C"]))) for k in MAT_KEYS}
for k in MAT_KEYS:
    print(f"  [{k}] max rel. diff C vs MAT_OPS: {_C_check[k]:.1e} | k_e vs MAT_OPS (all {len(act):,} elements): {_ke_check[k]:.1e}"
          f" | eig(C) = {np.array2string(b9_ops[k]['eigC'], precision=1)}")

_rows, b9_eval = [], {}
for k, name, own, rb, stored, vf in b9_designs:
    t1 = time.time()
    ev, _, sol_e = b9_evaluate(rb, b9_ops[k], k)
    b9_eval[(k, name)] = ev
    mass = b9_ops[k]["rho"] * float(V_elem @ rb)
    for c in range(n_cases):
        e, s = ev[c], stored[c]
        rel = lambda a, b: abs(a - b) / max(abs(b), 1e-30)
        _rows.append(dict(material=k, design=name, case=f"C{c + 1}", own_case=(own == c) if own is not None else True,
                          vf_nominal=vf, vf_real=float(V_elem @ rb) / V0, mass_g=mass,
                          util_stored=s["util"], util_b9=e["util"], governs=e["governs"],
                          d_util=rel(e["util"], s["util"]), d_u_verde=rel(e["u_crit"], s["u_crit"]),
                          d_u_resto=rel(e["u_rem"], s["u_rem"]), d_vm_p995=rel(e["vm_p995"], s["vm_p995"]),
                          d_J=rel(e["J"], s["J"]), **{kk: e[kk] for kk in e if kk not in ("governs",)}))
    print(f"  {k:<7}{name:<11} evaluated in {time.time() - t1:5.1f} s | {len(sol_e):,} solid elements")
b9_repro = pd.DataFrame(_rows)
_dcols = ["d_util", "d_u_verde", "d_u_resto", "d_vm_p995", "d_J"]
b9_max_repro = float(b9_repro[_dcols].to_numpy().max())
print(f"  -> Max relative difference vs stored results (util, u_crit, u_rem, σ_p99.5, J): {b9_max_repro:.2e} "
      f"[{'REPRODUCED' if b9_max_repro < B9_CFG['TOL_REPRO'] else 'CHECK'}]")
_b9_save(b9_repro, "T02_reevaluation_all_designs_cases")

# Design-level summary (governing utilization: own case for per-case designs, all cases otherwise)
_rows = []
for k, name, own, rb, stored, vf in b9_designs:
    ev = b9_eval[(k, name)]
    cases = [own] if own is not None else list(range(n_cases))
    c_g = max(cases, key=lambda c: ev[c]["util"])
    m0 = b9_ops[k]["rho"] * V0
    mass = b9_ops[k]["rho"] * float(V_elem @ rb)
    _rows.append(dict(material=k, design=name, vf_nominal=vf, vf_real=float(V_elem @ rb) / V0,
                      mass_g=mass, mass_saving_pct=100 * (1 - mass / m0), governing_case=f"C{c_g + 1}",
                      util=ev[c_g]["util"], margin=1 - ev[c_g]["util"], governs=ev[c_g]["governs"],
                      util_all_cases_max=max(e["util"] for e in ev),
                      FS_VM_p995=ev[c_g]["FS"], FS_VM_max=b9_ops[k]["Sy"] / ev[c_g]["vm_max"]))
b9_summary = pd.DataFrame(_rows)
print("\n  DESIGN SUMMARY (util = governing utilization in the design's own case set)")
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 4):
    print(b9_summary.to_string(index=False))
_b9_save(b9_summary, "T03_design_summary")

## 9.4 Equilibrium, energy and load accounting

Worst values over all the designs and cases: residual, energy identity of Eq. (2.7), axial balance and load applied on removed or restrained nodes.

In [ ]:
_q = b9_repro
print("\n" + "=" * 96)
print(" 9.4 EQUILIBRIUM / ENERGY / LOAD ACCOUNTING (worst value over all designs and cases)")
print("=" * 96)
print(f"  max ||K_ff U - F_f|| / ||F_f||          : {_q['residual'].max():.2e}")
print(f"  max |F·U - U·K·U| / |F·U|               : {_q['energy_err'].max():.2e}")
print(f"  max |sum_z (K U)| / sum|F_z| (rigid mode): {_q['axial_balance'].max():.2e}")
print(f"  max load applied on REMOVED nodes [%]    : {_q['load_on_removed_pct'].max():.3f}")
print(f"  max load applied on SUPPORTED dofs [%]   : {_q['load_on_supports_pct'].max():.3f}  "
      f"(absorbed directly by the supports, never enters the structure)")
_axial = {}
for c in range(n_cases):
    Fz = F_design[1::2, c]
    _axial[f"C{c + 1}"] = dict(sum_Fz_N=float(Fz.sum()), sum_Fr_N=float(F_design[0::2, c].sum()))
_p_top = float(lam_c[0]) * P_NOM["comp"]
print(f"  C1 axial resultant: {_axial['C1']['sum_Fz_N']:,.1f} N  (λ·P_nom = {_p_top:.4g} MPa on the top face)")

## 9.5 Directional strength check (Sec. 5.9)

The in-plane tensile stress is compared with $S_{y,T}$, and the axial compressive stress with $S_y$. The interlaminar tensile and shear stresses are only reported, because their strengths were not characterized (T04).

In [ ]:
_rows = []
for k, name, own, rb, stored, vf in b9_designs:
    ev = b9_eval[(k, name)]
    ref = B9_CHAR_REF[k]; Sy = b9_ops[k]["Sy"]
    for c in range(n_cases):
        e = ev[c]
        u_terms = max(e["u_crit"] / U_ADM_CRIT, e["u_rem"] / U_ADM_REM)
        util_dir = max(u_terms, FS_VM * e["t_ip_p995"] / ref["Sy_t"], FS_VM * e["c_z_p995"] / Sy)
        _rows.append(dict(material=k, design=name, case=f"C{c + 1}",
                          own_case=(own == c) if own is not None else True,
                          vm_p995=e["vm_p995"], FS_VM=Sy / e["vm_p995"],
                          t_inplane_p995=e["t_ip_p995"], FS_tension_xy=ref["Sy_t"] / max(e["t_ip_p995"], 1e-30),
                          c_axial_p995=e["c_z_p995"], FS_compression_z=Sy / max(e["c_z_p995"], 1e-30),
                          t_interlaminar_p995=e["t_z_p995"], tau_rz_p995=e["tau_p995"],
                          util_VM=e["util"], util_directional=util_dir,
                          feasible_VM=e["util"] <= 1.0, feasible_directional=util_dir <= 1.0))
b9_dir = pd.DataFrame(_rows)
_own = b9_dir[b9_dir["own_case"]]
print("\n" + "=" * 96)
print(" 9.5 DIRECTIONAL STRENGTH CHECK (own cases) — Sy_t,xy from tension tests, Sy_c,z from compression")
print("=" * 96)
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 3):
    print(_own[["material", "design", "case", "FS_VM", "FS_tension_xy", "FS_compression_z", "t_interlaminar_p995",
                "tau_rz_p995", "util_VM", "util_directional", "feasible_directional"]].to_string(index=False))
_flip = _own[_own["feasible_VM"] != _own["feasible_directional"]]
print(f"  -> Feasibility changes when the directional criterion is used: {len(_flip)} design-case pairs")
_b9_save(b9_dir, "T04_directional_strength")

## 9.6 Robustness of the searches (Sec. 5.6)

Monotonicity, abrupt changes and margin of each of the ten searches (T05).

In [ ]:
_rows = []
_all_fronts = [((k, n), pts, selected.get((k, n))) for (k, n), pts in pareto.items()] + \
              [((k, f"C{cc + 1}"), pts, selected_single.get((k, cc))) for (k, cc), pts in pareto_single.items()]
for (k, n), pts, chosen in _all_fronts:
    q = sorted(pts, key=lambda p: p["vf"])
    vf_ = np.array([p["vf"] for p in q]); ut = np.array([p["util"] for p in q])
    non_mono = int(np.sum(np.diff(ut) > 0.02))               # more material but clearly worse utilization
    slope = np.abs(np.diff(ut)) / np.maximum(np.diff(vf_), 1e-9)
    row = dict(material=k, problem=n, n_runs=len(q), n_feasible=int(np.sum(ut <= 1.0)),
               non_monotonic_steps=non_mono, max_cliff_dutil_per_dvf=float(slope.max()) if len(slope) else np.nan)
    if chosen is not None:
        below = [p for p in q if p["vf"] < chosen["vf"] and not p["feasible"]]
        nb = max(below, key=lambda p: p["vf"]) if below else None
        row.update(chosen_vf=chosen["vf"], chosen_util=chosen["util"], chosen_margin=1 - chosen["util"],
                   nearest_infeasible_vf=nb["vf"] if nb else np.nan, nearest_infeasible_util=nb["util"] if nb else np.nan,
                   gap_vf=(chosen["vf"] - nb["vf"]) if nb else np.nan, iterations=chosen["nit"])
    _rows.append(row)
b9_pareto = pd.DataFrame(_rows)
print("\n" + "=" * 96)
print(" 9.6 ROBUSTNESS OF THE PARETO SEARCHES")
print("=" * 96)
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 4):
    print(b9_pareto.to_string(index=False))
_b9_save(b9_pareto, "T05_pareto_robustness")

## 9.7 Sensitivity to the interlaminar shear modulus (Table 5.10.1)

The stored geometries are re-evaluated, without re-optimizing them, with $\alpha_G = 0.32$, $0.41$ and Huber's value (T06).

In [ ]:
b9_sens = pd.DataFrame()
if B9_CFG["RUN_SENSITIVITY"]:
    print("\n" + "=" * 96)
    print(" 9.7 SENSITIVITY OF THE EXISTING DESIGNS (re-evaluation only, same loads and constraints)")
    print("=" * 96)
    _sets = [("nominal", None)] + [(f"alpha_G={a}", a) for a in B9_CFG["ALPHA_G_SWEEP"]]
    _rows = []
    for label, aG in _sets:
        for k in MAT_KEYS:
            props = dict(MATERIAL_CATALOG[k])
            if aG == "huber":
                aG_k = derive_constants(props, alpha_G=ALPHA_G)["alpha_huber"]
            else:
                aG_k = ALPHA_G if aG is None else float(aG)
            m = derive_constants(props, alpha_G=aG_k)
            ops = b9_ops[k] if label == "nominal" else _b9_material_ops(m)
            for kk, name, own, rb, stored, vf in b9_designs:
                if kk != k:
                    continue
                ev = b9_eval[(k, name)] if label == "nominal" else b9_evaluate(rb, ops, k)[0]
                cases = [own] if own is not None else list(range(n_cases))
                c_g = max(cases, key=lambda c: ev[c]["util"])
                _rows.append(dict(set=label, material=k, design=name, alpha_G=aG_k, E_z=m["E_z"], G_rz=m["G_rz"],
                                  util=ev[c_g]["util"], governing_case=f"C{c_g + 1}", governs=ev[c_g]["governs"],
                                  feasible=ev[c_g]["util"] <= 1.0))
        print(f"  set '{label}' done ({time.time() - _b9_t0:.0f} s elapsed)")
    b9_sens = pd.DataFrame(_rows)
    _piv = b9_sens.pivot_table(index=["material", "design"], columns="set", values="util", sort=False)
    with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 3):
        print(_piv.to_string())
    _b9_save(b9_sens, "T06_sensitivity_existing_designs")

## 9.8 Mesh study of the solid (Figure 5.2.1)

The original solid is evaluated under the axial reference load with meshes of 0.5, 1.0 and 1.5 times the reference density (T07). The factor 1.0 must reproduce Block 2.

In [ ]:
b9_mesh = pd.DataFrame()
if B9_CFG["RUN_MESH_STUDY"]:
    print("\n" + "=" * 96)
    print(f" 9.8 MESH STUDY OF THE SOLID BASELINE ({ACTIVE_MATERIAL}, Block 2 conditions, p = {P_BENCH} MPa)")
    print("=" * 96)
    _m_act = MATERIALS[ACTIVE_MATERIAL]
    _C_act = np.linalg.inv(_b9_compliance_matrix(_m_act)); _C_act = 0.5 * (_C_act + _C_act.T)
    _rows = []
    for f in B9_CFG["MESH_FACTORS"]:
        nr_, nz_ = int(round(f * nelr)), int(round(f * nelz))
        t1 = time.time()
        dr_, dz_ = (R_out - R_in) / nr_, H_part / nz_
        rg, zg = np.linspace(R_in, R_out, nr_ + 1), np.linspace(0.0, H_part, nz_ + 1)
        Rm, Zm = np.meshgrid(rg, zg); rn, zn = Rm.ravel(), Zm.ravel(); nn = rn.size; ne = nr_ * nz_
        jj_, ii_ = np.meshgrid(np.arange(nz_), np.arange(nr_), indexing="ij")
        n1 = (jj_ * (nr_ + 1) + ii_).ravel()
        con = np.column_stack([n1, n1 + 1, n1 + nr_ + 2, n1 + nr_ + 1])
        rc, zc = rn[con].mean(1), zn[con].mean(1)
        cr = np.zeros(ne, np.int32)                                # Jordan-curve parity voxelization
        for (r1, z1), (r2, z2) in contour_segments:
            cz = (z1 > zc) != (z2 > zc)
            if np.any(cz):
                cr[cz] += (r1 + (zc[cz] - z1) * (r2 - r1) / (z2 - z1)) > rc[cz]
        rho_m = (cr % 2 == 1)
        ea = np.where(rho_m)[0]; na = np.unique(con[ea])
        ke_c, CB_c = _b9_column_operators(_C_act, nr_, dr_, dz_, R_in)
        col_ = np.arange(ne) % nr_
        ged = np.empty((ne, 8), np.int64); ged[:, 0::2] = 2 * con; ged[:, 1::2] = 2 * con + 1
        K = sp.coo_matrix((ke_c[col_[ea]].reshape(len(ea), 64).ravel(),
                           (np.repeat(ged[ea], 8, 1).ravel(), np.tile(ged[ea], (1, 8)).ravel())),
                          shape=(2 * nn, 2 * nn)).tocsr()
        sl = na[zn[na] <= 1e-6]; em = na[rn[na] >= R_out - 0.5 * dr_]; ina = np.setdiff1d(np.arange(nn), na)
        fx = np.unique(np.concatenate([2 * sl + 1, 2 * em, 2 * em + 1, 2 * ina, 2 * ina + 1]))
        fr = np.setdiff1d(np.arange(2 * nn), fx)
        jj2, ii2 = np.divmod(np.arange(ne), nr_)
        up_void = np.ones(ne, bool); m_ = jj2 < nz_ - 1
        up_void[m_] = ~rho_m[(jj2[m_] + 1) * nr_ + ii2[m_]]
        et = np.where(rho_m & up_void & (zc >= H_part - 1.5 * dz_))[0]
        a_, b_ = con[et, 3], con[et, 2]; ra, rb_ = rn[a_], rn[b_]; L = np.abs(rb_ - ra)
        F = np.zeros(2 * nn)
        np.add.at(F, 2 * a_ + 1, -2 * np.pi * L * P_BENCH * (2 * ra + rb_) / 6)
        np.add.at(F, 2 * b_ + 1, -2 * np.pi * L * P_BENCH * (ra + 2 * rb_) / 6)
        U = np.zeros(2 * nn); U[fr] = np.asarray(_b9_solve(K[fr, :][:, fr], F[fr])).ravel()
        s = np.einsum("eik,ek->ei", CB_c[col_[ea]], U[ged[ea]])
        vm = np.sqrt(0.5 * ((s[:, 0] - s[:, 1]) ** 2 + (s[:, 1] - s[:, 2]) ** 2 + (s[:, 2] - s[:, 0]) ** 2) + 3 * s[:, 3] ** 2)
        Va = 2 * np.pi * rc[ea] * dr_ * dz_
        um = np.hypot(U[0::2], U[1::2])
        e_mx = ea[np.argmax(vm)]
        _rows.append(dict(factor=f, nelr=nr_, nelz=nz_, h_mm=dr_, n_dof_free=len(fr), V_mm3=float(Va.sum()),
                          A_top_mm2=float(np.sum(np.pi * (rb_ ** 2 - ra ** 2))), F_total_N=float(-F[1::2].sum()),
                          n_clamped_nodes=len(em), clamped_z_span_mm=float(np.ptp(zn[em])) if len(em) else 0.0,
                          n_sliding_nodes=len(sl),
                          J_Nmm=float(F @ U), u_max_mm=float(um[na].max()), vm_max=float(vm.max()),
                          vm_max_at=f"({rc[e_mx]:.2f}, {zc[e_mx]:.2f})", vm_p995=float(np.percentile(vm, 99.5)),
                          vm_p995_vol=_b9_wpercentile(vm, Va, 99.5), vm_p99_vol=_b9_wpercentile(vm, Va, 99.0),
                          t_s=time.time() - t1))
        print(f"  {nr_:>4} x {nz_:<4} h = {dr_:.4f} mm | J = {_rows[-1]['J_Nmm']:,.3f} | vm_max = {_rows[-1]['vm_max']:.3f}"
              f" at {_rows[-1]['vm_max_at']} | p99.5 = {_rows[-1]['vm_p995']:.3f} | {_rows[-1]['t_s']:.1f} s")
    b9_mesh = pd.DataFrame(_rows)
    _seg = np.asarray(contour_segments)
    _flat = (np.abs(_seg[:, 0, 1] - H_part) < 1e-6 * H_part) & (np.abs(_seg[:, 1, 1] - H_part) < 1e-6 * H_part)
    b9_A_top_exact = float(np.sum(np.pi * np.abs(_seg[_flat, 1, 0] ** 2 - _seg[_flat, 0, 0] ** 2)))
    b9_mesh["A_top_exact_mm2"] = b9_A_top_exact
    b9_mesh["A_top_error_pct"] = 100 * (b9_mesh["A_top_mm2"] - b9_A_top_exact) / b9_A_top_exact
    print(f"  -> Exact loaded area from the STL contour (flat segments at z = H): {b9_A_top_exact:,.1f} mm^2 | "
          f"voxelized: " + ", ".join(f"{a:,.1f}" for a in b9_mesh["A_top_mm2"]) + " mm^2")
    _cur = b9_mesh[b9_mesh["factor"] == 1.0]
    if len(_cur):
        print(f"  -> Reproduction of Block 2 at {nelr}x{nelz}: J rel. diff = "
              f"{abs(_cur['J_Nmm'].iloc[0] - J_0) / J_0:.2e} | vm_max rel. diff = "
              f"{abs(_cur['vm_max'].iloc[0] - vm_max_0) / vm_max_0:.2e} | p99.5 rel. diff = "
              f"{abs(_cur['vm_p995'].iloc[0] - vm_p995_0) / vm_p995_0:.2e}")
    _b9_save(b9_mesh, "T07_mesh_study_baseline")

## 9.9 Review of the virtual test

Recomputes $K_{ini}$ of the twelve designs independently (T08) and measures how far the material crosses the rigid platens, since Block 8 has no contact formulation (T09).

In [ ]:
print("\n" + "=" * 96)
print(f" 9.9 VIRTUAL COMPRESSION TEST REVIEW (Block 8: platens '{PLATEN_CONTACT}')")
print("=" * 96)
_r8 = pd.DataFrame(vtest_results)
_r8["eps_nom_at_Fmax_pct"] = 100 * _r8["strain_nom_Fmax"]
_r8["small_strain_valid(<=5%)"] = _r8["strain_nom_Fmax"] <= 0.05
_r8["F_per_mass_vs_base"] = np.nan
for k in _r8["material"].unique():
    base_ = _r8[(_r8["material"] == k) & (_r8["design"] == "Base")]
    if len(base_):
        mk = _r8["material"] == k
        _r8.loc[mk, "F_per_mass_vs_base"] = _r8.loc[mk, "F_spec_N_g"] / base_["F_spec_N_g"].iloc[0]
_r8["sigma_nom_over_Sy"] = [r["stress_nom_Fmax_MPa"] / MATERIALS[r["material"]]["Sy"] for _, r in _r8.iterrows()]

if B9_CFG["RUN_B8_ELASTIC_CHECK"]:
    _kin = []
    for _, r in _r8.iterrows():
        k, name = r["material"], r["design"]
        rb = rho_0 if name == "Base" else (selected[(k, "Multi-load")]["rho_bin"] if name == "Multi-load"
                                           else selected_single[(k, int(name[1:]) - 1)]["rho_bin"])
        sol_e = np.where((rb >= 0.5) & (rho_0 == 1.0))[0]; n_sol = np.unique(connectivity[sol_e])
        bot, top = n_sol[z_nodes[n_sol] <= 1e-6], n_sol[z_nodes[n_sol] >= H_part - 1e-6]
        ina = np.setdiff1d(np.arange(n_nodes), n_sol)
        fx = [2 * bot + 1, 2 * ina, 2 * ina + 1, 2 * top + 1]
        if PLATEN_CONTACT == "bonded":
            fx += [2 * bot, 2 * top]
        fx = np.unique(np.concatenate(fx)); fr = np.setdiff1d(np.arange(2 * n_nodes), fx)
        K = sp.coo_matrix((b9_ops[k]["ke_col"][_b9_col[sol_e]].reshape(len(sol_e), 64).ravel(),
                           (_b9_Iall[sol_e].ravel(), _b9_Jall[sol_e].ravel())), shape=(2 * n_nodes, 2 * n_nodes)).tocsr()
        U = np.zeros(2 * n_nodes); U[2 * top + 1] = -1.0                           # unit platen displacement
        U[fr] = np.asarray(_b9_solve(K[fr, :][:, fr], -(K[fr, :] @ U))).ravel()
        F1 = -float((K @ U)[2 * top + 1].sum())
        _kin.append(F1)
    _r8["K_ini_b9_N_mm"] = _kin
    _r8["d_K_ini"] = np.abs(_r8["K_ini_b9_N_mm"] - _r8["K_ini_N_mm"]) / _r8["K_ini_N_mm"]

b9_b8 = _r8
with pd.option_context("display.width", 260, "display.max_columns", 40, "display.precision", 4):
    cols8 = ["material", "design", "mass_g", "K_ini_N_mm", "F_max_N", "delta_Fmax_mm", "eps_nom_at_Fmax_pct",
             "small_strain_valid(<=5%)", "sigma_nom_over_Sy", "F_per_mass_vs_base"]
    cols8 += [c for c in ["K_ini_b9_N_mm", "d_K_ini"] if c in _r8]
    print(_r8[cols8].to_string(index=False))

b9_penetration = []
for k, name in B9_CFG["B8_RERUN_PENETRATION"]:
    if "virtual_compression_test" not in globals():
        print("  (virtual_compression_test not in memory: run Block 8 to enable the penetration check)"); break
    rb = rho_0 if name == "Base" else (selected[(k, "Multi-load")]["rho_bin"] if name == "Multi-load"
                                       else selected_single[(k, int(name[1:]) - 1)]["rho_bin"])
    t1 = time.time()
    _res, _h, _pk, _esol = virtual_compression_test(k, name, rb)
    n_sol = np.unique(connectivity[_esol]); Uf = _pk["U"]; zdef = z_nodes[n_sol] + Uf[2 * n_sol + 1]
    top_pl = H_part - _pk["delta"]
    pen_top = np.maximum(zdef - top_pl, 0.0); pen_bot = np.maximum(-zdef, 0.0)
    it_, ib_ = int(np.argmax(pen_top)), int(np.argmax(pen_bot))
    b9_penetration.append(dict(material=k, design=name, delta_Fmax_mm=_pk["delta"], F_max_N=_res["F_max_N"],
                               F_max_N_stored=float(_r8[(_r8.material == k) & (_r8.design == name)]["F_max_N"].iloc[0]),
                               max_pen_top_mm=float(pen_top.max()), n_nodes_pen_top=int((pen_top > 1e-3).sum()),
                               worst_top_node_rz=f"({r_nodes[n_sol[it_]]:.2f}, {z_nodes[n_sol[it_]]:.2f})",
                               max_pen_bottom_mm=float(pen_bot.max()), n_nodes_pen_bottom=int((pen_bot > 1e-3).sum()),
                               worst_bottom_node_rz=f"({r_nodes[n_sol[ib_]]:.2f}, {z_nodes[n_sol[ib_]]:.2f})",
                               t_s=time.time() - t1))
    print(f"  re-run {k} {name}: F_max = {_res['F_max_N'] / 1e3:.1f} kN | max penetration top/bottom = "
          f"{pen_top.max():.3f} / {pen_bot.max():.3f} mm ({int((pen_top > 1e-3).sum())} / {int((pen_bot > 1e-3).sum())} nodes)")
b9_penetration = pd.DataFrame(b9_penetration)
_b9_save(b9_b8, "T08_virtual_test_review")
if len(b9_penetration):
    _b9_save(b9_penetration, "T09_platen_penetration")

## 9.10 Figures F01 and F02

Reproduction, strength and sensitivity (F01), and mesh study (F02).

In [ ]:
_blue, _orange, _aqua, _gray = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
fig, axs = plt.subplots(1, 3, figsize=(17, 5.2), constrained_layout=True)
ax = axs[0]
for k, mk, colr in [("PA6", "o", _blue), ("PA6_CF", "s", _orange)]:
    d = b9_repro[b9_repro["material"] == k]
    ax.plot(d["util_stored"], d["util_b9"], mk, ms=8, color=colr, alpha=0.8, label=k, linestyle="none")
lim = [b9_repro[["util_stored", "util_b9"]].min().min() * 0.8, b9_repro[["util_stored", "util_b9"]].max().max() * 1.2]
ax.plot(lim, lim, ":", color=_gray, lw=1); ax.set_xlabel("utilization stored (Blocks 5-6)")
ax.set_ylabel("utilization recomputed (Block 9)"); ax.legend(fontsize=8)
ax.set_title(f"Reproduction of all design-case pairs\nmax rel. diff {b9_max_repro:.1e}", fontweight="bold")
ax = axs[1]
d = b9_dir[b9_dir["own_case"] & (b9_dir["design"] != "Base")]
lab = [f"{a}\n{b[:5]}" + ("" if b.startswith("C") else f"-{c}") for a, b, c in
       zip(d["material"], d["design"], d["case"])]
x = np.arange(len(d)); w = 0.27
ax.bar(x - w, FS_VM / d["FS_VM"], w, color=_blue, label="von Mises vs Sy_c (used in Blocks 5-6)")
ax.bar(x, FS_VM / d["FS_tension_xy"], w, color=_orange, label="in-plane tension vs Sy_t,xy")
ax.bar(x + w, FS_VM / d["FS_compression_z"], w, color=_aqua, label="axial compression vs Sy_c,z")
ax.set_xticks(x, lab, fontsize=6, rotation=90)
ax.set_ylabel(f"stress utilization {FS_VM:g}·σ_p99.5 / S [-]"); ax.legend(fontsize=7)
ax.set_title("Stress utilization by criterion (own cases)", fontweight="bold")
ax = axs[2]
if len(b9_sens):
    sets = list(dict.fromkeys(b9_sens["set"]))
    dd = b9_sens[b9_sens["design"] != "Base"]
    keys = list(dict.fromkeys(zip(dd["material"], dd["design"])))
    xx = np.arange(len(keys)); ww = 0.8 / len(sets)
    pal = [_blue, _orange, _aqua, "#eda100", "#6250d6", "#e87ba4"]
    for i, s_ in enumerate(sets):
        vals = [float(dd[(dd["set"] == s_) & (dd["material"] == a) & (dd["design"] == b)]["util"].iloc[0]) for a, b in keys]
        ax.bar(xx + (i - (len(sets) - 1) / 2) * ww, vals, ww, color=pal[i % len(pal)], label=s_)
    ax.axhline(1.0, color=_gray, ls=":", lw=1)
    ax.set_xticks(xx, [f"{a}\n{b}" for a, b in keys], fontsize=6.5, rotation=90)
    ax.set_ylabel("governing utilization [-]")
    ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=3)
    ax.set_title("Existing designs vs material uncertainty", fontweight="bold")
_b9_savefig(fig, "F01_reproduction_strength_sensitivity")
plt.show()

if len(b9_mesh):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
    axs[0].plot(b9_mesh["h_mm"], b9_mesh["vm_max"], "o-", color=_orange, lw=2, ms=8, label="σ_VM max")
    axs[0].plot(b9_mesh["h_mm"], b9_mesh["vm_p995"], "s-", color=_blue, lw=2, ms=8, label="σ_VM p99.5 (count)")
    axs[0].plot(b9_mesh["h_mm"], b9_mesh["vm_p995_vol"], "^-", color=_aqua, lw=2, ms=8, label="σ_VM p99.5 (volume)")
    axs[0].invert_xaxis(); axs[0].set_xlabel("element size h [mm]"); axs[0].set_ylabel("stress [MPa]")
    axs[0].legend(fontsize=8); axs[0].set_title("Baseline stress vs mesh", fontweight="bold")
    axs[1].plot(b9_mesh["h_mm"], b9_mesh["J_Nmm"], "o-", color=_blue, lw=2, ms=8)
    axs[1].axhline(J_0, color=_gray, ls=":", lw=1, label="Block 2 J_0")
    axs[1].invert_xaxis(); axs[1].set_xlabel("element size h [mm]"); axs[1].set_ylabel("compliance [N mm]")
    axs[1].legend(fontsize=8); axs[1].set_title("Baseline compliance vs mesh", fontweight="bold")
    for a in axs:
        a.grid(True, ls=":", alpha=0.4)
    _b9_savefig(fig, "F02_mesh_study_baseline")
    plt.show()

print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Block 9 completed in {(time.time() - _b9_t0) / 60:.1f} min"
      + (f" | outputs in '{b9_run_dir}'" if b9_run_dir else ""))

B9 = dict(cfg=B9_CFG, material=b9_material, reevaluation=b9_repro, summary=b9_summary,
            directional=b9_dir, pareto=b9_pareto, sensitivity=b9_sens, mesh=b9_mesh,
            virtual_test=b9_b8, penetration=b9_penetration, max_repro=b9_max_repro,
            ops=b9_ops, evals=b9_eval, run_dir=b9_run_dir)

---
# Block 10. Thesis figures and tables (Sec. 3.9)

No optimization runs here. Every topology is read from memory and only re-solved linearly, with the solver function of Block 5.

## 10.0 Configuration

Block 10 solves again the twelve designs in the four load cases and regenerates the figures and tables of Chapter 5, which the thesis cites by file name. The outputs are in `results/thesis_figures/`. Only the figures used in the thesis are shown here, and all of them are saved.

In [ ]:
_b10_t0 = time.time()
print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] BLOCK 10: thesis figures and tables")
B10_SHOWN = {"F10_01_load_cases_overview", "F10_02_base_constraint_ratios", "F10_03_pareto_grid_all", "F10_03_pareto_PA6_vs_PA6CF",
             "F10_04_topology_gallery", "F10_04_mass_comparison", "F10_04_constraint_breakdown", "F10_04_cross_utilization",
             "F10_06_virtual_test_curves"}

CFG = dict(
    OUT_ROOT=str(DIR_THESIS),
    DPI=300,
    SAVE_PDF=False,                # True: also a PDF per figure (vector text, fields rasterized at PDF_DPI);
    PDF_DPI=200,                   #       costs ~0.3 GB RAM per large figure while saving
    SHOW=True,                     # display the figures used in the thesis (all are always saved)
    DEF_SCALE="auto",              # displacement magnification; 'auto' -> 0.5 mm becomes 4 % of the section width
    ZOOM_HALF_MM=6.0,              # half-size of the geometric zoom window around the critical point [mm]
    DEV_ZOOM_HALF_MM=15.0,         # half-width of the contour-development zoom [mm of contour length]
    BASE_TOLERANCE_FIGURES=True,   # also produce tolerance figures for the base part (reference)
    S_TICK_MM=50.0,                # spacing of the contour-length markers drawn on the sections [mm]
)
required = ["MATERIALS", "MAT_KEYS", "nelr", "nelz", "n_elem", "n_nodes", "R_in", "R_out", "H_part",
            "r_nodes", "z_nodes", "connectivity", "centroid_r", "centroid_z", "rho_0", "V_elem", "V0",
            "R_mesh", "Z_mesh", "contour_segments", "nodes_clamped", "nodes_sliding", "nodes_crit",
            "nodes_rem", "CRIT_ZONE_1", "CRIT_ZONE_2", "U_ADM_CRIT", "U_ADM_REM", "FS_VM", "MAT_OPS", "F_design",
            "P_NOM", "lam_c", "n_cases", "evaluate_design", "vm_field", "eval_solid", "selected", "pareto",
            "selected_single", "pareto_single", "top_face", "cavity_bottom", "cavity_right", "bore_wall", "cavity_left", "shell_2d",
            "vtest_results", "vtest_curves", "PLATEN_CONTACT"]
missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(f"Missing variables (run Blocks 1, 2, 3, 5, 6 and 8 first): {missing}")
if r_nodes.size != n_nodes or rho_0.size != n_elem or F_design.shape != (2 * n_nodes, n_cases):
    raise RuntimeError("Inconsistent kernel state: restart and run Blocks 1 -> 8 in order.")

run_dir = CFG["OUT_ROOT"]
subdirs = {k: os.path.join(run_dir, v) for k, v in dict(
    load="01_load_cases", base="02_base_response", pareto="03_pareto", designs="04_designs",
    tol="05_deformation_tolerance", test="06_virtual_test", tables="07_tables").items()}
for d in subdirs.values():
    os.makedirs(d, exist_ok=True)
print(f"  output folder: {run_dir}")

## Style and shared data

Colors, pressures and designs used by the whole block.

In [ ]:
COL = {"PA6": "#2a78d6", "PA6_CF": "#eb6834"}            # material identity (fixed order)
MRK = {"PA6": "o", "PA6_CF": "s"}
MAT_LABEL = {"PA6": "PA6", "PA6_CF": "PA6-CF"}
C_GREEN, C_REST, C_GRAY, C_TEXT = "#008300", "#2a78d6", "#8a8986", "#52514e"
C_AXIAL, C_INT, C_RAD, C_SUP = "#6250d6", "#e34948", "#c98500", "#0b0b0b"
CASE_EN = ["C1 Axial compression", "C2 Internal pressure", "C3 Combined (axial + internal)", "C4 Radial pressure (+r)"]
DES_ORDER = ["C1", "C2", "C3", "C4", "Multi-load"]
rc = {"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
      "axes.edgecolor": C_TEXT, "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": ":",
      "figure.dpi": 100, "savefig.dpi": CFG["DPI"]}
captions = []

def save(fig, key, name, caption):
    path = os.path.join(subdirs[key], name)
    fig.savefig(path + ".png", dpi=CFG["DPI"], bbox_inches="tight")
    if CFG["SAVE_PDF"]:
        fig.savefig(path + ".pdf", dpi=CFG["PDF_DPI"], bbox_inches="tight")
    captions.append((os.path.relpath(path + ".png", run_dir), caption))
    if CFG["SHOW"] and name in B10_SHOWN:
        plt.show()
    plt.close(fig)
    del fig
    gc.collect()

p = dict(C1_top=lam_c[0] * P_NOM["comp"], C2_int=lam_c[1] * P_NOM["int"],
         C3_top=lam_c[2] * P_NOM["comp_comb"], C3_int=lam_c[2] * P_NOM["int_comb"],
         C4_rad=lam_c[3] * P_NOM["radial"])
LOADS = [[("top", p["C1_top"], C_AXIAL)],
         [("int", p["C2_int"], C_INT)],
         [("top", p["C3_top"], C_AXIAL), ("int", p["C3_int"], C_INT)],
         [("rad", p["C4_rad"], C_RAD)]]
LOAD_TXT = [f"p = {p['C1_top']:.3g} MPa on the top faces",
            f"p_i = {p['C2_int']:.3g} MPa on the inner cavity",
            f"p = {p['C3_top']:.3g} MPa (top) + p_i = {p['C3_int']:.3g} MPa (cavity)",
            f"p_r = {p['C4_rad']:.3g} MPa on the inner radial walls"]

designs = {}
for k in MAT_KEYS:
    m0 = MAT_OPS[k]["rho"] * V0
    designs[(k, "Base")] = dict(rb=rho_0, vf=1.0, ev=eval_solid[k], own=list(range(n_cases)), m0=m0)
    pm = selected.get((k, "Multi-load"))
    if pm is not None:
        designs[(k, "Multi-load")] = dict(rb=pm["rho_bin"], vf=pm["vf"], ev=pm["eval"], own=list(range(n_cases)), m0=m0)
    for cc in range(n_cases):
        pc = selected_single.get((k, cc))
        if pc is not None:
            designs[(k, f"C{cc + 1}")] = dict(rb=pc["rho_bin"], vf=pc["vf"], ev=pc["eval"], own=[cc], m0=m0)
for key, d in designs.items():
    d["mass"] = MAT_OPS[key[0]]["rho"] * float(V_elem @ d["rb"])
    d["saving"] = 100.0 * (1.0 - d["mass"] / d["m0"])
    d["cg"] = max(d["own"], key=lambda c: d["ev"][c]["util"])          # governing own case
    d["util"] = d["ev"][d["cg"]]["util"]
print(f"  solving {len(designs)} designs x {n_cases} load cases (Block 5 solver)...")
for key, d in designs.items():
    _, d["U"] = evaluate_design(d["rb"], MAT_OPS[key[0]], F_design)
    d["n_sol"] = np.unique(connectivity[np.where(d["rb"] >= 0.5)[0]])

## Contour of the component

Ordered loops of the meridian contour, and the position of each control node along the contour length $s$.

In [ ]:
segs = np.asarray(contour_segments, float)
ends = segs.reshape(-1, 2)
tree = cKDTree(ends)
tol_join = 1e-3
used = np.zeros(len(segs), bool)
loops = []
for i0 in range(len(segs)):
    if used[i0]:
        continue
    used[i0] = True
    chain = [segs[i0, 0], segs[i0, 1]]
    cur = segs[i0, 1]
    while True:
        nxt = None
        for j in tree.query_ball_point(cur, tol_join):
            if not used[j // 2]:
                nxt = (j // 2, j % 2); break
        if nxt is None:
            break
        used[nxt[0]] = True
        cur = segs[nxt[0], 1 - nxt[1]]
        chain.append(cur)
    c = np.array(chain)
    keep = np.r_[True, np.linalg.norm(np.diff(c, axis=0), axis=1) > 1e-9]
    c = c[keep]
    if np.linalg.norm(c[0] - c[-1]) < tol_join:
        c = c[:-1]
    if len(c) >= 3:
        loops.append(c)
loops.sort(key=lambda c: -np.sum(np.linalg.norm(np.diff(np.vstack([c, c[:1]]), axis=0), axis=1)))
loop_info, s_offset = [], 0.0
for li, c in enumerate(loops):
    cc_ = np.vstack([c, c[:1]])
    A2 = np.sum(cc_[:-1, 0] * cc_[1:, 1] - cc_[1:, 0] * cc_[:-1, 1])
    is_hole = any(MplPath(o).contains_point(c[0]) for o in loops[:li])
    sgn = (1.0 if A2 > 0 else -1.0) * (-1.0 if is_hole else 1.0)
    d_ = np.diff(cc_, axis=0); L = np.linalg.norm(d_, axis=1)
    nrm = sgn * np.column_stack([d_[:, 1], -d_[:, 0]]) / L[:, None]          # outward (away from solid)
    s0 = s_offset + np.r_[0.0, np.cumsum(L)[:-1]]
    loop_info.append(dict(P=cc_, L=L, nrm=nrm, s0=s0, perim=L.sum(), hole=is_hole))
    s_offset += L.sum() + 5.0
A_all = np.vstack([li["P"][:-1] for li in loop_info]); B_all = np.vstack([li["P"][1:] for li in loop_info])
N_all = np.vstack([li["nrm"] for li in loop_info]); S0_all = np.concatenate([li["s0"] for li in loop_info])
L_all = np.concatenate([li["L"] for li in loop_info])
print(f"  contour: {len(loops)} closed loop(s), perimeter {sum(li['perim'] for li in loop_info):.1f} mm")

cn = np.union1d(nodes_crit, nodes_rem)                      # nodes where the displacement limits apply
is_green = np.isin(cn, nodes_crit)
tol_node = np.where(is_green, U_ADM_CRIT, U_ADM_REM)
P = np.column_stack([r_nodes[cn], z_nodes[cn]])
AB = B_all - A_all
t = np.clip(np.einsum("nsk,sk->ns", P[:, None, :] - A_all[None], AB) / (L_all ** 2)[None], 0, 1)
dist = np.linalg.norm(P[:, None, :] - (A_all[None] + t[..., None] * AB[None]), axis=2)
js = np.argmin(dist, axis=1)
s_node = S0_all[js] + t[np.arange(len(cn)), js] * L_all[js]
n_node = N_all[js]
order = np.argsort(s_node)
cn, is_green, tol_node, s_node, n_node = cn[order], is_green[order], tol_node[order], s_node[order], n_node[order]

def in_box(r, z, b):
    return (r >= b["r"][0]) & (r <= b["r"][1]) & (z >= b["z"][0]) & (z <= b["z"][1])

W = R_out - R_in
M = 0.04 * W / U_ADM_REM if CFG["DEF_SCALE"] == "auto" else float(CFG["DEF_SCALE"])

## Drawing helpers

Functions shared by the figures: deformed section, admissible tolerance band and displacement along the contour.

In [ ]:
def frame(ax, pad=3.0):
    ax.set_aspect("equal"); ax.set_xlim(R_in - pad, R_out + pad); ax.set_ylim(-pad, H_part + pad)
    ax.set_xlabel("r [mm]"); ax.set_ylabel("z [mm]")

def draw_contour(ax, **kw):
    for li in loop_info:
        ax.plot(li["P"][:, 0], li["P"][:, 1], **kw)

def draw_solid(ax, rb, color="#d9d8d4", alpha=1.0, z=1):
    ax.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(rb.reshape(nelz, nelr) < 0.5, np.ones((nelz, nelr))),
                  cmap=ListedColormap([color]), alpha=alpha, shading="flat",
                  rasterized=True, zorder=z)

_smp, _gz = [], []
for li in loop_info:
    for a, b in zip(li["P"][:-1], li["P"][1:]):
        nseg = max(2, int(np.ceil(np.linalg.norm(b - a) / 0.03)))
        q = a + np.linspace(0, 1, nseg, endpoint=False)[:, None] * (b - a)
        _smp.append(q)
_smp = np.vstack(_smp)
_gz = in_box(_smp[:, 0], _smp[:, 1], CRIT_ZONE_1) | in_box(_smp[:, 0], _smp[:, 1], CRIT_ZONE_2)
_tree_c = cKDTree(_smp)

def tolerance_band(ax, scale, window, res):
    """Admissible band: every point closer to the base contour than scale*tol (tol of the nearest contour
    point: 0.1 mm on the critical walls, 0.5 mm elsewhere). Exact at corners (distance field, no offsets)."""
    (r0, r1), (z0, z1) = window
    rr, zz = np.meshgrid(np.arange(r0, r1 + res, res), np.arange(z0, z1 + res, res))
    dd, ix = _tree_c.query(np.column_stack([rr.ravel(), zz.ravel()]))
    g = _gz[ix].reshape(rr.shape); f = dd.reshape(rr.shape) - scale * np.where(g, U_ADM_CRIT, U_ADM_REM)
    inside = f <= 0
    ax.pcolormesh(rr, zz, np.ma.masked_where(~(inside & g), np.ones_like(f)), cmap=ListedColormap([C_GREEN]),
                  alpha=0.30, shading="auto", rasterized=True, zorder=2)
    ax.pcolormesh(rr, zz, np.ma.masked_where(~(inside & ~g), np.ones_like(f)), cmap=ListedColormap([C_REST]),
                  alpha=0.18, shading="auto", rasterized=True, zorder=2)
    ax.contour(rr, zz, f, levels=[0], colors=[C_TEXT], linewidths=0.5, zorder=4)

def s_markers(ax):
    step = CFG["S_TICK_MM"]
    for li in loop_info:
        sv = np.arange(np.ceil(li["s0"][0] / step) * step, li["s0"][0] + li["perim"], step)
        for sv_ in sv:
            j = np.searchsorted(li["s0"], sv_, side="right") - 1
            f = (sv_ - li["s0"][j]) / li["L"][j]
            q = li["P"][j] + f * (li["P"][j + 1] - li["P"][j])
            ax.plot(*q, "|", color=C_TEXT, ms=6, zorder=6)
            ax.annotate(f"s={sv_:.0f}", q, xytext=(3, 3), textcoords="offset points", fontsize=6, color=C_TEXT)

def deformed_polys(ax, d, c, scale, cmap="Blues", vmax=None, alpha=1.0, window=None):
    e = np.where(d["rb"] >= 0.5)[0]
    if window is not None:
        (r0, r1), (z0, z1) = window
        e = e[(centroid_r[e] > r0 - 2) & (centroid_r[e] < r1 + 2) & (centroid_z[e] > z0 - 2) & (centroid_z[e] < z1 + 2)]
    U = d["U"][:, c]; ur, uz = U[0::2], U[1::2]; um = np.hypot(ur, uz)
    conn = connectivity[e]
    verts = np.stack([r_nodes[conn] + scale * ur[conn], z_nodes[conn] + scale * uz[conn]], axis=-1)
    pc = PolyCollection(verts, array=um[conn].mean(1), cmap=cmap, edgecolors="face", linewidths=0.1,
                        alpha=alpha, rasterized=True, zorder=3)
    pc.set_clim(0, vmax if vmax is not None else um[d["n_sol"]].max())
    ax.add_collection(pc)
    return pc

def node_data(d, c):
    U = d["U"][:, c]; ur, uz = U[0::2][cn], U[1::2][cn]
    um = np.hypot(ur, uz); un = ur * n_node[:, 0] + uz * n_node[:, 1]
    alive = np.isin(cn, d["n_sol"])
    ratio = np.where(alive, um / tol_node, np.nan)
    return ur, uz, um, un, alive, ratio

def development(ax, d, c, color, s_window=None, title=None):
    ur, uz, um, un, alive, ratio = node_data(d, c)
    s = s_node
    # tolerance band (step) and green-wall spans
    ax.fill_between(s, -tol_node, tol_node, step="mid", color=C_REST, alpha=0.10, lw=0, label="tolerance band ±0.5 mm")
    gmask = is_green.astype(float)
    ax.fill_between(s, -U_ADM_CRIT * gmask, U_ADM_CRIT * gmask, where=is_green, step="mid", color=C_GREEN,
                    alpha=0.25, lw=0, label="critical walls ±0.1 mm")
    ax.step(s, tol_node, where="mid", color=C_TEXT, lw=0.8); ax.step(s, -tol_node, where="mid", color=C_TEXT, lw=0.8)
    ax.axhline(0, color="k", ls="--", lw=1.0, label="base geometry (u = 0)")
    ax.plot(s[alive], un[alive], "-", color=color, lw=1.4, label="normal displacement u·n")
    ax.plot(s[alive], um[alive], ":", color=C_TEXT, lw=1.2, label="|u| (constraint measure)")
    k_ = int(np.nanargmax(ratio))
    ax.plot(s[k_], um[k_], "*", ms=13, color="#d03b3b", mec="k", zorder=6,
            label=f"critical: |u| = {um[k_]:.4f} mm ({100 * ratio[k_]:.1f} % of limit)")
    ax.set_xlabel("contour length s [mm]"); ax.set_ylabel("displacement [mm]")
    if s_window is not None:
        ax.set_xlim(*s_window)
        sel = (s >= s_window[0]) & (s <= s_window[1]) & alive
        lim = max(np.nanmax(np.abs(np.r_[un[sel], um[sel]])) if sel.any() else 0, tol_node[sel].max() if sel.any() else 0.5)
        ax.set_ylim(-1.15 * lim, 1.15 * lim)
    else:
        ax.set_xlim(s.min(), s.max()); ax.set_ylim(-1.15 * U_ADM_REM, 1.15 * U_ADM_REM)
    if title:
        ax.set_title(title, fontweight="bold")
    return k_

def tolerance_figure(key, c, color):
    k, name = key; d = designs[key]; e = d["ev"][c]
    ur, uz, um, un, alive, ratio = node_data(d, c)
    kc = int(np.nanargmax(ratio)); nc = cn[kc]
    rc_, zc_ = r_nodes[nc], z_nodes[nc]
    fig = plt.figure(figsize=(15, 10.5), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, height_ratios=[1.35, 1], width_ratios=[1.15, 1])
    # (a) magnified deformed section
    ax = fig.add_subplot(gs[0, 0])
    draw_solid(ax, d["rb"], color="#efeeec")
    pc = deformed_polys(ax, d, c, M, alpha=0.85)
    draw_contour(ax, color="k", ls="--", lw=0.9, zorder=5)
    tolerance_band(ax, M, ((R_in - 3, R_out + 3), (-3, H_part + 3)), 0.12)
    ax.plot(r_nodes[nodes_crit], z_nodes[nodes_crit], ".", color=C_GREEN, ms=3, zorder=6)
    ax.plot(rc_ + M * ur[kc], zc_ + M * uz[kc], "*", ms=15, color="#d03b3b", mec="k", zorder=7)
    h = CFG["ZOOM_HALF_MM"]
    ax.add_patch(plt.Rectangle((rc_ - h, zc_ - h), 2 * h, 2 * h, fill=False, ec="#d03b3b", lw=1.2, zorder=7))
    s_markers(ax)
    fig.colorbar(pc, ax=ax, fraction=0.04, pad=0.02).set_label("|u| [mm]")
    frame(ax)
    ax.set_title(f"(a) {MAT_LABEL[k]} | {name} under {CASE_EN[c]} — displacements ×{M:.1f}\n"
                 f"dashed: base contour | shaded: admissible band ±0.1 (green) / ±0.5 mm (blue), same ×{M:.1f}", fontweight="bold")
    # (b) zoom at the critical point (exact |u| criterion = circle of radius tol around the original node)
    ax = fig.add_subplot(gs[0, 1])
    win = ((rc_ - h, rc_ + h), (zc_ - h, zc_ + h))
    draw_solid(ax, d["rb"], color="#efeeec")
    deformed_polys(ax, d, c, M, alpha=0.55, window=win)
    draw_contour(ax, color="k", ls="--", lw=1.2, zorder=5)
    tolerance_band(ax, M, win, 0.02)
    sel = alive & (np.abs(r_nodes[cn] - rc_) < h) & (np.abs(z_nodes[cn] - zc_) < h)
    sc = ax.scatter(r_nodes[cn][sel] + M * ur[sel], z_nodes[cn][sel] + M * uz[sel], c=ratio[sel], cmap="Oranges",
                    vmin=0, vmax=1, s=22, edgecolors=np.where(ratio[sel] > 1, "#d03b3b", "k"), linewidths=0.4, zorder=7)
    tl = tol_node[kc] * M
    ax.add_patch(Circle((rc_, zc_), tl, fill=False, ec=C_GREEN if is_green[kc] else C_REST, lw=2.0, ls="-", zorder=8))
    ax.annotate("", xy=(rc_ + M * ur[kc], zc_ + M * uz[kc]), xytext=(rc_, zc_),
                arrowprops=dict(arrowstyle="->", color="#d03b3b", lw=1.8), zorder=9)
    ax.plot(rc_, zc_, "o", color="k", ms=4, zorder=9)
    ax.text(0.02, 0.02, f"critical node (r, z) = ({rc_:.2f}, {zc_:.2f}) mm\n|u| = {um[kc]:.4f} mm  vs  limit "
            f"{tol_node[kc]:.1f} mm  ({100 * ratio[kc]:.1f} %)\ncircle: admissible |u| (×{M:.1f})",
            transform=ax.transAxes, fontsize=8, va="bottom", bbox=dict(fc="white", ec=C_GRAY, alpha=0.9))
    fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02).set_label("|u| / admissible [-]")
    ax.set_xlim(*win[0]); ax.set_ylim(*win[1]); ax.set_aspect("equal"); ax.set_xlabel("r [mm]"); ax.set_ylabel("z [mm]")
    ax.set_title("(b) Zoom at the point closest to its tolerance", fontweight="bold")
    # (c) development along the whole contour and (d) zoom
    ax = fig.add_subplot(gs[1, 0])
    development(ax, d, c, color, title="(c) Displacement along the full contour vs tolerance")
    ax.legend(loc="upper left", ncol=2, fontsize=7)
    ax = fig.add_subplot(gs[1, 1])
    hw = CFG["DEV_ZOOM_HALF_MM"]
    development(ax, d, c, color, s_window=(s_node[kc] - hw, s_node[kc] + hw), title="(d) Zoom of (c) at the critical point")
    fig.suptitle(f"{MAT_LABEL[k]} — {name}: mass {d['mass']:.1f} g | u_green = {e['u_crit']:.4f} mm (≤ 0.1) | "
                 f"u_rest = {e['u_rem']:.4f} mm (≤ 0.5) | FS = {e['FS']:.2f} | utilization {e['util']:.3f}",
                 fontsize=11, fontweight="bold")
    cap = (f"{MAT_LABEL[k]}, {name} design under load case {CASE_EN[c]} ({LOAD_TXT[c]}). (a) Deformed meridian "
           f"section with displacements magnified ×{M:.1f}; the dashed line is the base contour and "
           f"the shaded band is the admissible region (±0.1 mm on the critical walls, ±0.5 mm elsewhere) at the same magnification. "
           f"(b) Zoom at the most critical contour node: the circle is the admissible displacement and the arrow the "
           f"computed displacement (|u| = {um[kc]:.4f} mm, {100 * ratio[kc]:.1f} % of its {tol_node[kc]:.1f} mm limit). "
           f"(c) Normal displacement and |u| along the contour length s against the tolerance band; (d) zoom of (c).")
    return fig, cap

rc_backup = plt.rcParams.copy()
plt.rcParams.update(rc)

## 10.1 Load cases (F10_01, Figure 4.1.2)

Supports, loaded surfaces and critical walls on the meridian section.

In [ ]:
print("  10.1 load cases")
jj, ii = np.divmod(np.arange(n_elem), nelr)
def edge_arrows(mask, na, nb, direction):
    a, b = connectivity[mask, na], connectivity[mask, nb]
    return 0.5 * (r_nodes[a] + r_nodes[b]), 0.5 * (z_nodes[a] + z_nodes[b]), direction
EDGES = {"top": [edge_arrows(top_face, 3, 2, (0, -1))],
         "int": [edge_arrows(cavity_bottom, 3, 2, (0, -1)), edge_arrows(cavity_right, 0, 3, (1, 0)), edge_arrows(cavity_left, 1, 2, (-1, 0))],
         "rad": [edge_arrows(cavity_right, 0, 3, (1, 0)), edge_arrows(bore_wall, 0, 3, (1, 0))]}
pmax = max(v for v in p.values())

def load_case_axes(ax, c, show_legend=True):
    draw_solid(ax, rho_0, color="#e3e2de")
    draw_contour(ax, color="k", lw=0.8, zorder=5)
    em = nodes_clamped; sl = nodes_sliding
    ax.plot(r_nodes[em], z_nodes[em], "-", color=C_SUP, lw=3.0, zorder=6)
    for zz in np.linspace(z_nodes[em].min(), z_nodes[em].max(), 14):
        ax.plot([R_out, R_out + 1.8], [zz, zz + 1.8], color=C_SUP, lw=0.8, zorder=6)
    ax.plot(r_nodes[sl][::6], z_nodes[sl][::6] - 0.9, "o", mfc="white", mec=C_SUP, ms=4, zorder=6)
    ax.plot(r_nodes[nodes_crit], z_nodes[nodes_crit], "o", color=C_GREEN, ms=3, zorder=7)
    for kind, val, colr in LOADS[c]:
        Lr = 7.0 * val / pmax
        for (rm, zm, (dr_, dz_)) in EDGES[kind]:
            st = max(1, len(rm) // 14)
            ax.quiver(rm[::st] - dr_ * Lr, zm[::st] - dz_ * Lr, dr_ * Lr * np.ones_like(rm[::st]),
                      dz_ * Lr * np.ones_like(zm[::st]), angles="xy", scale_units="xy", scale=1, color=colr,
                      width=0.004, headwidth=4, zorder=8)
    fz = abs(float(F_design[1::2, c].sum()))
    frame(ax, pad=5)
    ax.set_title(f"{CASE_EN[c]}\n{LOAD_TXT[c]}\naxial resultant |F_z| = {fz:,.0f} N", fontweight="bold", fontsize=9)
    if show_legend:
        h_ = [Line2D([], [], color=C_SUP, lw=3, label="clamped outer wall (u_r = u_z = 0)"),
              Line2D([], [], marker="o", ls="", mfc="white", mec=C_SUP, label="sliding base (u_z = 0)"),
              Line2D([], [], marker="o", ls="", color=C_GREEN, label="critical walls |u| ≤ 0.1 mm"),
              Line2D([], [], color="k", lw=0.8, label="rest of contour |u| ≤ 0.5 mm"),
              Line2D([], [], color=C_AXIAL, marker=">", label="axial pressure"),
              Line2D([], [], color=C_INT, marker=">", label="internal pressure"),
              Line2D([], [], color=C_RAD, marker=">", label="radial pressure")]
        return h_
    return None

fig, axs = plt.subplots(1, 4, figsize=(19, 7.0), constrained_layout=True)
for c in range(n_cases):
    h_ = load_case_axes(axs[c], c, show_legend=(c == 0)) or h_
fig.legend(handles=h_, loc="lower center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Load cases and boundary conditions on the base part (meridian section, axisymmetric model)",
             fontsize=12, fontweight="bold")
save(fig, "load", "F10_01_load_cases_overview",
     "Load cases C1-C4 applied to the base part (meridian section). Black hatched line: clamped outer wall; "
     "open circles: sliding base; green dots: critical walls with |u| ≤ 0.1 mm; rest of the contour: |u| ≤ 0.5 mm. "
     "Arrow length is proportional to the applied pressure. " + "; ".join(f"{CASE_EN[c]}: {LOAD_TXT[c]}" for c in range(n_cases)) + ".")
for c in range(n_cases):
    fig, ax = plt.subplots(figsize=(7.5, 8.6), constrained_layout=True)
    h1 = load_case_axes(ax, c)
    fig.legend(handles=h1, loc="lower center", ncol=2, fontsize=7.5, bbox_to_anchor=(0.5, -0.09))
    save(fig, "load", f"F10_01_load_case_C{c + 1}", f"Load case {CASE_EN[c]}: {LOAD_TXT[c]}; same symbols as the overview figure.")

## 10.2 Response of the original component (F10_02, Figure 5.4.1, Table 5.4.1)

Deformed shape, von Mises stress and displacement against the tolerances, in the four load cases and both materials.

In [ ]:
print("  10.2 base response")
for k in MAT_KEYS:
    d = designs[(k, "Base")]
    vms = [vm_field(MAT_OPS[k], d["U"], c) for c in range(n_cases)]
    vcap = max(np.percentile(v[rho_0 == 1], 99.5) for v in vms)
    ucap = max(np.hypot(d["U"][0::2, c], d["U"][1::2, c])[d["n_sol"]].max() for c in range(n_cases))
    fig = plt.figure(figsize=(17, 20), constrained_layout=True)
    gs = fig.add_gridspec(n_cases, 3, width_ratios=[1, 1, 1.6])
    for c in range(n_cases):
        e = d["ev"][c]
        ax = fig.add_subplot(gs[c, 0])
        pc = deformed_polys(ax, d, c, M, vmax=ucap)
        draw_contour(ax, color="k", ls="--", lw=0.8, zorder=5)
        frame(ax); ax.set_title(f"{CASE_EN[c]}\n|u| (×{M:.1f}), max {np.hypot(d['U'][0::2, c], d['U'][1::2, c])[d['n_sol']].max():.3f} mm",
                                fontweight="bold")
        fig.colorbar(pc, ax=ax, fraction=0.05, pad=0.02).set_label("|u| [mm]")
        ax = fig.add_subplot(gs[c, 1])
        pv = ax.pcolormesh(R_mesh, Z_mesh, np.ma.masked_where(rho_0.reshape(nelz, nelr) < 0.5, vms[c].reshape(nelz, nelr)),
                           cmap="Purples", vmin=0, vmax=vcap, shading="flat", rasterized=True)
        draw_contour(ax, color="k", lw=0.6, zorder=5)
        frame(ax); ax.set_title(f"σ_VM, p99.5 = {e['vm_p995']:.2f} MPa | FS = {e['FS']:.1f}", fontweight="bold")
        fig.colorbar(pv, ax=ax, fraction=0.05, pad=0.02, extend="max").set_label("σ_VM [MPa]")
        ax = fig.add_subplot(gs[c, 2])
        development(ax, d, c, COL[k], title=f"u_green = {e['u_crit']:.4f} mm | u_rest = {e['u_rem']:.4f} mm | "
                                            f"utilization {e['util']:.3f}")
        if c == 0:
            ax.legend(loc="upper left", ncol=2, fontsize=7)
    fig.suptitle(f"Base part response — {MAT_LABEL[k]} (E_z = {MATERIALS[k]['E_z']:.2f} MPa, Sy = {MATERIALS[k]['Sy']:.2f} MPa)",
                 fontsize=13, fontweight="bold")
    save(fig, "base", f"F10_02_base_response_{k}",
         f"Response of the base part in {MAT_LABEL[k]} to the four load cases. Left: magnified deformed shape "
         f"(×{M:.1f}) coloured by |u| with the base contour dashed. Centre: von Mises stress (colour scale capped at "
         f"the 99.5th percentile, {vcap:.2f} MPa). Right: displacement along the contour against the ±0.1 / ±0.5 mm tolerances.")
    if CFG["BASE_TOLERANCE_FIGURES"]:
        for c in range(n_cases):
            fig, cap = tolerance_figure((k, "Base"), c, COL[k])
            save(fig, "base", f"F10_02_base_tolerance_{k}_C{c + 1}", cap)

# base comparison bars
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
x = np.arange(n_cases); w = 0.38
for i, k in enumerate(MAT_KEYS):
    ev = designs[(k, "Base")]["ev"]
    for ax, key, lim, lab in [(axs[0], "u_crit", U_ADM_CRIT, "u_green / 0.1 mm"),
                              (axs[1], "u_rem", U_ADM_REM, "u_rest / 0.5 mm"),
                              (axs[2], "util", 1.0, "utilization (max of the three)")]:
        v = [ev[c][key] / lim for c in range(n_cases)]
        b = ax.bar(x + (i - 0.5) * w, v, w, color=COL[k], label=MAT_LABEL[k])
        ax.bar_label(b, fmt="%.2f", fontsize=7)
        ax.set_ylabel(lab)
for ax in axs:
    ax.axhline(1, color=C_TEXT, ls=":", lw=1); ax.set_xticks(x, [f"C{c + 1}" for c in range(n_cases)]); ax.legend()
fig.suptitle("Base part: constraint ratios per load case (1.0 = limit)", fontweight="bold")
save(fig, "base", "F10_02_base_constraint_ratios",
     "Constraint ratios of the base part for both materials under the four design load cases.")

## 10.3 Searches on the volume fraction (F10_03, Figures 5.6.1 and 5.7.2)

Each marker is one SIMP run. Lines connect the non-dominated points, and stars mark the selected designs.

In [ ]:
print("  10.3 Pareto fronts")
def front(k, prob):
    if prob == "Multi-load":
        pts = pareto.get((k, "Multi-load"), []); ch = selected.get((k, "Multi-load"))
        ub = max(designs[(k, "Base")]["ev"][c]["util"] for c in range(n_cases))
    else:
        cc = int(prob[1:]) - 1
        pts = pareto_single.get((k, cc), []); ch = selected_single.get((k, cc))
        ub = designs[(k, "Base")]["ev"][cc]["util"]
    return sorted(pts, key=lambda q: q["mass"]), ch, ub

def plot_front(ax, k, prob, normalized=False, annotate=True, label=None):
    pts, ch, ub = front(k, prob)
    m0 = designs[(k, "Base")]["m0"]
    f = (lambda m: m / m0) if normalized else (lambda m: m)
    ms = np.array([f(q["mass"]) for q in pts]); us = np.array([q["util"] for q in pts])
    fe = np.array([q["feasible"] for q in pts])
    ax.scatter(ms[fe], us[fe], marker=MRK[k], s=42, color=COL[k], edgecolors="k", linewidths=0.4, zorder=4,
               label=(label or MAT_LABEL[k]) + " feasible")
    ax.scatter(ms[~fe], us[~fe], marker=MRK[k], s=42, facecolors="white", edgecolors=COL[k], linewidths=1.2, zorder=4,
               label=(label or MAT_LABEL[k]) + " infeasible")
    fr, best = [], np.inf
    for q in sorted(pts, key=lambda q: (q["mass"], q["util"])):
        if q["util"] < best:
            fr.append(q); best = q["util"]
    ax.plot([f(q["mass"]) for q in fr], [q["util"] for q in fr], "-", color=COL[k], lw=1.4, zorder=3)
    ax.scatter(f(m0), ub, marker="D", s=60, color=COL[k], edgecolors="k", zorder=5)
    if ch is not None:
        ax.scatter(f(ch["mass"]), ch["util"], marker="*", s=320, color=COL[k], edgecolors="k", zorder=6)
    if annotate:
        for q in pts:
            ax.annotate(f"{q['vf']:.3g}", (f(q["mass"]), q["util"]), xytext=(4, -9), textcoords="offset points",
                        fontsize=6, color=C_TEXT)
    return pts, ch

def front_axes(ax, normalized):
    ys = np.concatenate([np.asarray(c_.get_offsets())[:, 1] for c_ in ax.collections
                         if hasattr(c_, "get_offsets") and len(c_.get_offsets())] or [np.array([1.0])])
    top = max(2.0, 1.8 * float(np.nanmax(ys))); bot = min(0.1, 0.6 * float(np.nanmin(ys)))
    ax.axhline(1.0, color="#d03b3b", ls="--", lw=1.1)
    ax.axhspan(1.0, top, color="#d03b3b", alpha=0.06)
    ax.set_yscale("log"); ax.set_ylim(bot, top)
    ax.set_xlabel("mass / base mass [-]" if normalized else "mass [g]")
    ax.set_ylabel("governing utilization [-] (log)")

legend_sym = [Line2D([], [], marker="o", ls="", color=C_GRAY, mec="k", label="feasible run (VF annotated)"),
              Line2D([], [], marker="o", ls="", mfc="white", mec=C_GRAY, label="infeasible run"),
              Line2D([], [], marker="*", ls="", color=C_GRAY, mec="k", ms=14, label="selected: minimum feasible mass"),
              Line2D([], [], marker="D", ls="", color=C_GRAY, mec="k", label="base part"),
              Line2D([], [], color="#d03b3b", ls="--", label="feasibility limit (utilization = 1)")]
for k in MAT_KEYS:
    for prob in DES_ORDER:
        fig, ax = plt.subplots(figsize=(8, 5.2), constrained_layout=True)
        pts, ch = plot_front(ax, k, prob)
        front_axes(ax, False)
        m0 = designs[(k, "Base")]["m0"]
        sec = ax.secondary_xaxis("top", functions=(lambda m, m0=m0: 100 * (1 - m / m0), lambda s, m0=m0: m0 * (1 - s / 100)))
        sec.set_xlabel("mass saving [%]")
        ax.legend(handles=legend_sym, loc="upper right", fontsize=7)
        ttl = "all four load cases" if prob == "Multi-load" else CASE_EN[int(prob[1:]) - 1]
        ax.set_title(f"Pareto search — {MAT_LABEL[k]} | {prob} ({ttl})", fontweight="bold")
        ch_txt = (f"selected VF = {ch['vf']:.4g}, mass = {ch['mass']:.1f} g, utilization = {ch['util']:.3f}"
                  if ch is not None else "no feasible design")
        save(fig, "pareto", f"F10_03_pareto_{k}_{prob.replace('-', '')}",
             f"Mass-utilization Pareto search for {MAT_LABEL[k]}, {prob} problem: each marker is one SIMP run "
             f"(annotated with its volume fraction), filled = feasible, open = infeasible; the line is the "
             f"non-dominated front, the diamond the base part and the star the selected design ({ch_txt}).")
fig, axs = plt.subplots(len(MAT_KEYS), len(DES_ORDER), figsize=(22, 8.5), constrained_layout=True, squeeze=False)
for i, k in enumerate(MAT_KEYS):
    for j, prob in enumerate(DES_ORDER):
        plot_front(axs[i, j], k, prob, annotate=False); front_axes(axs[i, j], False)
        axs[i, j].set_title(f"{MAT_LABEL[k]} | {prob}", fontweight="bold")
axs[0, -1].legend(handles=legend_sym, fontsize=7, loc="upper right")
fig.suptitle("Pareto searches for every material and problem", fontsize=13, fontweight="bold")
save(fig, "pareto", "F10_03_pareto_grid_all", "Overview of the ten Pareto searches (rows: material; columns: problem).")

fig, axs = plt.subplots(2, len(DES_ORDER), figsize=(22, 9), constrained_layout=True)
for j, prob in enumerate(DES_ORDER):
    for k in MAT_KEYS:
        plot_front(axs[0, j], k, prob, annotate=False)
        plot_front(axs[1, j], k, prob, normalized=True, annotate=False)
    front_axes(axs[0, j], False); front_axes(axs[1, j], True)
    axs[0, j].set_title(f"{prob}: absolute mass", fontweight="bold")
    axs[1, j].set_title(f"{prob}: mass relative to each base part", fontweight="bold")
h_ = [Line2D([], [], marker=MRK[k], ls="-", color=COL[k], mec="k", label=MAT_LABEL[k]) for k in MAT_KEYS]
axs[0, 0].legend(handles=h_ + legend_sym, fontsize=7, loc="upper right")
fig.suptitle("PA6 vs PA6-CF: Pareto fronts per problem", fontsize=13, fontweight="bold")
save(fig, "pareto", "F10_03_pareto_PA6_vs_PA6CF",
     "Direct comparison of the PA6 and PA6-CF Pareto fronts for each problem; top row in absolute mass, bottom "
     "row normalised by the mass of the respective base part.")

## 10.4 Selected designs (F10_04, Figures 5.5.1, 5.7.1, 5.7.3 and 5.8.1)

Topologies, masses, constraint ratios, cross-utilization and comparison between materials.

In [ ]:
print("  10.4 designs")
fig, axs = plt.subplots(len(MAT_KEYS), len(DES_ORDER), figsize=(21, 10.5), constrained_layout=True, squeeze=False)
for i, k in enumerate(MAT_KEYS):
    for j, name in enumerate(DES_ORDER):
        ax = axs[i, j]; d = designs.get((k, name))
        if d is None:
            ax.axis("off"); continue
        draw_solid(ax, d["rb"], color=COL[k], alpha=0.85)
        draw_solid(ax, (shell_2d.ravel() & (d["rb"] >= 0.5)).astype(float), color="#3b3a37", alpha=0.35, z=2)
        draw_contour(ax, color="k", ls="--", lw=0.7, zorder=5)
        ax.plot(r_nodes[nodes_crit], z_nodes[nodes_crit], ".", color=C_GREEN, ms=3, zorder=6)
        frame(ax)
        ax.set_title(f"{MAT_LABEL[k]} | {name}\nVF {d['vf']:.4g} | {d['mass']:.1f} g (−{d['saving']:.1f} %) | util {d['util']:.3f}",
                     fontweight="bold")
fig.suptitle("Optimized topologies (dark band: 2 mm passive shell; dashed: base contour; green: critical walls)",
             fontsize=13, fontweight="bold")
save(fig, "designs", "F10_04_topology_gallery",
     "Minimum-mass feasible topologies for each load case and for the multi-load problem in PA6 (top) and "
     "PA6-CF (bottom). Volume fraction, mass, mass saving and governing utilization are given above each section.")

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
x = np.arange(len(DES_ORDER)); w = 0.38
for i, k in enumerate(MAT_KEYS):
    ms = [designs[(k, n)]["mass"] if (k, n) in designs else np.nan for n in DES_ORDER]
    b = ax.bar(x + (i - 0.5) * w, ms, w, color=COL[k], label=MAT_LABEL[k])
    ax.bar_label(b, labels=[f"{m:.0f} g\n−{designs[(k, n)]['saving']:.1f} %" if (k, n) in designs else ""
                            for m, n in zip(ms, DES_ORDER)], fontsize=7)
    ax.axhline(designs[(k, "Base")]["m0"], color=COL[k], ls="--", lw=1.1,
               label=f"{MAT_LABEL[k]} base part ({designs[(k, 'Base')]['m0']:.0f} g)")
ax.set_xticks(x, DES_ORDER); ax.set_ylabel("mass [g]"); ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax.set_ylim(0, 1.15 * max(d["m0"] for d in designs.values()))
ax.set_title("Mass of the selected designs and saving relative to the base part", fontweight="bold")
save(fig, "designs", "F10_04_mass_comparison", "Mass of each selected design and its saving relative to the base part, PA6 vs PA6-CF.")

fig, axs = plt.subplots(1, len(MAT_KEYS), figsize=(17, 5), constrained_layout=True, sharey=True)
for ax, k in zip(np.atleast_1d(axs), MAT_KEYS):
    names = [n for n in DES_ORDER if (k, n) in designs]
    ratios = np.array([[designs[(k, n)]["ev"][designs[(k, n)]["cg"]]["u_crit"] / U_ADM_CRIT,
                        designs[(k, n)]["ev"][designs[(k, n)]["cg"]]["u_rem"] / U_ADM_REM,
                        FS_VM * designs[(k, n)]["ev"][designs[(k, n)]["cg"]]["vm_p995"] / MAT_OPS[k]["Sy"]] for n in names])
    xx = np.arange(len(names)); ww = 0.27
    for j, (lab, colr) in enumerate([("u_green / 0.1 mm", C_GREEN), ("u_rest / 0.5 mm", C_REST),
                                     (f"{FS_VM:g}·σ_p99.5 / Sy", C_AXIAL)]):
        b = ax.bar(xx + (j - 1) * ww, ratios[:, j], ww, color=colr, label=lab)
        ax.bar_label(b, fmt="%.2f", fontsize=6.5)
    ax.axhline(1, color="#d03b3b", ls="--", lw=1)
    ax.set_xticks(xx, [f"{n}\n(gov. C{designs[(k, n)]['cg'] + 1})" for n in names])
    ax.set_title(f"{MAT_LABEL[k]}: constraint ratios in the governing case", fontweight="bold")
    ax.set_ylabel("ratio to limit [-]"); ax.legend(fontsize=7, loc="upper right")
save(fig, "designs", "F10_04_constraint_breakdown",
     "Ratio of each constraint to its limit for the selected designs in their governing load case: the "
     "displacement of the critical walls (0.1 mm) governs every design; stresses remain far below the limit.")

fig, axs = plt.subplots(1, len(MAT_KEYS), figsize=(15, 5.2), constrained_layout=True)
for ax, k in zip(np.atleast_1d(axs), MAT_KEYS):
    names = [n for n in DES_ORDER if (k, n) in designs]
    Mx = np.array([[designs[(k, n)]["ev"][c]["util"] for c in range(n_cases)] for n in names])
    im = ax.imshow(np.log10(Mx), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    for a in range(Mx.shape[0]):
        for b in range(Mx.shape[1]):
            ax.text(b, a, f"{Mx[a, b]:.2f}", ha="center", va="center", fontsize=9,
                    fontweight="bold" if Mx[a, b] > 1 else "normal", color="k")
    ax.set_xticks(range(n_cases), [f"C{c + 1}" for c in range(n_cases)]); ax.set_yticks(range(len(names)), names)
    ax.set_xlabel("load case evaluated"); ax.set_ylabel("design"); ax.grid(False)
    ax.set_title(f"{MAT_LABEL[k]}: cross-utilization (>1 violates)", fontweight="bold")
    cb = fig.colorbar(im, ax=ax, fraction=0.046); cb.set_ticks([-1, 0, 1]); cb.set_ticklabels(["0.1", "1", "10"])
    cb.set_label("utilization (log scale, centred at 1)")
save(fig, "designs", "F10_04_cross_utilization",
     "Utilization of every selected design evaluated in every load case. Single-case designs fail the other "
     "cases (values > 1); only the multi-load design satisfies all four simultaneously.")

fig, axs = plt.subplots(1, 4, figsize=(20, 4.6), constrained_layout=True)
metr = [("mass [g]", lambda d: d["mass"]), ("mass saving [%]", lambda d: d["saving"]),
        ("governing utilization [-]", lambda d: d["util"]),
        ("FS = Sy / σ_p99.5 (governing case)", lambda d: d["ev"][d["cg"]]["FS"])]
x = np.arange(len(DES_ORDER)); w = 0.38
for ax, (lab, fn) in zip(axs, metr):
    for i, k in enumerate(MAT_KEYS):
        v = [fn(designs[(k, n)]) if (k, n) in designs else np.nan for n in DES_ORDER]
        b = ax.bar(x + (i - 0.5) * w, v, w, color=COL[k], label=MAT_LABEL[k])
        ax.bar_label(b, fmt="%.3g", fontsize=6.5)
    ax.set_xticks(x, DES_ORDER, fontsize=8); ax.set_ylabel(lab); ax.set_title(lab, fontweight="bold")
axs[2].axhline(1, color="#d03b3b", ls="--", lw=1); axs[0].legend()
fig.suptitle("PA6 vs PA6-CF: selected designs", fontsize=12, fontweight="bold")
save(fig, "designs", "F10_04_PA6_vs_PA6CF_metrics",
     "Direct comparison of the selected PA6 and PA6-CF designs: mass, mass saving, governing utilization and "
     "von Mises safety factor based on the 99.5th percentile stress.")

## 10.5 Deformation against tolerance (F10_05)

Every optimized topology under its own load case(s): magnified deformed shape, admissible band and the most critical contour point.

In [ ]:
print("  10.5 deformation vs tolerance")
summary_rows = []
for k in MAT_KEYS:
    for name in DES_ORDER:
        if (k, name) not in designs:
            continue
        d = designs[(k, name)]
        for c in d["own"]:
            fig, cap = tolerance_figure((k, name), c, COL[k])
            save(fig, "tol", f"F10_05_tolerance_{k}_{name.replace('-', '')}_C{c + 1}", cap)
            _, _, um, _, alive, ratio = node_data(d, c)
            kc = int(np.nanargmax(ratio))
            summary_rows.append(dict(material=k, design=name, case=f"C{c + 1}",
                                     u_green=d["ev"][c]["u_crit"], u_rest=d["ev"][c]["u_rem"],
                                     crit_r=r_nodes[cn[kc]], crit_z=z_nodes[cn[kc]], crit_s=s_node[kc],
                                     crit_u=um[kc], crit_limit=tol_node[kc], crit_ratio=ratio[kc]))
tol_sum = pd.DataFrame(summary_rows)
fig, axs = plt.subplots(1, 2, figsize=(17, 5), constrained_layout=True, sharey=True)
for ax, (col, lim, lab) in zip(axs, [("u_green", U_ADM_CRIT, "critical walls: u_green / 0.1 mm"),
                                     ("u_rest", U_ADM_REM, "rest of contour: u_rest / 0.5 mm")]):
    labs = [f"{n}\n{cs}" for n, cs in tol_sum[tol_sum.material == MAT_KEYS[0]][["design", "case"]].values]
    xx = np.arange(len(labs)); ww = 0.38
    for i, k in enumerate(MAT_KEYS):
        v = tol_sum[tol_sum.material == k][col].values / lim
        b = ax.bar(xx[:len(v)] + (i - 0.5) * ww, v, ww, color=COL[k], label=MAT_LABEL[k])
        ax.bar_label(b, fmt="%.2f", fontsize=6.5)
    ax.axhline(1, color="#d03b3b", ls="--", lw=1); ax.set_xticks(xx, labs, fontsize=7)
    ax.set_ylabel("ratio to tolerance [-]"); ax.set_title(lab, fontweight="bold"); ax.legend()
save(fig, "tol", "F10_05_tolerance_summary",
     "Maximum displacement of the critical walls and of the rest of the contour, as a fraction of the "
     "respective tolerance, for every optimized design under its own load case(s).")

## 10.6 Virtual compression test (F10_06)

Force-displacement curves and initial stiffness per unit mass.

In [ ]:
print("  10.6 virtual test")
r8 = pd.DataFrame(vtest_results)
r8["design_en"] = r8["design"]
LS = {"Base": "-", "Multi-load": "-", "C1": "--", "C2": "-.", "C3": ":", "C4": (0, (5, 1, 1, 1))}
LW = {"Base": 2.4, "Multi-load": 2.0}
dlim = 0.05 * H_part
fig, axs = plt.subplots(1, len(MAT_KEYS), figsize=(17, 5.6), constrained_layout=True)
for ax, k in zip(np.atleast_1d(axs), MAT_KEYS):
    for _, r in r8[r8.material == k].iterrows():
        h_ = vtest_curves[(k, r["design"])]
        ax.plot(np.r_[0, h_[:, 0]], np.r_[0, h_[:, 1]] / 1e3, ls=LS.get(r["design_en"], "-"),
                lw=LW.get(r["design_en"], 1.3),
                color={"Base": "#0b0b0b", "Multi-load": COL[k]}.get(r["design_en"], C_GRAY),
                label=r["design_en"])
        ax.plot(r["delta_Fmax_mm"], r["F_max_N"] / 1e3, "*", ms=11, color=COL[k], mec="k")
    ax.axvspan(dlim, ax.get_xlim()[1], color=C_GRAY, alpha=0.12, label="δ/H > 5 % (indicative)")
    ax.set_xlabel("platen displacement δ [mm]"); ax.set_ylabel("force F [kN]"); ax.legend(fontsize=7)
    ax.set_title(f"{MAT_LABEL[k]}: force-displacement between platens (★ peak)", fontweight="bold")
save(fig, "test", "F10_06_virtual_test_curves",
     f"Virtual compression test between rigid platens ({PLATEN_CONTACT} contact) for the base part and the "
     f"selected designs. The shaded region marks nominal strains above 5 %, where the linear-kinematics model "
     f"is indicative only; contact between the free surfaces and the platens is not modelled.")
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
order_d = ["Base"] + DES_ORDER
x = np.arange(len(order_d)); w = 0.38
for ax, (col, lab, sc) in zip(axs, [("K_ini_N_mm", "initial stiffness / mass [N/mm/g]", "mass"),
                                    ("F_max_N", "peak force / mass [N/g]", "mass"),
                                    ("F_max_N", "peak force [kN]", None)]):
    for i, k in enumerate(MAT_KEYS):
        rr = r8[r8.material == k].set_index("design_en")
        v = [(rr.loc[n, col] / rr.loc[n, "mass_g"] if sc else rr.loc[n, col] / 1e3) if n in rr.index else np.nan
             for n in order_d]
        b = ax.bar(x + (i - 0.5) * w, v, w, color=COL[k], label=MAT_LABEL[k])
        ax.bar_label(b, fmt="%.3g", fontsize=6.5)
    ax.set_xticks(x, order_d, fontsize=8); ax.set_ylabel(lab); ax.set_title(lab, fontweight="bold"); ax.legend()
save(fig, "test", "F10_06_virtual_test_bars",
     "Virtual platen test: initial stiffness per unit mass (elastic, most reliable for comparison with the "
     "physical test), peak force per unit mass and peak force, PA6 vs PA6-CF.")

## 10.7 Tables T10_01 to T10_08

Each table is written as CSV, LaTeX and Markdown, and the captions of the figures in `figure_captions.md`.

In [ ]:
print("  10.7 tables")
def write_table(df, name, caption, fmt="{:.4g}"):
    base = os.path.join(subdirs["tables"], name)
    df.to_csv(base + ".csv", index=False)
    cols = list(df.columns)
    f_ = lambda v: (fmt.format(v) if isinstance(v, (float, np.floating)) else str(v)).replace("_", r"\_").replace("%", r"\%")
    tex = [r"\begin{table}[htbp]", r"\centering", rf"\caption{{{caption}}}", rf"\label{{tab:{name}}}",
           r"\begin{tabular}{" + "l" * 2 + "r" * (len(cols) - 2) + "}", r"\toprule",
           " & ".join(f_(c) for c in cols) + r" \\", r"\midrule"]
    tex += [" & ".join(f_(v) for v in row) + r" \\" for row in df.itertuples(index=False)]
    tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(base + ".tex", "w", encoding="utf-8").write("\n".join(tex) + "\n")
    md = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    md += ["| " + " | ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row) + " |"
           for row in df.itertuples(index=False)]
    open(base + ".md", "w", encoding="utf-8").write("\n".join(md) + "\n")
    return df

T = {}
T["materials"] = write_table(pd.DataFrame([dict(material=MAT_LABEL[k], E_r_MPa=m["E_r"], E_z_MPa=m["E_z"], G_rz_MPa=m["G_rz"],
                                                G_rtheta_MPa=m["G_rtheta"], nu_rtheta=m["nu_rtheta"], nu_rz=m["nu_rz"],
                                                nu_zr=m["nu_zr"], Sy_MPa=m["Sy"], density_g_cm3=m["density_g_cm3"])
                                           for k, m in ((k, MATERIALS[k]) for k in MAT_KEYS)]),
                             "T10_01_material_constants", "Transversely isotropic constants used in the model.")
T["loads"] = write_table(pd.DataFrame([dict(case=CASE_EN[c], loads=LOAD_TXT[c],
                                            axial_resultant_abs_Fz_N=abs(float(F_design[1::2, c].sum()))) for c in range(n_cases)]),
                         "T10_02_load_cases", "Design load cases (assumed pressures).")
rows = []
for k in MAT_KEYS:
    for c in range(n_cases):
        e = designs[(k, "Base")]["ev"][c]
        rows.append(dict(material=MAT_LABEL[k], case=f"C{c + 1}", u_green_mm=e["u_crit"], u_rest_mm=e["u_rem"],
                         vm_p995_MPa=e["vm_p995"], FS=e["FS"], compliance_Nmm=e["J"], utilization=e["util"]))
T["base"] = write_table(pd.DataFrame(rows), "T10_03_base_response", "Response of the base part to each load case.")
rows = []
for k in MAT_KEYS:
    for n in DES_ORDER:
        if (k, n) not in designs:
            continue
        d = designs[(k, n)]; e = d["ev"][d["cg"]]
        gov = ["u_green", "u_rest", "stress"][int(np.argmax([e["u_crit"] / U_ADM_CRIT, e["u_rem"] / U_ADM_REM,
                                                             FS_VM * e["vm_p995"] / MAT_OPS[k]["Sy"]]))]
        rows.append(dict(material=MAT_LABEL[k], design=n, VF=d["vf"], mass_g=d["mass"], saving_pct=d["saving"],
                         governing_case=f"C{d['cg'] + 1}", governing_limit=gov, u_green_mm=e["u_crit"],
                         u_rest_mm=e["u_rem"], FS=e["FS"], utilization=d["util"], margin_pct=100 * (1 - d["util"]),
                         max_util_all_cases=max(q["util"] for q in d["ev"])))
T["designs"] = write_table(pd.DataFrame(rows), "T10_04_selected_designs", "Selected minimum-mass feasible designs.")
rows = []
for n in DES_ORDER:
    if all((k, n) in designs for k in MAT_KEYS):
        a, b = designs[(MAT_KEYS[0], n)], designs[(MAT_KEYS[1], n)]
        rows.append(dict(design=n, mass_PA6_g=a["mass"], mass_PA6CF_g=b["mass"], mass_ratio_CF_over_PA6=b["mass"] / a["mass"],
                         saving_PA6_pct=a["saving"], saving_PA6CF_pct=b["saving"], util_PA6=a["util"], util_PA6CF=b["util"],
                         FS_PA6=a["ev"][a["cg"]]["FS"], FS_PA6CF=b["ev"][b["cg"]]["FS"]))
T["compare"] = write_table(pd.DataFrame(rows), "T10_05_PA6_vs_PA6CF", "Direct comparison PA6 vs PA6-CF.")
rows = []
for k in MAT_KEYS:
    for n in DES_ORDER:
        pts, ch, _ = front(k, n)
        for q in sorted(pts, key=lambda q: -q["vf"]):
            rows.append(dict(material=MAT_LABEL[k], problem=n, VF=q["vf"], mass_g=q["mass"], utilization=q["util"],
                             feasible=bool(q["feasible"]), selected=(ch is not None and q is ch), iterations=q["nit"]))
T["pareto"] = write_table(pd.DataFrame(rows), "T10_06_pareto_runs", "All SIMP runs of the Pareto searches.")
T["tolerance"] = write_table(tol_sum, "T10_07_tolerance_critical_points",
                             "Most critical contour point of every design under its own load case(s).")
T["test"] = write_table(r8[["material", "design_en", "mass_g", "K_ini_N_mm", "F_yield_N", "F_max_N", "delta_Fmax_mm",
                            "strain_nom_Fmax", "F_spec_N_g"]].rename(columns=dict(design_en="design", mass_g="mass_g",
                            F_yield_N="F_yield_onset_N", strain_nom_Fmax="nominal_strain_at_Fmax", F_spec_N_g="F_max_per_mass_N_g")),
                        "T10_08_virtual_platen_test", "Virtual compression test between platens.")

with open(os.path.join(run_dir, "figure_captions.md"), "w", encoding="utf-8") as fh:
    fh.write("# Figure captions (Block 10)\n\n")
    for f_, c_ in captions:
        fh.write(f"**{f_}** — {c_}\n\n")

plt.rcParams.update(rc_backup)
n_png = len(captions)
print(f"\n-> {n_png} figures, {len(T)} tables (CSV + LaTeX + Markdown) and figure_captions.md in '{run_dir}'")
print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Block 10 completed in {(time.time() - _b10_t0) / 60:.1f} min")
B10 = dict(run_dir=run_dir, tables=T, captions=captions, scale=M, designs=designs)

---
## 10.8 Check against the thesis

The last cell compares the results of this run with the values printed in the thesis, each one to the precision with which it is printed. The table is stored in `results/report_check.csv`.

In [ ]:
def _target(name, computed, printed, decimals):
    """A value agrees within half a unit of the last printed digit, plus a margin for values rounded in two steps."""
    tol = 0.55 * 10.0 ** (-decimals)
    return dict(item=name, computed=float(computed), thesis=float(printed), agrees=abs(float(computed) - float(printed)) <= tol)


_chk = []
# Section 5.3: benchmark
_chk += [_target("Benchmark mass [g]", mass_opt_g, 898.8, 1), _target("Benchmark iterations", it_total, 112, 0), _target("Benchmark IE", IE, 1.41, 2),
         _target("Benchmark compliance ratio", r_J, 1.184, 3), _target("Benchmark displacement ratio", r_u, 1.063, 3), _target("Benchmark stress ratio", r_s, 1.319, 3)]
# Section 4.5: discretization
_chk += [_target("Solid elements", rho_0.sum(), 38238, 0), _target("Free DOFs", len(dofs_free), 77309, 0), _target("Passive elements", passive.sum(), 9954, 0),
         _target("Solid mass PA6 [g]", MAT_OPS["PA6"]["rho"] * V0, 1500.73, 2), _target("Solid mass PA6-CF [g]", MAT_OPS["PA6_CF"]["rho"] * V0, 1408.41, 2),
         _target("Critical nodes", len(nodes_crit), 70, 0), _target("Remaining nodes", len(nodes_rem), 1450, 0)]
# Table 5.4.1: solid under the design loads
for _k, _u in [("PA6", [0.478, 0.470, 0.477, 0.470]), ("PA6_CF", [0.204, 0.193, 0.195, 0.193])]:
    for _c, _v in enumerate(_u):
        _chk.append(_target(f"Solid U, {_k} C{_c + 1}", eval_solid[_k][_c]["util"], _v, 3))
# Table 5.5.1: Vf, mass, saving, u_crit, u_rem, FS, U in the governing case
_T551 = {("PA6", "Multi-load"): (0.6375, 964.7, 35.7, 0.0985, 0.2438, 18.31, 0.985), ("PA6", "C1"): (0.4250, 635.9, 57.6, 0.0951, 0.1718, 46.94, 0.951),
         ("PA6", "C2"): (0.3563, 533.0, 64.5, 0.0912, 0.2859, 15.01, 0.912), ("PA6", "C3"): (0.4562, 684.7, 54.4, 0.0996, 0.2299, 17.64, 0.996),
         ("PA6", "C4"): (0.4625, 692.2, 53.9, 0.0996, 0.2519, 18.31, 0.996), ("PA6_CF", "Multi-load"): (0.3812, 537.0, 61.9, 0.0994, 0.1539, 5.84, 0.994),
         ("PA6_CF", "C1"): (0.3250, 455.2, 67.7, 0.0990, 0.0992, 12.87, 0.990), ("PA6_CF", "C2"): (0.3312, 465.0, 67.0, 0.0706, 0.1796, 6.30, 0.706),
         ("PA6_CF", "C3"): (0.3125, 438.4, 68.9, 0.0948, 0.3400, 3.42, 0.948), ("PA6_CF", "C4"): (0.3438, 483.0, 65.7, 0.0929, 0.1416, 6.38, 0.929)}
for (_k, _n), _v in _T551.items():
    _p = selected[(_k, "Multi-load")] if _n == "Multi-load" else selected_single[(_k, int(_n[1:]) - 1)]
    _cg = int(np.argmax([_e["util"] for _e in _p["eval"]])) if _n == "Multi-load" else int(_n[1:]) - 1
    _e = _p["eval"][_cg]
    _m0 = MAT_OPS[_k]["rho"] * V0
    for _lab, _val, _print, _dec in [("Vf", _p["vf"], _v[0], 4), ("mass [g]", _p["mass"], _v[1], 1), ("saving [%]", 100 * (1 - _p["mass"] / _m0), _v[2], 1), ("u_crit [mm]", _e["u_crit"], _v[3], 4),
                                     ("u_rem [mm]", _e["u_rem"], _v[4], 4), ("FS", _e["FS"], _v[5], 2), ("U", _e["util"], _v[6], 3)]:
        _chk.append(_target(f"{_k} {_n} {_lab}", _val, _print, _dec))
# Table 5.11.1: initial stiffness [kN/mm] and collapse load [kN]
_T5111 = {("PA6", "Base"): (82.9, 1069.9), ("PA6", "Multi-load"): (27.0, 312.3), ("PA6", "C1"): (14.4, 168.9), ("PA6", "C2"): (10.7, 123.7), ("PA6", "C3"): (16.6, 189.0), ("PA6", "C4"): (9.3, 85.6),
          ("PA6_CF", "Base"): (192.3, 517.9), ("PA6_CF", "Multi-load"): (30.3, 77.4), ("PA6_CF", "C1"): (22.5, 58.9), ("PA6_CF", "C2"): (20.0, 47.4), ("PA6_CF", "C3"): (22.4, 56.7), ("PA6_CF", "C4"): (14.1, 34.5)}
for _r in vtest_results:
    _v = _T5111[(_r["material"], _r["design"])]
    _chk.append(_target(f"{_r['material']} {_r['design']} K_ini [kN/mm]", _r["K_ini_N_mm"] / 1e3, _v[0], 1))
    _chk.append(_target(f"{_r['material']} {_r['design']} F_col [kN]", _r["F_max_N"] / 1e3, _v[1], 1))

report_check = pd.DataFrame(_chk)
report_check.to_csv(RESULTS / "report_check.csv", index=False)
print(f"{int(report_check.agrees.sum())} of {len(report_check)} values agree with the thesis at its printed precision")
_diff = report_check[~report_check.agrees]
if len(_diff):
    print("\nValues that differ:")
    print(_diff.assign(rel_pct=lambda d: 100 * (d.computed - d.thesis) / d.thesis).round(5).to_string(index=False))